<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/tictactoe_mcts_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monte Carlo Tree Search, on a game we can solve

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eth-fdd-fs26/FDD-WE6-private/blob/main/02_mcts_tictactoe/tictactoe_mcts_exercises.ipynb)

*From Data to Solutions, Weekend 6. The exercise hour after the "Agents, RL, and MCTS"
lecture.*

This morning you saw the search as a loop over four phases. Here you write it, following
the pseudocode from the lecture line for line, and point it at tic tac toe, which is
small enough that every answer can be checked against a perfect player.

Here is the whole search, which you will assemble at the end:

```
def mcts_move(state, n_attempts, w):
    root = Node(state, None)

    for i in range(n_attempts):
        leaf = descend(root, w)            # 1. Selection
        leaf = expand(leaf)                # 2. Expansion
        r    = rollout_reward(leaf.state)  # 3. Simulation
        backprop(leaf, r)                  # 4. Backpropagation

    return most_visited_child(root)
```

8 gaps, each marked 🎯 and each with a checker underneath it. Every heading carrying a 🎯 has work for you in it; everything else is given.


In [ ]:
# @title Imports { display-mode: "form" }
# Nothing to install. Standard library only, plus IPython.display for the board,
# which ships with the Colab kernel.
import math
import random
import time
from IPython.display import HTML, display

print("ready")


In [ ]:
# @title The rules of tic tac toe, and the checkers { display-mode: "form" }
X, O, EMPTY = "X", "O", "."

LINES = ((0, 1, 2), (3, 4, 5), (6, 7, 8),      # rows
         (0, 3, 6), (1, 4, 7), (2, 5, 8),      # columns
         (0, 4, 8), (2, 4, 6))                 # diagonals

EMPTY_BOARD = (EMPTY,) * 9
W = 1.4                                         # the exploration weight


def legal_actions(state):
    """The empty cells, which are exactly the legal moves."""
    return [i for i, mark in enumerate(state) if mark == EMPTY]


def winner(state):
    """X, O, or None if nobody has three in a row."""
    for a, b, c in LINES:
        if state[a] != EMPTY and state[a] == state[b] == state[c]:
            return state[a]
    return None


def game_over(state):
    """True when somebody has won, or there is nothing left to play."""
    return winner(state) is not None or len(legal_actions(state)) == 0


def player_to_move(state):
    """X moves first, so whoever has fewer marks is to move."""
    if state.count(X) == state.count(O):
        return X
    return O


def other_player(p):
    if p == X:
        return O
    return X


def reward_for(state, p):
    """1.0 if p won, 0.0 if p lost, 0.5 for a draw."""
    champion = winner(state)
    if champion is None:
        return 0.5
    if champion == p:
        return 1.0
    return 0.0


class env:
    @staticmethod
    def step(state, action):
        """The state that results from the player to move taking `action`."""
        marks = list(state)
        marks[action] = player_to_move(state)
        return tuple(marks)


MARK_COLOR = {X: "#3b82f6", O: "#f97316", EMPTY: "#9ca3af"}

# Monospace for the display panels. Always a span carrying this, never a <code>
# tag: Colab paints its own themed chip behind inline code, which fights the
# panels in the dark theme.
MONO = "font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"


def board_html(state, size=52):
    """The board as one HTML string. `show` displays it, the panels embed it."""
    cells = []
    for i, mark in enumerate(state):
        glyph = mark if mark != EMPTY else ""
        cell = (
            '<div style="width:%dpx;height:%dpx;border-radius:6px;'
            'background:rgba(128,128,128,0.15);position:relative;'
            'display:flex;align-items:center;justify-content:center;'
            'font-family:monospace;font-size:%dpx;font-weight:700;color:%s;">'
            '<span style="position:absolute;top:3px;left:6px;font-size:10px;'
            'font-weight:400;color:#9ca3af;">%d</span>%s</div>'
        ) % (size, size, size // 2, MARK_COLOR[mark], i, glyph)
        cells.append(cell)
    return (
        '<div style="display:grid;grid-template-columns:repeat(3, %dpx);'
        'gap:4px;margin:4px 0;">%s</div>'
    ) % (size, "".join(cells))


def show(state):
    """Render the board as a small HTML grid, cell numbers in the corner."""
    display(HTML(board_html(state)))


def chosen_action(state, child_state):
    """Which cell the move from `state` to `child_state` filled in."""
    for i in range(9):
        if state[i] != child_state[i]:
            return i
    raise ValueError("those two states are identical")


# ==== the three positions we keep coming back to ===================================

WIN_NOW = tuple("XX.OO....")      # X to move, and X can finish at cell 2
BLOCK_NOW = tuple("XOX.O....")    # X to move, and X must take cell 7 or lose
FORK = tuple("..O...XOX")         # X to move, and cell 0 makes two threats at once


# ==== the checkers =================================================================

def ok(msg):
    print("  \u2705  " + msg)


def bad(msg):
    print("  \u274c  " + msg)


def hint(msg):
    print("      \U0001f4a1 hint: " + msg)


def report(title, passed):
    if passed:
        print("\n\u2705  PASSED   " + title)
    else:
        print("\n\u274c  NOT YET  " + title
              + "   (see the hints above)")
    return passed


def check_node(cls):
    print("\U0001f50d  checking your Node class ...\n")
    passed = True

    root = cls(EMPTY_BOARD, None)

    # A gap left as `...` sets the field to Ellipsis, which sails past hasattr and
    # then blows up further down. Name the unfilled ones instead.
    unfilled = [f for f in ("state", "parent", "children", "untried", "N", "V")
                if getattr(root, f, None) is Ellipsis]
    if unfilled:
        bad("still `...` in __init__: %s" % ", ".join(unfilled))
        hint("each of the six lines needs the value the comment above it names")
        return report("Exercise 1, the Node class", False)

    for field, want in (("state", EMPTY_BOARD), ("parent", None),
                        ("children", []), ("N", 1), ("V", 0.0)):
        if not hasattr(root, field):
            bad("a fresh Node has no attribute `%s`" % field)
            hint("__init__ sets six things: state, parent, children, untried, N, V")
            return report("Exercise 1, the Node class", False)
        got = getattr(root, field)
        if field == "N" and got != 1:
            bad("a fresh Node has N = %r, it should be 1" % got)
            hint("born at 1, so that log(parent.N) is defined on the very first pass")
            passed = False
        elif field == "V" and got != 0.0:
            bad("a fresh Node has V = %r, it should be 0.0" % got)
            passed = False
        elif field in ("state", "parent", "children") and got != want:
            bad("a fresh Node has %s = %r, expected %r" % (field, got, want))
            passed = False

    if not hasattr(root, "untried"):
        bad("a fresh Node has no `untried`")
        return report("Exercise 1, the Node class", False)
    if sorted(root.untried) != list(range(9)):
        bad("on the empty board untried is %r, expected all nine cells"
            % (root.untried,))
        hint("untried starts as legal_actions(state), the moves not yet made into "
             "children")
        passed = False
    if root.untried is getattr(cls(EMPTY_BOARD, None), "untried") is None:
        pass
    if passed:
        ok("state, parent, children, untried, N = 1, V = 0.0")

    kid = cls(env.step(EMPTY_BOARD, 4), root)
    if kid.parent is not root:
        bad("Node(state, root).parent is not the root you passed in")
        hint("backprop walks up through parent, so it has to be stored")
        passed = False
    else:
        ok("a child remembers its parent")

    # untried must be a fresh list per node, or expand() on one node silently
    # empties another.
    a = cls(EMPTY_BOARD, None)
    b = cls(EMPTY_BOARD, None)
    a.untried.pop(0)
    if len(b.untried) != 9:
        bad("popping from one Node's untried changed another Node's untried")
        hint("legal_actions builds a new list each call, so do not share one list "
             "between nodes")
        passed = False
    else:
        ok("each Node gets its own untried list")

    # UCT
    parent = cls(EMPTY_BOARD, None)
    parent.N = 100
    child = cls(env.step(EMPTY_BOARD, 4), parent)
    child.N, child.V = 25, 0.6
    got = child.UCT(1.4)
    if got is Ellipsis or got is None:
        bad("UCT returned %r, so the gap is still empty" % (got,))
        return report("Exercise 1, the Node class", False)
    want = 0.6 + 1.4 * math.sqrt(math.log(100) / 25)
    if abs(got - want) > 1e-6:
        bad("with V = 0.6, N = 25 and parent.N = 100 you got %.6f, expected %.6f"
            % (got, want))
        hint("exploit = self.V")
        hint("explore = w * sqrt( log(self.parent.N) / self.N )")
        if abs(got - (0.6 + 1.4 * math.sqrt(math.log(25) / 100))) < 1e-6:
            hint("you have self.N and self.parent.N the wrong way round")
        passed = False
    else:
        ok("UCT matches the formula, %.5f" % got)

    quiet = cls(env.step(EMPTY_BOARD, 0), parent)
    quiet.N, quiet.V = 25, 0.6
    loud = cls(env.step(EMPTY_BOARD, 1), parent)
    loud.N, loud.V = 2, 0.6
    if not loud.UCT(1.4) > quiet.UCT(1.4):
        bad("two children with equal V, and the rarely visited one did not score higher")
        hint("that is what the explore term is for")
        passed = False
    else:
        ok("between equal V, the less visited child scores higher")
    return report("Exercise 1, the Node class", passed)


print("game rules and checkers loaded")


## 0. The game

A state is nine marks in a tuple, cell 0 top left to cell 8 bottom right. Everything
about the rules is given:

| | |
|---|---|
| `legal_actions(state)` | the empty cells |
| `game_over(state)` | somebody won, or nothing left to play |
| `player_to_move(state)` | `X` or `O` |
| `other_player(p)` | the other one |
| `env.step(state, action)` | the state after that move |
| `reward_for(state, p)` | `1.0` won, `0.5` drew, `0.0` lost |
| `show(state)` | render it as a small grid, with cell numbers in the corner |

One invariant matters, because breaking it crashes the search rather than weakening it:
**`game_over` is true exactly when there is nothing left to play.** If a state could have
no legal actions and still not be "over", the search would descend into it and then look
for a child that is not there.


In [ ]:
# @title The board, and what the code sees { display-mode: "form" }
# Two positions, side by side, and the three questions the rules answer about
# each one. Every value below is computed, never typed in, so it cannot drift.

def _tuple_html(state):
    """The nine marks, written the way Python actually holds them."""
    parts = []
    for mark in state:
        parts.append('<span style="color:%s;">&#39;%s&#39;</span>'
                     % (MARK_COLOR[mark], mark))
    return ('<span style="%sfont-size:10.5px;color:#94a3b8;">(%s)</span>'
            % (MONO, ", ".join(parts)))


def _board_card(state, name, caption):
    return (
        '<div style="flex:0 1 auto;text-align:center;padding:0 10px;">'
        '<div style="%sfont-size:12px;font-weight:700;color:#334155;'
        'margin-bottom:8px;">%s</div>'
        '<div style="display:flex;justify-content:center;">%s</div>'
        '<div style="font-size:11px;color:#64748b;line-height:1.45;width:210px;'
        'min-height:52px;margin:8px auto 0;">%s</div>'
        '<div style="margin-top:8px;">%s</div></div>'
    ) % (MONO, name, board_html(state, 48), caption, _tuple_html(state))


def _fact_row(icon, question, call, answer, color):
    return (
        '<div style="display:flex;align-items:center;gap:10px;padding:9px 2px;'
        'border-top:1px solid rgba(100,116,139,0.16);flex-wrap:wrap;">'
        '<span style="font-size:15px;width:20px;text-align:center;">%s</span>'
        '<span style="flex:1 1 160px;font-size:12px;color:#475569;">%s</span>'
        '<span style="%sfont-size:11.5px;font-weight:700;color:#64748b;">%s</span>'
        '<span style="color:#cbd5e1;font-size:13px;">&#10142;</span>'
        '<span style="%sflex:0 0 auto;min-width:96px;font-size:13px;'
        'font-weight:700;color:%s;">%s</span>'
        '</div>'
    ) % (icon, question, MONO, call, MONO, color, answer)


_arrow = (
    '<div style="display:flex;flex-direction:column;align-items:center;'
    'justify-content:center;flex:0 0 78px;color:#94a3b8;">'
    '<div style="font-size:24px;line-height:1;">&#10142;</div>'
    '<div style="font-size:10px;margin-top:4px;text-align:center;line-height:1.3;">'
    'four moves<br>later</div></div>'
)

_boards = (
    '<div style="display:flex;align-items:center;justify-content:center;'
    'flex-wrap:wrap;gap:4px;margin:14px 0 18px;">%s%s%s</div>'
) % (_board_card(EMPTY_BOARD, "EMPTY_BOARD",
                 "Nothing played yet. Nine legal moves, one per empty cell."),
     _arrow,
     _board_card(WIN_NOW, "WIN_NOW",
                 "X to move, and a win is available right now. Finding it is the "
                 "first thing we will ask the search to do."))

if game_over(WIN_NOW):
    _over, _over_color = "True", "#b91c1c"
else:
    _over, _over_color = "False", "#b45309"

_facts = (
    _fact_row("&#128100;", "Whose turn is it?", "player_to_move(WIN_NOW)",
              player_to_move(WIN_NOW), MARK_COLOR[X])
    + _fact_row("&#128205;", "Which cells may be filled?", "legal_actions(WIN_NOW)",
                ", ".join(str(a) for a in legal_actions(WIN_NOW)), "#0f766e")
    + _fact_row("&#127937;", "Is the game finished?", "game_over(WIN_NOW)",
                _over, _over_color)
)

display(HTML(
    '<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;'
    'border-radius:16px;padding:18px 18px 16px;margin:8px 0;">'
    '<div style="display:flex;align-items:baseline;gap:8px;">'
    '<span style="font-family:monospace;font-size:16px;font-weight:800;color:'
    + MARK_COLOR[X] + ';">X</span>'
    '<span style="font-family:monospace;font-size:16px;font-weight:800;color:'
    + MARK_COLOR[O] + ';">O</span>'
    '<span style="font-size:17px;font-weight:800;color:#1e293b;">'
    'The board, and what the code sees</span></div>'
    '<div style="font-size:12px;color:#64748b;margin-top:5px;line-height:1.5;">'
    'A position is just nine marks in a row. The small grey number in each corner '
    'is that cell\'s address, 0 at the top left to 8 at the bottom right, and a '
    'move is nothing more than the number of the cell you fill.</div>'
    + _boards
    + '<div style="font-size:11px;font-weight:700;color:#475569;'
      'letter-spacing:0.04em;text-transform:uppercase;">'
      'What the rules can tell us about WIN_NOW</div>'
    + _facts
    + '<div style="font-size:11.5px;color:#64748b;margin-top:12px;line-height:1.5;">'
      'Those three answers are everything the search will ever need from the game: '
      '<b>who is choosing</b>, <b>what they are allowed to choose</b>, and '
      '<b>whether it is time to stop</b>. Nothing you write from here on knows any '
      'tic tac toe beyond that.</div>'
    '</div>'
))


## 1. The node 🎯

The tree is made of these. Four fields are bookkeeping, and two are the letters in the
formula.


In [ ]:
# @title One node, and the six things it holds { display-mode: "form" }
# One Node, drawn as the piece of the tree it actually is. The six fields are
# not six unrelated variables: four of them are arrows to other nodes, and two
# are counters written inside this one.
#
# The counters are a small worked example, and they agree with each other:
# N = 9 is the 1 every node is born with, plus 1 result from this node's own
# first playout, plus the 4 and the 3 that the two children carried up. V is
# the mean of those 8 results with the reward flipped at each step, which comes
# to (1.0 + (4 - 4 x 0.310) + (3 - 3 x 0.550)) / 8 = 0.639. Nothing here is
# measured and nothing here is quoted anywhere else.

_EX_STATE = tuple("XOX.O.X..")   # X has just moved, so O chooses next
_EX_KIDS = ((3, 5, "0.310"), (5, 4, "0.550"))
_EX_UNTRIED = (7, 8)

_INK = "#334155"
_DIM = "#94a3b8"
_HAIR = "#cbd5e1"
_XCOL = MARK_COLOR[X]
_OCOL = MARK_COLOR[O]
_XFILL, _OFILL = "#eff6ff", "#fff7ed"
_ODEEP = "#9a3412"


def _txt(x, y, s, size, color, weight="400", anchor="middle", extra=""):
    return ('<text x="%d" y="%d" text-anchor="%s" font-size="%s" font-weight="%s" '
            'fill="%s" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,'
            'monospace"%s>%s</text>') % (x, y, anchor, size, weight, color, extra, s)


def _circ(cx, cy, r, fill, stroke, dashed=False):
    dash = ' stroke-dasharray="5 4"' if dashed else ""
    return ('<circle cx="%d" cy="%d" r="%d" fill="%s" stroke="%s" '
            'stroke-width="2.2"%s/>') % (cx, cy, r, fill, stroke, dash)


def _edge(x1, y1, x2, y2, color, dashed=False, arrow=True):
    dash = ' stroke-dasharray="5 4"' if dashed else ""
    tip = ' marker-end="url(#tttarrow)"' if arrow else ""
    return ('<line x1="%d" y1="%d" x2="%d" y2="%d" stroke="%s" stroke-width="2"'
            '%s%s/>') % (x1, y1, x2, y2, color, dash, tip)


_svg = ['<svg viewBox="0 0 600 442" style="width:600px;max-width:100%;height:auto;">',
        '<defs><marker id="tttarrow" viewBox="0 0 10 10" refX="9" refY="5" '
        'markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
        '<path d="M 0 0 L 10 5 L 0 10 z" fill="' + _DIM + '"/></marker></defs>']

# the parent, and the arrow this node keeps pointing up at it
_svg.append(_edge(200, 130, 200, 84, _DIM))
_svg.append(_circ(200, 50, 30, _OFILL, _OCOL))
_svg.append(_txt(200, 55, "N = 21", "11.5", _ODEEP, "700"))
_svg.append(_txt(244, 47, ".parent", "11.5", _INK, "700", "start"))
_svg.append(_txt(244, 63, "one node up, or None at the root", "9.5", _DIM, "400", "start"))
_svg.append(_txt(244, 77, "UCT reads this N", "9.5", _DIM, "400", "start"))

# the two children already created, and the moves that made them
for (cell, n, v), (x1, y1, x2, y2, lx) in zip(
        _EX_KIDS, ((164, 252, 119, 322, 112), (220, 259, 239, 318, 258))):
    _svg.append(_edge(x1, y1, x2, y2, _DIM))
    _svg.append(_txt(lx, 292, "plays " + str(cell), "9.5", _DIM))
for (cell, n, v), cx in zip(_EX_KIDS, (100, 250)):
    _svg.append(_circ(cx, 352, 36, _OFILL, _OCOL))
    _svg.append(_txt(cx, 348, "N = " + str(n), "11.5", _ODEEP, "700"))
    _svg.append(_txt(cx, 364, "V = " + v, "10", _ODEEP, "400"))

# the moves that have NOT been turned into nodes yet, drawn as outlines
for cell, (x1, y1, x2, y2), cx in zip(_EX_UNTRIED,
                                      ((252, 237, 376, 334), (258, 227, 469, 338)),
                                      (400, 495)):
    _svg.append(_edge(x1, y1, x2, y2, _HAIR, True, False))
    _svg.append(_circ(cx, 352, 30, "none", _HAIR, True))
    _svg.append(_txt(cx, 359, str(cell), "17", _DIM, "700"))

# this node
_svg.append(_circ(200, 196, 66, _XFILL, _XCOL))
_svg.append(_txt(200, 172, "THIS NODE", "9.5", "#64748b", "700",
                 extra=' letter-spacing="1.4"'))
_svg.append(_txt(200, 198, "N = 9", "16", _INK, "700"))
_svg.append(_txt(200, 222, "V = 0.639", "15", _INK, "700"))

_svg.append(_txt(175, 414, ".children", "11.5", _INK, "700"))
_svg.append(_txt(175, 430, "moves already turned into nodes", "9.5", _DIM))
_svg.append(_txt(447, 414, ".untried", "11.5", _INK, "700"))
_svg.append(_txt(447, 430, "moves not turned into nodes yet", "9.5", _DIM))
_svg.append("</svg>")


def _field(name, color, desc):
    return (
        '<div style="flex:1 1 252px;display:flex;gap:10px;align-items:baseline;'
        'padding:8px 2px;border-top:1px solid rgba(100,116,139,0.16);">'
        '<span style="%sfont-size:12px;font-weight:700;color:%s;flex:0 0 68px;">'
        '%s</span>'
        '<span style="font-size:11.5px;color:#475569;line-height:1.45;">%s</span>'
        '</div>'
    ) % (MONO, color, name, desc)


_legend = (
    '<div style="display:flex;flex-wrap:wrap;column-gap:22px;margin-top:14px;">'
    + _field("state", _INK, "the position this node stands for, on the left")
    + _field("parent", _ODEEP, "the node one step up, or None at the root")
    + _field("children", _ODEEP, "the moves already turned into nodes below")
    + _field("untried", "#64748b", "the legal moves here with no node yet")
    + _field("N", _XCOL, "how many results have come back through this node")
    + _field("V", _XCOL, "the mean of those results, for whoever moved IN")
    + '</div>')

_state_card = (
    '<div style="flex:0 0 auto;text-align:center;padding:0 16px 0 4px;">'
    '<div style="' + MONO + 'font-size:12px;font-weight:700;color:' + _INK + ';'
    'margin-bottom:8px;">.state</div>'
    '<div style="display:flex;justify-content:center;">' + board_html(_EX_STATE, 44)
    + '</div>'
    '<div style="font-size:11px;color:#64748b;line-height:1.45;width:150px;'
    'margin:8px auto 0;">X has just moved, so this node is scored for X, and it '
    'is O who chooses next.</div></div>')

display(HTML(
    '<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;'
    'border-radius:16px;padding:18px 18px 16px;margin:8px 0;">'
    '<div style="font-size:17px;font-weight:800;color:#1e293b;">'
    'One node, and the six things it holds</div>'
    '<div style="font-size:12px;color:#64748b;margin-top:5px;line-height:1.5;">'
    'A node is not a lonely record. Four of its six fields are arrows to other '
    'nodes, which is what makes the collection of them a tree, and the remaining '
    'two are the counters written inside this one.</div>'
    '<div style="display:flex;align-items:center;justify-content:center;'
    'flex-wrap:wrap;gap:8px;margin:14px 0 6px;">'
    + _state_card + '<div style="flex:1 1 420px;min-width:300px;">'
    + "".join(_svg) + '</div></div>'
    + _legend
    + '<div style="font-size:11.5px;color:#64748b;margin-top:14px;line-height:1.5;">'
      'Two things the picture gives away. The colours <b>alternate</b> down the '
      'tree, because the players do: this node was reached by X moving, so its '
      'parent and its children all belong to O. And <b>untried shrinks as children '
      'grow</b>, one move moving across each time the tree is expanded, until '
      'there is nothing left to try here.</div>'
    '</div>'
))


**`V` deserves a careful sentence, because it is the one thing in this notebook that is
easy to get backwards.** `V` is the mean reward *from the point of view of the player who
moved into this node*, not from the root player's. A node reached by O playing is scored
for O. That is what lets the same tree serve both sides, and it is why the reward flips
on the way back up in part 3.


In [ ]:
# @title Meet the two points of view { display-mode: "form" }
_IMG_X = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAIgAAACWCAMAAAAyuDCLAAAA/1BMVEUhKUzO2uUxhbYZGDfxdjMvb5mM4OxSsdJYUmArSm36jzF10+T19vRqamuYnKpPLjRWkaz7yzpPUmafWjf60EVfc4qkpaVfYW8vMT04k8JpSzrfpUmYZUcTE2auuMVMNkPSc0OukUjL1uHO2eI1OFJ1d4Zqa5WGh5EAAP/X1beqijwSYmI9QVm/v//DubaJfYcA//9vkZj//wCGf4ZucoOBOjG3xbfCtaeBgX28x8f/AP9/hX8A/wAAAABJqs39/f0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACQfr59AAAAQHRSTlP7/P7+/v7+/vj7/v4KBv7+/v+c/f7+HWQE/v7+/gP+/f7+m1eoWCBNAQ/9A68E//8BIQEkk/8WEzUXASgBAP7+3dBW8gAAFPNJREFUeNqtXAl7o8jRLmgDDWoOgwDrlj2y58zut8km+S7J//9fpaq6uaQG5J3p58lkzdG81H0hON+xdvi/f73mp9Mpryv6++ktPn9g4cXqNQdaYR6r8/np9na4c5sf0Kz8TR9+UvcjUTX0Vk4vE6sPAiEYdYivUmZBEGQlbZTXTJj4f+8CcT6/0v3Zwse10DvcQoF5HDUIKAO/WVnIr3Wqv/LZO3AgU8pFez9iwQ3zWg1vnwHydK7MNotFu1cGIKXA91IsPZPrv84KNwj4fr0D/otQkERDKDC9C5IDwgYG/oPcCfBfEM+bLW5Wn3fTkqKeEEfYo4ZefkBklfW5exOYZgu+TdbsEVyaFVxAbJYbiUQ5z1G0w0GvoW8P8K3KVmxngcTnrx2OHgzaKiMkyw3yWu3iqS1yES4MjP4GlwUzmGhq2AMTm8QoVIHP+wxgaCRy+bjcpJM0ic+vAAvm6+JyvRalkFt6E40ExokaA0mZfZdL4AlE8riUiORpFEdFFGUkF8sWodhuDiA1EpigB++ysO9yuYTiGWmCSOoxLSYB0ZwNrDuQpKGgya+0AUy8jChvpaNbWkyWG9YdO47aCOrIDhcX0ihaE02+jAHZKRLqyV2CDJmD6xmgstEkPr+RgNCrjOxwWXmiICQ5Xgxj0o4EySZ3ITHZEpIthOqsbm3QV9rBLmENECZJJJCmT2DH8RnSlF9nfBcjJign61uBjZ92+C7lNI6L64YCgUT0JmB3+yFsRUjbBBM42Jo8ourwK11vwgIyiQOBMG8iEniwEqSGdCO8mW2YOeulFtj/G4pJVdfsYfzJN0EgsGaSgLLLiITjEXVmBkhAmoMr3ZKYdPFJrHQQhKHDYnID13U1kDXUYFfdw0YDmXqfi6cDgpCccU9MSG3BwxUizDAL7gBSQA52/S+ihxkgJCAQ4uNcIvCzhNeOOZJxMBaEcgeQSEiwG8Ti4YGEdQKIp2HoBVv0f1Vr16AFQlDgEoxpbwsECWvTmZN4eHhIYUJ70U90MBDIGq1J3gD5O4DbPzmGhGlZjFPkfA4RSLTluGpUXVocrkeyT9ak1WHZowidDUc5EwoNBAXNBgQAgbC02oEErgj1O4chS6v49vi4MaZeoawLXnSKkQgvGOMMvXIUbaysoTdC1kQHoNjQ7jYBqYo0108DHRE8o/fafSERE2WGS6MEZk42xpn0gdYRpFVGfsCReMOuws4Yj1XX8zLthylIMrHJrvG5FN9eMg91y7OShAiChpVwoPrWI1qzjfC8gJFYJARSy+wS6CBUB0loYcnUU/ywaE0hXkBQIAxsBHENQVAxYitFalQqPF2Q/11YnR3D6OsQer/l47MApUIiZM+0M5RbiiCMladfmF7ZZuK/kwM355HEtpgovOJ5QN4PiYI6nIsbn4scsoQiJCEHTRA0rFant9MWjaSktJPEsKTJDjAVRUvyyCEsmOiBTgZBC8XKGC0hD0cyhqPel67AgDIbCxXx4Y2v8TArxkCYNGfD8TaKqF7emKdx2YZoCYnIFj6NxCPA3IuOLHh2V4OCIssEl0RpdDnlQoqggJSUKyfJHs+MOT3WGMN/1F0yQDAawh9Zrzg8sruaxHk3C58YciyNvERxKdsTDp7xZhiDKlNPxqxyw1eR5tykV/j+yXt/7dm+IgEwa3IGZxKL03NdHRJpSRVSjaYT8U5JKFrNWdxkEjB8Gi4pyjLMyiuARBUUmyESt68xD0J7hgGQL3E/F9AstGmOBQchCfzsFgcjufJU2slELWOeLJnermdeWagj0pwBc4LQhgORgBUHIhEDw6o1RtM7akO7HpAvZ6V6SDDFKiIt1AOSoEG3Pg7fXEjrCZQTNxgKSKe5YHIiDUQpbT3g1CZtnHQeIzZrQ82Bkcfh8xz7CUfClaAazX2ANsKEjicVar/s0kcKXQ+t5rRmDQmyHzwELYl5focQD26THqq9aMwJCwhmTY3mtjF3R5Hvr8R6CRRTDL0wueGWOUE40E9HUj7Bh5yOIM6hPWq41kiJJohsNRcDyw4IpSHIEwbidAEfM6fQN4SdmEA2ZMAhTVOR4rESevjS5qj+s4QBY44GRy99B2YB2em6AqJ52UiPYqnRzFy3Zg2jxP0QyO/47imJqZQDZTZHDas0bwYCchT90gpRhHmbwJ9QvmuSxLqwtYsbHe6YgyLid6bd4UcCPTN559vpKMuLPvqbAeIzEPL9reZuDoOMuQWCSLR9kCcU3O9abtDA8m0YSRvmBB44LZCDdPBdKWillwej1enWcaQ5ujVAHKDQaOBzr4pemjU+XV0KNhB7zpSo+J5XWoeb0IQ0h6xZy3mUAuPve0CujvaADBhTAPx7WAKPYwVavqW+T+ZUX5UJehTFOsyidSAdHlKEJJKjeBDr9NABSQ0QPNoA8RGIa0x742L+e1gCj7mUwZqAToH4n4CKedOEhTo3yqbFBGWkx5oGiNziI2V5BY+PtjIyMO0P4roGiDJSvalcW20HIxp8CqbmRmrU+a31wxyaYPiJWtMg2Qp+JhLyN9xAQqMwzdHEOAOSpExHy+vo2pI1QKoT2p5ckuo6xlYlgk2TDycKkShxMzrMYoLEankj0vUaCSB/k3hHY+F/a44m8tCIiAS3l09hHiOvi27wAzCoo2jPd1pnliSaUWxR2ggWXwPjUbKsnf6i+Vyv5Zq09729//f2qGNwoGXtCSq5un9fFyJBa/+e9M3sk+yNMTRmj3LI1Fgg1GG3x5v3JOWlITS8MUfFwXlvOeP1o3aAzzc1NzgZj+A0LivR0YbfmhtmTtH64QB1rEPi/C6lNP4tE1kXq+LR5g8kCLCgrhuXa6n9wZvR3ffWh5pdoS2edrE0BekLDH96SAZxqzZIVzGAgwi9LgaxCKrWGoxCBh7b0LeEfu7XRAQC48FFiFb+Fgm+dgbh7WFmzGrVWrI1Bsu39WGyI1UOYdY6kEQ2gV938d+/q741CcjeXCEhxcioeTI8jpc5exH2LNnA5Q4ogrFhdQJokhFHmvfr0y8+fw21NWExweh5z8/onucjPQKqz/cx0iXkjFhA2nw7tnYzQDtaVbe6ayRG5Go3rPDrV+Jw7RJiFtVf+DT281RrTPonfMn0QEGNWotq7e+Aad6dT31BcxIohza4sybU1Qoo0yv95mEJiNBkUQFixLC/wYdWwetbssNonwmaIn7VaA8TNNu/h/lN+WatrQkxJ+D2ryzLUl6VdTkrpjOSzKR34WKfsWTcPvgyBYTbb1pM9LtcCQkFbEobJC5p6VpQrxowLBToyhaeoP/uLBnGqF/H2l3QrxPJxHf8vbYSZFj/6F/51uQXVAoOmsKI/ue2ZqFPBJz4NxZkTGGuupy78z9O/IZsVUpxw81dIyYpzHQtLllX2CKfawhCFjWOn/SKr6YUYJBtVjVpAycJZUjyra7EhLbEIG+ma0F8aXI711uFoi1QXZUN35S1AU1UyyGjkJ6UMey6ww2Qk/gWRUuYo0hGvf+sBeKJZ25BIGc+v9Y5r/pzpdqH3vZ9q6eK2YMmmZP+XJ2f4i89DaZK2TPFJf1iiY0kGXhBrzvUFtj6S+af1Xm8E66qzxw98H6SmvfEUFyU5WweHzf9vA+vCa+hBAGVM7Ms6FXcv1FlWkDIxVkedsh0CY6mA8Y74TUi8VwqUNGVDR8rLrlzmaxXRc2IjUG/3qjLqt0xjog21PlzV65WpsBMTZAxynV75RYIyjWNODASNlsyr1/jP+IYvQ0SeH1VuGFTygYjaGvewwLeiqRkvYzQByN1+qCJNCEPKox2wn8wkoCLmC1PidPfKNPyb0YndB+Cm2dwU71bMXOW7Pzwv1d66YtoGAX+HCvm/T+VfMHTRMHEyDTFlkZArnWG6QC6MYK3BNZuptg8D3pN5IJYxRc+qae9vHlWO24Qur3VCKq1m0T0ps5Ir9h821ddFk0Xh7MvIqJLsAlJNTU/ErZQPKItzUZwVcAfK0WTjLDeWxorFAgso+dnpIvUjrmkcrC3IiTBSAPaWA1VS2j7mNwIkRO94JWh3GgfEfSYBxWFu2ADLRYK4iIca0C3A2imkRwaSxZOzj6gHE42vUnIWO+SXnHN5eEa+J+JGaMvpN7Va56fDp0lm3N3k3MAbGDJoPQyNE9P+VQwPbyl/08SjqUeOAr+IpDAGNhHoqy8AuJNsaY1bwpgTlDvI4nWPFxbk/Nz5kVAwnkgOkSk5tT80MI8c8AwR1ImH5LtCSmEykY6WNeDAumSPFbo+z+FQ8+tFEwSFLc9pUAUXFPXadKOtLM1IvpZQe3EpGEOemJo6rTZBeR5BzM4vqL6k6D+pIB0zEkNc0TTX6JqYIVp08xobw4FC2r5Ewpzq8OIQyZNvbYU41NX/Q6wsWSLn2WMQUIUZmFtSgUotSc1N71Zw6jL/evMEYijy5r3mM1hvh9PTm9WpnVJraPFDGO0rwnu0RzZ72/IUlcvYdqCFKaFOiuobbhwD5KyV7jwRcIVEJi0IHe4uq5faCKGyxzlvEFLkKrl1ZSwVh+xIG4XQ80iocyv18dwMpFPAYkp5/7GTjubxaHlowmkgjuYs+9XQGBKRoyAPG7vsCC6yh5SwZkD9Q8xxykldWbGLCuV76TW3HkLYuixpvTcu0NgyayVXe0tLOF1DEiruWJi9Ko/K9SUHjhlmAjUzGABmjWn7W2BhB/quxWI+q4kxyD3aK7bb8MUXDG7gySmvOzsyeUkYmSM1KTbd/n+lTscfbhLTIg5iQlaqVA2AoQZozV3FkcjqISjWDNz7hATCurBpxDN9x1nPzbIgj7XaG5gnzG6ZcyaS8ooJ1Svv0OHmTlN6ZHbICNjpMLUQe4UkFRX6hIhab78DuZwOgzZ3vcpSHu1GzSy7Rwtw2JOc/uCSkmCFNvoeBdz6DbK2IVYb8enwAH4w4jRWcWhZdc9FFZIBxBJoXXYvSOmpxx0uQS70yPOSA7bgxnGcD+ZBzCpgZKYFgUJy7yYsOnRkfSSJiMsCRaFZRwewozGaIVhSxa1CW1Cf95hYHUdWOeghd3Es5eJIpoC9xd3CkjR+dPfqU81LyYBVxg2j1EUPX6jmVwbkBNwyT2bmyXvGsq9bBYFtqCpsdW0qQ+4+cojOzR6bbMjBkgB2cxwvXZ1BZvTZDDhU8zrcGDmjJtK8CiQaAZIiwMFdT0cOkKBPW5ghjmr3ihaMcaaHMwM6WJGcdGSRSSoV0NHCRw2W9bh8XCtM8g8VmsHwmPxk0B4pE1b1OgINzNOgEjWYjJc0wq3Nl2cz7YIjZqr7L3KUSArHYKwoB4B4HYijQflJ5gTuL1B43Qsr1H8BG5UTXh+j11/dLBMn1EYup009W5/dIIHBWB0rjaFxWTZTocgawgt44mCBppMkLSaCGJM+4T6jWD/ZimlxmrmT6RSJgQRoScsveiQkKRah1ejJF23U5M7+6gxBorbCInuX1y7XTexMjXH3FsRISEhST4aHV5ZVa7tJ3ExYDRCgyP5j4U33CVo0yg9BUpiYBtQpOkIvGRr/LBrIynrfkuQsZg1h0MUpSIjB7u6rul6TR9X0PNsI5SJnkqQuvs8gNJuYiTkYPrLI3kNMgetnhiW47u0Um/CTtazTUo6pv0tDyM7tCpDoxPfx9skNEFJlBNa3G72EHrsgZ9mHW7lgRE8J2EEiRlHS+l7i6mPSlFMJJL/eIuE5i96OFz7UGkzU8QNQitNTT7WfFM6lYTnGPcdDxAOYBBb9LsYQYTEPtpK3R4XRpCEJIKkc3k7SgKTH4OD3ErBfRVGwemz9t2pIIKsViPjvnuaXTG8gWuq4rusN9H2wE00NfvhMV5B34PLLQXbuvdEnTIaiY2Qt4BA+GtMxz527K3IYID+uYDQXfVhIKWLw+D75+li3hMShVqiB8mNJ5qVbT6HgTwHj3QxBPt0MUorIUGd4C30m7j80Y846E/bq0F7G2Z/M4Bn/Y7b7bYgH0emMCWanoBt5NgAtASORShTILpC20Izi354YPfl/s/1Yz3/i0by+BDpcAphSLK9NDzcK3RcLRq04hZ2FWsW9xvg9T/6zfj7fsCAWjbmdyHE4aB/uuAVj/+hB1THRtNJbVY861mb9joNBfxZ169visXz5tcl5n9bgrtHqs5P+jcq8G1UXX/OeRzC9UYn0DWQDPLX+vP1sNdTfP4rP3LRfMmg1Ffe0dCZhqxG7KqOBFzdsKeFWfbuCV/oaUczBjODLNNkic1Qy+7pFeTzZrvGXJ6+YB4b1ifbugowDSyeN88HdLBfZp5wJ5BBjL/cYH72kIbBHJALGnL6HHBD0fGvBhLnQDjQQMOErNKso7fKgL9LjJ7hNX77tUBiSjb6QPbv7+Nq0wDZojWZ+92UDwEhj12j0ysYSDhJkZKMPANB50ZV7tEfXPg4EDJNJ0GfFVBJRHj8UeL7mPqiVb+EKcX55T7RUOJfAgTJUfGUukxQOYsH8j7pdkx9E7HlC6JUvjtSJpjq1GqKKPABctTk/0rHKWH/Duj6yAOOCUmJwRNeEBXUF0rek/KAuH4FReLzVxoXL5Dh5fseJOslAUnGgfDMl5SABEzpRmHs/U8AUVTk4nHx9GGzlv6h4KdMAaEwjq9Jy70oNgWUyb6c+I2QWaenqr/R3JWCZia7iIpUfz/N2dFgerVbpR6w5B8FQOqlusmbUAGCNlQfAqLU38wNlcobYXDM5+QaBk022oGA+aiNV1q2HDNRGW6t7gOi9PyZUp8+vdBYQ2vL92kHY78fpQg4GaSGKkXY+cLSeXn5pEGoSs0DoUvUp5eXZogXwnYAWFKcVqQiJDUGOw4noVGiDDi4jNb7TqtNBZ7QnM9ndQdFPr309t1DCdRE55dN/5nS3Pi7L4X0nTEk9HGpg/Ip0n8KfWOSgQxldwlhUdNAkCHDbX00HYnUYXAoy8ThD1fHBKSRV1EiGf2sNDfKcv8u5eCaT2oynUCg17vqz00cM4PuJPimchKGoQrad6e70cmuZeprPJlgqerTDRJBvPF9P0nKUD/gjsXfqSAF+cYM/8icj1BEy8jtnmDyApndhcJpJt+aZCIsByL18ulfczLC4qyG8soymyT4cs6Hl7+/uU9rzR2/dRVXjdi+vDi/dH3MjtAPAHUXkllDQD8FiW7/1MqEqir1IV8T4w1V3FdswsSgXu559AtdjTep3o43IGj9B3ST6KhuhGa8AAAAAElFTkSuQmCC"
_IMG_O = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAJ0AAACWCAMAAADzP3onAAAA/1BMVEXc2+FrFROdIh7TMTDpVVLeZi309faSXV3xoUuymJuoUCRyHyDsjTbLq62TY2F1c3OjXV0hDQWxoJ2LUVB0NDPk4ubj4+Z3T03Gu8P/AADItrZzQy+wkY+7usR8TEx7Q0M/DgfcPkH/f3+BOThsNjabfICEQD6r5uarj4+8wMlwPECqAAAA//+pg36jgX94QT6qqv9/fz9/f/+EP0B//38A/wC6xLp8PkDQu7ieg3vMv8wAAADWKir3+PvjMzLhLi0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABbHldKAAAAQHRSTlP9/P7+/v4M+v7+/QP+/l8DEv8hmKCeXPj+ASL/Sv5njwT+ArNo+7AJk/7+AwFN/2kDBAL/AgEapng6FAD+/v7/aEo+BgAAEfpJREFUeNrFnAlj2jgWgCUZW7HBxgUakjTpOe3c1967dsz//1f7DkmWbBkMtBnNlKY4wMe79SRZNFeN9ZL+yupqu9NSKvgj9W5bPWbmF5bLj1e8vbiGrcaHh2r7SdJQPPgfcrutqgeCXK7/AjoU26bSjJUs/MGQ+LitHkjGL0yHn1d9IrRFfJAUkbDaWDm/EF0Ggqvww5PFsWHVvK0v06+4VKkVqm5xchgBSv0Ir8pegg4El22HbJ0/0hQfP9CFVElVFEp+2ZyvXnGRxVWBTrvp8cHIr7gpQL9Zs7z/xnR1KLgoVOL9THxS3dzAi6qmuf+mdMtm88mxpRMiS5LEIyQ8mRrx1d+Qrm5qFlx5TKWJJ7+kRbxEgnZBfNvvz4l94lybY7iyPGpuPeYB5Ni2idVuIdWmWX4jurpZM9wptsTTMdAZvILw6vl44jzJAVw7Aw4khho9MB3iIZ/Bk/PxzqH72DzMkxzgHZLDoYX/WwPX46VSrufa3hl098tsF4FLeAxkRzz0YOAQL7F4m5l44pxYsh3C9VQBXnvAiJIYrXp4rZQrsr1sXlgWZ3hENYBDIvhoLEPazhcfEpliT/VwKEeIezeIt51nemK+R2wgonpwSNO6WlP5wgOJSTdUG+KBbm+UrGbhifl6/YKiu/HgWuUKYnBlkx04ECtbKFMR6tEBXkp49RzTE7NFB3rtejhAUK5WJwZfQh5deA2lCkEZPSObUVCJ2f4qZeJn1sQHsBKyntANLvZ8VrdoeuuvRbdml/A8Ivh8Q5e4EV5Fxfd45Leg2xlhRcyEy1RgdL5w/AkZpy3yzXCeNhJeOkd4YmY02fqis3DSG72gKMap8FKgWxZeAX5bfw269T2ksN4lEvwANaLyaZDPezoQHic0LKey5degoyyROpcg0cXQYP6AMwhprZBoWHKe8DChzRTeHLqsyVB0C9/qAjTyykKDYd4sCsvt5diB284Xnpifw7w6XRn1tQ6ugOs3aYFgGgf+QtKXAL5q24Vz2/qE8MQsxRaQw/zKBNOoVRvJJV2kNG+QuXimITThRUdqhLc6mW7FvAyrhtMvLtuM5FBwKQouZzIi1Ea845FwFT8j5ok5oqsoTQxrur4EAKPD0lIbuZHkYkWAU23KfpGeEt4srwh8wtZ1WAPovBTiLqc+lNQ+m4YLQpR4SUWFZ/3iSs1+RMV24dw1oRqg7GVVAoX9Vw5Xnt0VkUf4wC9ItcUJvxCzFIvBbgiXu8+nUVom7a7wJXKQId3CqnZ3VLVz6HYDswO4XlKWTgjG0r3g3JV8gJekM1Ub0GV1HQ3FXp6w8W4Mh3jiWUevDPAS57WF/O8x1YpBOzMSin8dKhbgygiCEBBS3JXgwki5NiCf8FoRSKmq8a+T8WRgWZ6InOOKwfAqPDu3TUm1+rRmP9YZYNQQGHbfD3pYy+aTGsQTJT2XFD6L06sYMuaD2Y81PAWGtz4tu/dQm2t4/91wqjkyu6Hocr3XuUHQ7nl4dg9pzdNtSJcwXSEfj6iW6TZVhhj4Zupd8NtrE+2iLkEBI6eUsGc6YVOZ2L+SlHajwks8w6uO02EDQsoq2xBdDrnvflCfDOi0r1fxXCLGK0nysXqFC2/oaYdXhn5hI97qaMQTaPaqFDkYAPgb2u+ueU/TsCwzhWcb0CV+HKYHiWJ6BdKTPvf+VSi9wC96t1DHIh5qdqc5OWr9TALIaIrYUAsfnEImAV1rgwYKLkc5vXpFILmw3GRvkp/dTxmec4sjdQrKbkdUAjInG0hFKx8ZL3I1oNjUd4rWGRfaHIjmjaHTd7aCQqT9K/N0NOT1dEdTLWuWXn5n1KAl9tXRFrcZu2zqm51P9xuq9JXh2GMoRkMU/rNOtXrkFsrOLpbTdNkaQsmzCab81yMylWWuAG8zQUf+qnuMQu5RduQoPd2bN3G6mdlC3NOKV2neoDSWh/aNnFnzSHVxkClKV37kjmNfQLTkb+nRSf1G3k3QpdI47b+Pya55eMy2Fo/ePpdbCi536CDj8kmSK9iyiUE0Fk7ss4C3N8+CKnqvkFMh5UguE9kXrrD5LVgomlR0V0JwwSzbDWoA7egEcFBc0/kbyYnMmAg9XeS5U+wg3iWW7mgRJXZFSZWt7OHg7a3oMNyN6KSXoACBBll/zpaBmPZpPVVEpX7Amwopgt7wmTzCZW3SD8BhJBoGY6NaO+5QcmD7AHf3lBv3Ijx6VosJxVJIMXQPR+jsG+b2M20ax0YCFsZAFy6UKOnXRp/fkGYB9OlJG/sVrI59kGbHdC7gTdJVVhKu4OEfQHTNe7AIjZoNF5da942C8XSXe8F3VN8NinePbjMZjgWYvRG/0ytprNS7XzDfjug6mGQ7EQUDtFx6uhzUxtN0j9Oyo6JT+5/GblfK7ygOadWN6Cbw0GXzKN4Ibi4drfHvQj6cQefyLQl8THeg5mEZUV5Js1kdhRvStT3ddCoTPNmpd36YQNFpeNGaNBulU0OI3E4icbIdXCllpKPiy+4oHe972QWWnuc/GU9HunRIx52n3Pt9DUnB1naBpZSad9NcTocl8IMfxZ7yvCz5VegVcTrqo9DA6OHmGiw+vqRpX5QM16OCXspqDh1V6A7vydZ5fzJdF6VL08I2oXT5HI5S91dKXKaakN0K6X494hVuevM2MCXMY2vIZO8gGsfoINKn6Sol93iODRfZ80m6lOkeT9PRtJr0YeHkullyrugiXkF06UrK51MDS6cxnUK6FdEdi8YeXvbW2jrAgday+5ry7GDZ2tAViOe1e/xxp7XrNUbplEoN3bEFvWEfZUva/cxLILtfmnU1oDt0Pt1KxenIUYSlaw+LMZ2i6nN1vAoYbHCCQp0b5mRauw1NZ0PRzaGDsKLt/FFiHRHS2dWrIqUKql7P699lTbbdwYRnkZiGdf0AOgjpOkOnkK6QoozCSZCf+RGzjU93UDJYBprePTjsLlIb4C16GQgfUwKu8oSKHdCVEdPL/e7jkA7h7C6qFBF3VTYR8cR4+9oSwwivxSXIN9QsGF5iAh6EFEiu2uu2Iw42yvonSup0hKLDCINhNOUtXHI/scFHTDVj1SKxa5ndWLOJoQOn1SQq6kJhkCuHiYNKsHREZ9aZD7iJC96riipXxFeyGS/h5dTEKpV8InTalG0/9xc8c/E8pOtCn1A9HowO1y+iK3oTXe2fcVXRrrQmgcey8FpjeKkNxyA0HHkphv5BJVgbCq9fbU7sFGhzv57dc1/y/kR0DpCisTcjO59uMqSEZneIRDuzFEQSXCyinTJxZBfgjhYKQYlMZ+B62UnjtCfyWDKma43PYxmhcEMXZt2syeavV9RUM5MGFlanhu7Q8iqZZ3iTg1pY3Vh28GrMY2AZCle2FirmGEdWU+65ZoaxYJ9wdFHDO6bYLpbHkA5ED4kOhafNPGb+Wg96UbbZyg/W8A6ealtTphw3PKfYw4gOawAwW+BPUbXy9zPp8AwApV4jvF7BRngnVStYdIeIVyiEW0GglO2CQkN1Nh2uTmVM54cU+LSEcvkprwXRpb1iDz4dKPaGFKsWTLc5y+68DRW2M9t7hl3sp2SWHxddehjToWLRJbQ0dFJGtiLPXAFdBHSd8cGTwtO8iNXGnIJzNC3tqgnFzqCjRcaFhetcUDE5iYQ3YXk5iW4U7Ey0uymUyTIQ8zDc/XLZvgBQbR/yDKNNmcUKhVdOR5NRsGPFFmBzqm8jxKeNM+hqq9o+pnQHD29qdiEoEEfg2CRod0iPF8kUs+hYtSkHZJtx2970CtTtGK/kDXuLQ5ROyZ/81Uns89cX0VEDFFXLUmM6a3k9XjmyOYKbEB3Wp8JfO/39Qs3Cl7IBuVds10dYsvBUBQUytp0S2nUWiK7/RtKHewK67y6la5bZJ+MXQTVgPow2gNJWHs3FHXd2wOZunOQOw+qp9CVX5nl8tWze/rumdn4R8B14DyrtBaSmiuIdWiS4XnKHMV249n1W5R6daCwWw3zR43VJsDkwSVNkM3BGykbWprkWGJ3Yy22W3V++s7Ie5ouez5xXSHh/NrCmqTlU43IeMx5sZafsuoaj218c72xE1mPL47TWhf1Hy9YdrNwOve1xZecWz6xmf9TRFZ/ZO3pplxtPt2N4w3NknfMH5nMOjltm00HvNpf7+GrZ3L3aXsLw8sUEX+rYWlN3+nBqdaP0QLGfddRp5+9zz5TNtiPpWUC/CAzp/EMCKS7K5p+91aKfIODFCvdzzgjUSvXFQIzQhUGPjnzVAdJW6BU1i5XfQv9R3Mm319D1uu2OkAWhZhB3qG1Ey+1Ih13MJ4P3I++VyK4/m5K4fsW03CI/W6OTK+7G7ivAy3W/3nJNROHuVCYNXndcq+OnEwuXmknshnoN3BzCptU1ucKa3r8UNX9mKneQji0cFjR4aGvz9i0ewdV76pC9v/o8GRQrl+CZgwM80yE4bDiRIrPHarfDs8kzu4tz8D6cwWcOlAVwHDzW2Mc8tjHxgnOMhLegE1Bz7M6DUynvPwkiWw2Iy/XXOIvH0vt+a/DGgO3wB9NBJMHB3J/h3n6jc4yMx4d7P9gDeHEJtg7NaLVY8b6imWe1LqTDGTuddl/0RwSjnuCxGTjs6ZxxxLK57FT5e3s2OgkHVUyd/wynCCzsTSTZnXN69sIT+TC3q+mIxZAvHHZijWw3OC+S2zPvunDZ/QLwTH7VH7OLg9lFZmKDmT8KrjnvlgGX3gmCbgTxxZ0x6k8CWjx3LoTkVlAv4iytXkPX3NdLPNrgnfBxbP0aWIH2xkvguCv37HtBXHGPj01dY/MoPFbWz8wKDw1Xws4WXHPlHUig4AMVglPq4GgbcBEYkBE53eNjeclHXEX3TvLKAG50zHV4Ks/+i9CaOmtelu4eW1NMZw8blTl3AhwkNQ3/knvLUPuCTkbHeoulpas+Li9Xj7jK7BKmi7cWmW7X/CV0Pzc7lXSUDaIrUBQI4630l/AK3CdNm/H0xC4UZ3gvT0e9i5QUm0/025muvuyGQdfRkdmlU2bnu8UVhicuN7svoNhpOnALnF6nV7mFuM7skimzw4VZPCiaymsM71I6Oj94lA7PU5BqN82LR2Mv2uXTW8euNbzLamOY5dU7olOTS3i0sR3p3r2HX79/Ybsz0W6SThCdAsMzwn4pus0WBtAdEO7N81G6hZLb3bZ6qRoFt2By57zjnb2Tuylk0u9fBd/4+AJ09hQ3th6OOQUtHdMyFd3awJ4D+8Z0f5Kz4pY1OmSvpjfL5Cy7D2YLanZBOXC+7Gqgo45cgmI5smWhRDrJdxVAuvW3p1svmwewuAOvLaqJ8ql3C3NwYaF2DR2I+JZ0dMtAZSbZim/eMSm732hOpOyse/sA3+0UX5Zl+CfLLqBb8tajXORKOTo9KTrJqRZcQpelOTzHHU/sryPDOvsbPG7oJ+JqbLcg+z47s6vdNA9bYMtxK7FWrd1UGmejEorWQhWdltI4aYOwV1PnP5toWWT/yDzRnUFnbhmodE6neBGPCzgR3USOLov7pbpE4eFQ2vwNgqZWipNddntLD7e3r2EIfsBXww+38DtnrPVkn6itQ7KQf6cw1o7o+hMuSIcLVK3C42ma2gbUsfhfRjSMEZ6ICf4N3HPp7vFmVQVO7/FQSamNY4R0wbsTHeElqtwrXGAucMdZ/lrMHK9n0+FKCrcIleq8m52ofvvd4LvjRh6zvNdi1qPNeoh3N5dOZLNXjyvcuMTvr8K9G7ED+Eyn3BZ5SHxpavHmocGbzqSjrsQqXdzw+/NOUt5DBj4bf/dc0hc58NqtKm4sXvy419AAX4PDzN4XsJHuy4PtfUhMyzUtJg6Q4XK6IjsASXfpQq3cq6dfYZz1ljwZ48tcugpDr/v2VECRj0Cg5bM95d3dwKBK3uVTcE/K6nV8mooCyWuH5KeN+XTpquhv61WwFPCoIo27mKnTppnUNPLswJ2YPReqz0dac+awAVnMTBOZ+RxPvciGcEfMiG4twz3Q/oXupNdtZqW0of/g/8uqgE3zqIzEUtfT1A7MxPnXE4D+K92p1deY0LI/sia7ukaBLP1a+23h/efcGYzVBGcnzk0+aa79l+ZPqFH8/T+yr1hBZbfV9t1ut3uHN/X2c7XVT4NN2H825nmTRzmJ/ueHH97ttvDSHx5vzfXN5vRH/h+BeMSwZBVc8AAAAABJRU5ErkJggg=="

# V is a mean reward, and a reward means nothing until you say for whom. The two
# characters carry that "for whom" through the whole panel: blue X and red O.
#
# Every number below is computed by calling reward_for, so the table cannot
# drift away from the rules cell above it.

_CASES = (
    (tuple("XXXOO...."), "X takes the top row"),
    (tuple("XOXXOOOXX"), "nobody wins, the board just fills"),
    (tuple("X.XOOOX.."), "O takes the middle row"),
)

_GOOD, _BAD, _MEH = "#15803d", "#b91c1c", "#b45309"


def _pov_color(v):
    if v > 0.75:
        return _GOOD
    if v < 0.25:
        return _BAD
    return _MEH


def _avatar(uri, px):
    return ('<img src="%s" alt="" style="height:%dpx;width:auto;display:block;"/>'
            % (uri, px))


def _pov_line(uri, mark, value):
    return (
        '<div style="display:flex;align-items:center;justify-content:center;'
        'gap:7px;margin-top:6px;">%s'
        '<span style="%sfont-size:11px;color:#64748b;">%s gets</span>'
        '<span style="%sfont-size:14px;font-weight:800;color:%s;">%.1f</span>'
        '</div>'
    ) % (_avatar(uri, 26), MONO, mark, MONO, _pov_color(value), value)


def _case(state, caption):
    return (
        '<div style="flex:1 1 168px;min-width:150px;text-align:center;padding:0 8px;">'
        '<div style="display:flex;justify-content:center;">%s</div>'
        '<div style="font-size:10.5px;color:#64748b;margin-top:7px;'
        'line-height:1.35;height:28px;">%s</div>%s%s</div>'
    ) % (board_html(state, 34), caption,
         _pov_line(_IMG_X, "X", reward_for(state, X)),
         _pov_line(_IMG_O, "O", reward_for(state, O)))


_cases_row = (
    '<div style="display:flex;flex-wrap:wrap;justify-content:center;'
    'margin:14px 0 4px;">' + "".join(_case(s, c) for s, c in _CASES) + '</div>')


def _chain_row(uri, mark, moved, expr, value, note):
    return (
        '<div style="display:flex;align-items:center;gap:12px;padding:9px 12px;'
        'border-radius:11px;background:rgba(148,163,184,0.10);flex-wrap:wrap;">'
        '%s<div style="flex:1 1 210px;">'
        '<div style="%sfont-size:12px;font-weight:700;color:#334155;">%s</div>'
        '<div style="font-size:10.5px;color:#64748b;margin-top:2px;">%s</div></div>'
        '<div style="text-align:right;">'
        '<div style="%sfont-size:11px;color:#64748b;">%s</div>'
        '<div style="%sfont-size:16px;font-weight:800;color:%s;">%.1f</div>'
        '</div></div>'
    ) % (_avatar(uri, 46), MONO, moved, note, MONO, expr, MONO,
         _pov_color(value), value)


_up = ('<div style="text-align:center;color:#94a3b8;font-size:15px;'
       'line-height:1.1;margin:3px 0;">&#8593;</div>')

_chain = (
    _chain_row(_IMG_X, "X", "X moved into this node", "r", 1.0,
               "two moves above the leaf, X's turn again")
    + _up
    + _chain_row(_IMG_O, "O", "O moved into this node", "1 - r", 0.0,
                 "the leaf's parent, and the other player's node")
    + _up
    + _chain_row(_IMG_X, "X", "X moved into this node", "r", 1.0,
                 "the leaf the playout started from"))

# ==== a node in the middle of a game, where there is no result to read =============
#
# Everything here is measured, not asserted: the playouts really run, seeded, so
# the chips, the curve and the two means below are recomputed every time the cell
# runs and cannot drift. They are quoted nowhere else.

_MID = tuple("X...O...X")          # X in two corners, O in the middle, 6 moves left
_MID_MOVER = other_player(player_to_move(_MID))
_SHOWN, _MANY = 8, 200


def _playout(state, rng):
    """Random legal moves to the end. The same shape as the rollout in part 2."""
    if game_over(state):
        return state
    return _playout(env.step(state, rng.choice(legal_actions(state))), rng)


def _sample(n):
    rng = random.Random(0)
    out = []
    for i in range(n):
        out.append(reward_for(_playout(_MID, rng), _MID_MOVER))
    return out


_rewards = _sample(_SHOWN)
_running = []
for i in range(len(_rewards)):
    _running.append(sum(_rewards[:i + 1]) / float(i + 1))
_settled = sum(_sample(_MANY)) / float(_MANY)

_ENDING = {1.0: "X wins", 0.5: "a draw", 0.0: "O wins"}


def _chip(i, r):
    return (
        '<div style="flex:1 1 76px;min-width:70px;text-align:center;padding:7px 4px;'
        'border-radius:9px;background:rgba(148,163,184,0.10);">'
        '<div style="%sfont-size:9px;color:#94a3b8;">playout %d</div>'
        '<div style="font-size:9.5px;color:#64748b;margin-top:2px;">%s</div>'
        '<div style="%sfont-size:15px;font-weight:800;color:%s;margin-top:2px;">'
        '%.1f</div></div>'
    ) % (MONO, i + 1, _ENDING[r], MONO, _pov_color(r), r)


def _stxt(x, y, s, size, color, weight="400", anchor="middle"):
    return ('<text x="%s" y="%s" text-anchor="%s" font-size="%s" font-weight="%s" '
            'fill="%s" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,'
            'monospace">%s</text>') % (x, y, anchor, size, weight, color, s)


def _curve():
    """The running mean after each playout, as a small step chart."""
    x0, x1, ytop, ybot = 42, 452, 16, 102
    span = float(len(_running) - 1)
    pts = []
    for i, v in enumerate(_running):
        pts.append((x0 + i * (x1 - x0) / span, ybot - v * (ybot - ytop)))
    out = ['<svg viewBox="0 0 470 134" style="width:470px;max-width:100%;'
           'height:auto;">']
    for v, label in ((1.0, "1.0"), (0.5, "0.5"), (0.0, "0.0")):
        y = ybot - v * (ybot - ytop)
        out.append('<line x1="%d" y1="%.1f" x2="%d" y2="%.1f" stroke="#e2e8f0" '
                   'stroke-width="1"/>' % (x0, y, x1, y))
        out.append(_stxt(x0 - 8, y + 3.5, label, "9.5", "#94a3b8", "400", "end"))
    ysettled = ybot - _settled * (ybot - ytop)
    out.append('<line x1="%d" y1="%.1f" x2="%d" y2="%.1f" stroke="#94a3b8" '
               'stroke-width="1.6" stroke-dasharray="5 4"/>'
               % (x0, ysettled, x1, ysettled))
    out.append('<polyline fill="none" stroke="%s" stroke-width="2.4" '
               'stroke-linejoin="round" points="%s"/>'
               % (MARK_COLOR[X], " ".join("%.1f,%.1f" % p for p in pts)))
    for i, (px, py) in enumerate(pts):
        out.append('<circle cx="%.1f" cy="%.1f" r="3.4" fill="#ffffff" '
                   'stroke="%s" stroke-width="2"/>' % (px, py, MARK_COLOR[X]))
        out.append(_stxt(px, 120, str(i + 1), "9", "#94a3b8"))
    out.append(_stxt(x1, ysettled + 15, "%.3f after %d playouts"
                     % (_settled, _MANY), "9.5", "#64748b", "400", "end"))
    out.append(_stxt(247, 133, "playouts that have come back through this node",
                     "9", "#94a3b8"))
    out.append("</svg>")
    return "".join(out)


_mid_block = (
    '<div style="display:flex;align-items:center;justify-content:center;gap:16px;'
    'flex-wrap:wrap;margin:14px 0 0;">'
    '<div style="text-align:center;">' + board_html(_MID, 44)
    + '<div style="font-size:10.5px;color:#64748b;width:150px;margin:7px auto 0;'
      'line-height:1.4;">X moved in, and nothing is settled. Six moves still to '
      'play, and no result to read.</div></div>'
    '<div style="flex:1 1 300px;min-width:280px;">'
    '<div style="display:flex;flex-wrap:wrap;gap:5px;">'
    + "".join(_chip(i, r) for i, r in enumerate(_rewards)) + '</div></div></div>'
    '<div style="display:flex;justify-content:center;margin-top:12px;">'
    + _curve() + '</div>'
    '<div style="font-size:11.5px;color:#64748b;margin-top:8px;line-height:1.5;">'
    'Watch the first three. All of them went X\'s way, so after three playouts '
    'this node claimed <b>V = 1.000</b>, and it was simply not true. Eight '
    'playouts later it says <b>' + ("%.3f" % _running[-1]) + '</b>, and after '
    + str(_MANY) + ' it has settled near <b>' + ("%.3f" % _settled) + '</b>. '
    'Meanwhile N just counts them: it starts at 1 and is '
    + str(1 + _SHOWN) + ' once eight results have come home. A high V off a '
    'couple of lucky playouts is a rumour, and that is exactly why the finished '
    'search plays the <b>most visited</b> child rather than the best scoring one.'
    '</div>')

display(HTML(
    '<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;'
    'border-radius:16px;padding:18px 16px 16px;margin:8px 0;">'
    '<div style="font-size:17px;font-weight:800;color:#1e293b;">'
    'Meet the two points of view</div>'
    '<div style="font-size:12px;color:#64748b;margin-top:5px;line-height:1.5;">'
    'A reward is not a property of a position. It is an opinion, and every '
    'opinion belongs to somebody. Ask these two how a finished game went and '
    'you get two different answers about the very same board.</div>'
    + _cases_row
    + '<div style="font-size:11.5px;color:#64748b;margin-top:10px;'
      'line-height:1.5;text-align:center;">'
      'That is all <span style="' + MONO + 'font-weight:700;color:#334155;">'
      'reward_for(state, player)</span> does: 1.0 to the winner, 0.0 to the '
      'loser, 0.5 each when nobody wins.</div>'
    '<div style="font-size:11px;font-weight:700;color:#475569;margin-top:18px;'
    'letter-spacing:0.04em;text-transform:uppercase;">'
    'So whose opinion does a node store</div>'
    '<div style="font-size:11.5px;color:#64748b;margin:6px 0 12px;'
    'line-height:1.5;">'
    'The player who <b>moved into it</b>. Follow one playout home. It ended with '
    'X winning, so it is worth <b>1.0</b> to X and <b>0.0</b> to O, and the nodes '
    'it passes through belong to X and O by turns. Read from the bottom up.</div>'
    + _chain
    # ==== PARKED, not shown =========================================================
    # The "one playout is not an opinion yet" section: the mid game node, the eight
    # seeded playouts, the running mean curve. Uncomment these five pieces to bring
    # it back. Everything it needs is still computed above, under the banner
    # "a node in the middle of a game".
    #
    # + '<div style="font-size:11px;font-weight:700;color:#475569;margin-top:20px;'
    #   'letter-spacing:0.04em;text-transform:uppercase;">'
    #   'But one playout is not an opinion yet</div>'
    # + '<div style="font-size:11.5px;color:#64748b;margin:6px 0 0;line-height:1.5;">'
    #   'Every node so far has been a finished game, where the reward is simply the '
    #   'result. Most nodes are nothing like that. Here is one with six moves still '
    #   'to play, so there is no result to read off it at all. What the search does '
    #   'instead is play the rest out at random, over and over, and keep the '
    #   '<b>running mean</b> of what comes back. That mean is V.</div>'
    # + _mid_block
    # ================================================================================
    + '<div style="margin-top:14px;padding:11px 14px;border-radius:10px;'
      'background:rgba(59,130,246,0.10);border-left:3px solid ' + MARK_COLOR[X] + ';'
      'font-size:12.5px;color:#1d4ed8;font-weight:600;line-height:1.5;">'
      'V is a running mean of results, always read from the side of whoever '
      'moved in, and it flips at every step on the way up. That single '
      'alternation is what lets one tree play both sides, and in part 3 it comes '
      'down to one short expression, which you will write yourself.</div>'
    '</div>'
))


You have two things to write here.

**`__init__` sets six fields**, and the comment names each one and what it starts as. Two
of them come straight from the arguments, `children` starts empty, `untried` starts as
every legal action, and the two counters start at `N = 1` and `V = 0.0`.

**`UCT` is the formula above, in two lines.** `exploit` is this node's `V`. `explore` is
`w` times the square root of the parent's log count over this node's count. In Python,
`math.sqrt` and `math.log` are the two you need, and `math.log` is already the natural
logarithm.


In [ ]:
# @title A score, plus a bonus for being under-explored { display-mode: "form" }
# Step through a real run of the selection rule. The walk was simulated at build
# time from the formula itself, so this is the rule's own behaviour.

display(HTML("""
<div class="uct">
  <div class="uct-h">A dot for what we know, a line for what we do not</div>
  <div class="uct-s">Three moves from one position. The <b>dot</b> is the average
    that move has earned so far. The <b>line above it</b> is how much better it
    could still turn out to be, and it is long exactly when a move has barely
    been tried. The search takes the move whose line <b>reaches highest</b>,
    not the one with the highest dot. Press <b>next</b> to spend one pass.</div>
  <div class="uct-f">
    <div class="uct-feq">
      <svg viewBox="0 0 300 96" style="width:300px;max-width:100%;height:auto">
        <text x="6" y="58" font-family="Georgia, 'Times New Roman', serif"
              font-size="19" font-weight="700" fill="#334155">UCT</text>
        <text x="58" y="58" font-family="Georgia, 'Times New Roman', serif"
              font-size="19" fill="#94a3b8">=</text>
        <text x="82" y="58" font-family="Georgia, 'Times New Roman', serif"
              font-size="20" font-style="italic" font-weight="700"
              fill="#3b82f6">V</text>
        <text x="102" y="58" font-family="Georgia, 'Times New Roman', serif"
              font-size="19" fill="#94a3b8">+</text>
        <text x="126" y="58" font-family="Georgia, 'Times New Roman', serif"
              font-size="19" font-style="italic" fill="#60a5fa">w</text>
        <path d="M148,53 L154,53 L162,76 L168,25 L276,25" fill="none"
              stroke="#60a5fa" stroke-width="2" stroke-linejoin="round"
              stroke-linecap="round"/>
        <text x="222" y="47" text-anchor="middle"
              font-family="Georgia, 'Times New Roman', serif" font-size="14"
              fill="#60a5fa">log&#8201;<tspan font-style="italic">N</tspan><tspan
              font-size="9.5" dy="3.5">parent</tspan></text>
        <line x1="178" y1="54" x2="266" y2="54" stroke="#60a5fa"
              stroke-width="1.4"/>
        <text x="222" y="72" text-anchor="middle"
              font-family="Georgia, 'Times New Roman', serif" font-size="14"
              font-style="italic" fill="#60a5fa">N</text>
      </svg>
    </div>
    <div class="uct-fk"><span class="uct-kd"></span>
      <span class="uct-fv">V</span> is the <b>dot</b>, the average this move has
      earned so far.</div>
    <div class="uct-fk"><span class="uct-kl"></span>
      <span class="uct-fb">the rest</span> is the <b>line</b> above it, and it holds
      two different counts.</div>
    <div class="uct-fc">
      <div><span class="uct-fn">N<sub>parent</sub></span> written <span class="uct-fn2">parent.N</span> in code: every pass spent at this
        position, <b>including the passes spent on the other moves</b></div>
      <div><span class="uct-fn">N</span> written <span class="uct-fn2">self.N</span> in code: only the passes that came to this move</div>
    </div>
    <div class="uct-fw">Visiting a move raises only the second count, which is why
      its line drops. But <b>parent.N rises on every single pass</b>, so a move you
      did not pick has its line creep <b>upward</b> while it waits. Nothing has to
      remember to go back and check it; the waiting does that by itself.</div>
  </div>
  <div class="uct-ctrl">
    <button class="uct-b alt" onclick="uctGo(-999)">reset</button>
    <button class="uct-b alt" onclick="uctGo(-1)">&#9664; prev</button>
    <button class="uct-b" id="uctPlay" onclick="uctPlay()">&#9654; play</button>
    <button class="uct-b alt" onclick="uctGo(1)">next &#9654;</button>
    <span class="uct-count" id="uctCount"></span>
  </div>
  <div class="uct-wrap">
    <div class="uct-axis" id="uctAxis"></div>
    <div class="uct-plot" id="uctPlot">
      <div class="uct-best" id="uctBest"></div>
    </div>
  </div>
  <div class="uct-feet" id="uctFeet"></div>
  <div class="uct-why" id="uctWhy"></div>
  <div class="uct-note">A move is never dropped for looking bad early, only for
    looking bad after a fair look. The line is what buys that look, and it shrinks
    on its own as the evidence arrives, so nothing has to decide when to stop being
    curious. Early on a line can reach past 1.0, which simply means the move has
    not been tried enough to rule anything out yet.</div>
  <div class="uct-cta">&#128071; Now build one. The next cell is the node
    with nothing inside it yet: six fields to fill in, and this formula in
    two lines.</div>
</div>
<style>
.uct{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;border-radius:16px;padding:18px 16px 16px;margin:8px 0;color:#1e293b}
.uct-h{font-size:17px;font-weight:800}
.uct-s{font-size:12px;color:#64748b;margin-top:5px;line-height:1.5}
.uct-f{margin-top:13px;padding:12px 14px;border-radius:11px;background:#fff;border:1px solid #e2e8f0}
.uct-feq{text-align:center;margin-bottom:10px}
.uct-fv{color:#3b82f6}
.uct-fb{color:#60a5fa}
.uct-fk{font-size:11px;color:#64748b;line-height:1.5;display:flex;align-items:baseline;gap:8px;margin-top:3px}
.uct-kd{flex:0 0 11px;width:11px;height:11px;border-radius:50%;background:#3b82f6;display:inline-block}
.uct-kl{flex:0 0 11px;width:3px;height:13px;border-radius:2px;background:#60a5fa;display:inline-block;margin-left:4px}
.uct-fc{margin-top:9px;padding-top:9px;border-top:1px solid #f1f5f9;font-size:11px;color:#64748b;line-height:1.6}
.uct-fn{font-family:Georgia,'Times New Roman',serif;font-style:italic;font-weight:700;color:#334155;display:inline-block;min-width:56px}
.uct-fn2{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:10px;color:#64748b}
.uct-fw{font-size:11px;color:#475569;line-height:1.55;margin-top:9px;padding-top:9px;border-top:1px solid #f1f5f9}
.uct-move{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:9.5px;margin-top:3px}
.uct-ctrl{display:flex;gap:7px;flex-wrap:wrap;align-items:center;margin-top:13px}
.uct-b{cursor:pointer;border:none;border-radius:9px;padding:7px 12px;font-size:11.5px;font-weight:700;color:#fff;background:#3b82f6;font-family:inherit}
.uct-b.alt{background:#fff;color:#475569;border:1px solid #cbd5e1}
.uct-count{font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:11px;color:#64748b;margin-left:4px}
.uct-wrap{display:flex;margin-top:18px}
.uct-axis{position:relative;width:44px;height:250px;flex:0 0 44px}
.uct-tick{position:absolute;right:6px;transform:translateY(-50%);font-family:ui-monospace,Menlo,Consolas,monospace;font-size:9.5px;color:#94a3b8}
.uct-plot{position:relative;flex:1 1 auto;height:250px;display:flex;border-left:1px solid #e2e8f0}
.uct-grid{position:absolute;left:0;right:0;height:1px;background:#e2e8f0}
.uct-best{position:absolute;left:0;right:0;height:0;border-top:1.5px dashed #93c5fd;transition:top .3s ease;z-index:0}
.uct-lane{position:relative;flex:1 1 0;min-width:82px}
.uct-whisk{position:absolute;left:50%;width:3px;margin-left:-1.5px;border-radius:2px;background:#93c5fd;transition:top .3s ease,height .3s ease}
.uct-cap{position:absolute;left:50%;width:30px;margin-left:-15px;height:3px;border-radius:2px;background:#93c5fd;transition:top .3s ease}
.uct-dot{position:absolute;left:50%;width:15px;height:15px;margin-left:-7.5px;margin-top:-7.5px;border-radius:50%;background:#3b82f6;box-shadow:0 0 0 3px rgba(255,255,255,.9);transition:top .3s ease}
.uct-top{position:absolute;left:50%;transform:translateX(-50%);margin-top:-19px;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:11.5px;font-weight:800;transition:top .3s ease;white-space:nowrap}
.uct-feet{display:flex;margin-left:44px}
.uct-foot{flex:1 1 0;min-width:82px;text-align:center;padding-top:8px}
.uct-name{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:11.5px;font-weight:700}
.uct-meta{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:10px;color:#94a3b8;margin-top:2px;line-height:1.45}
.uct-pill{display:inline-block;margin-top:6px;padding:2px 9px;border-radius:999px;font-size:9px;font-weight:700;font-family:ui-monospace,Menlo,Consolas,monospace}
.uct-why{font-size:11.5px;color:#475569;line-height:1.5;margin-top:14px;padding:10px 13px;border-radius:10px;background:#fff;border:1px solid #e2e8f0;min-height:34px}
.uct-note{font-size:11px;color:#94a3b8;margin-top:11px;line-height:1.5}
.uct-cta{margin-top:14px;padding:11px 14px;border-radius:10px;background:rgba(59,130,246,0.10);border-left:3px solid #3b82f6;font-size:12.5px;color:#1d4ed8;font-weight:600;line-height:1.5}
</style>
<script>
(function(){
 var D = {"top":2.4,"moves":[{"name":"move A","v":0.75},{"name":"move B","v":0.62},{"name":"move C","v":0.5}],"steps":[{"n":[1,1,1],"u":[1.467406,1.467406,1.467406],"t":[2.217406,2.087406,1.967406],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.0,0.0,0.0]},{"n":[2,1,1],"u":[1.165576,1.648374,1.648374],"t":[1.915576,2.268374,2.148374],"pick":1,"why":"move B is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.30183,0.180968,0.180968]},{"n":[2,2,1],"u":[1.255886,1.255886,1.776091],"t":[2.005886,1.875886,2.276091],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[0.09031,-0.392488,0.127717]},{"n":[2,2,2],"u":[1.325113,1.325113,1.325113],"t":[2.075113,1.945113,1.825113],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.069227,0.069227,-0.450978]},{"n":[3,2,2],"u":[1.127532,1.380939,1.380939],"t":[1.877532,2.000939,1.880939],"pick":1,"why":"move B is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.197581,0.055826,0.055826]},{"n":[3,3,2],"u":[1.165576,1.165576,1.427534],"t":[1.915576,1.785576,1.927534],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[0.038044,-0.215363,0.046595]},{"n":[3,3,3],"u":[1.198132,1.198132,1.198132],"t":[1.948132,1.818132,1.698132],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.032556,0.032556,-0.229402]},{"n":[4,3,3],"u":[1.062199,1.226522,1.226522],"t":[1.812199,1.846522,1.726522],"pick":1,"why":"move B is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.135933,0.02839,0.02839]},{"n":[4,4,3],"u":[1.08396,1.08396,1.251649],"t":[1.83396,1.70396,1.751649],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.021761,-0.142562,0.025127]},{"n":[5,4,3],"u":[0.986957,1.103451,1.274156],"t":[1.736957,1.723451,1.774156],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.097003,0.019491,0.022507]},{"n":[5,4,4],"u":[1.002726,1.121082,1.121082],"t":[1.752726,1.741082,1.621082],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.015769,0.017631,-0.153074]},{"n":[6,4,4],"u":[0.928489,1.137162,1.137162],"t":[1.678489,1.757162,1.637162],"pick":1,"why":"move B is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.074237,0.01608,0.01608]},{"n":[6,5,4],"u":[0.940548,1.030318,1.151931],"t":[1.690548,1.650318,1.651931],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.012059,-0.106844,0.014769]},{"n":[7,5,4],"u":[0.881093,1.042523,1.165576],"t":[1.631093,1.662523,1.665576],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.059455,0.012205,0.013645]},{"n":[7,5,5],"u":[0.890674,1.053859,1.053859],"t":[1.640674,1.673859,1.553859],"pick":1,"why":"move B is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[0.009581,0.011336,-0.111717]},{"n":[7,6,5],"u":[0.899613,0.971693,1.064437],"t":[1.649613,1.591693,1.564437],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.008939,-0.082166,0.010578]},{"n":[8,6,5],"u":[0.849345,0.98074,1.074346],"t":[1.599345,1.60074,1.574346],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[-0.050268,0.009047,0.009909]},{"n":[8,7,5],"u":[0.856711,0.915863,1.083664],"t":[1.606711,1.535863,1.583664],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.007366,-0.064877,0.009318]},{"n":[9,7,5],"u":[0.814266,0.923291,1.092453],"t":[1.564266,1.543291,1.592453],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.042445,0.007428,0.008789]},{"n":[9,7,6],"u":[0.820464,0.930318,1.004858],"t":[1.570464,1.550318,1.504858],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.006198,0.007027,-0.087595]},{"n":[10,7,6],"u":[0.783937,0.936984,1.012058],"t":[1.533937,1.556984,1.512058],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[-0.036527,0.006666,0.0072]},{"n":[10,8,6],"u":[0.789239,0.882396,1.018903],"t":[1.539239,1.502396,1.518903],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.005302,-0.054588,0.006845]},{"n":[11,8,6],"u":[0.757328,0.888045,1.025426],"t":[1.507328,1.508045,1.525426],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[-0.031911,0.005649,0.006523]},{"n":[11,8,7],"u":[0.761928,0.893439,0.955127],"t":[1.511928,1.513439,1.455127],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[0.0046,0.005394,-0.070299]},{"n":[11,9,7],"u":[0.766328,0.847207,0.960643],"t":[1.516328,1.467207,1.460643],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.0044,-0.046232,0.005516]},{"n":[12,9,7],"u":[0.73774,0.851869,0.965928],"t":[1.48774,1.471869,1.465928],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[-0.028588,0.004662,0.005285]},{"n":[13,9,7],"u":[0.71252,0.856342,0.971001],"t":[1.46252,1.476342,1.471001],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[-0.02522,0.004473,0.005073]},{"n":[13,10,7],"u":[0.716098,0.816477,0.975877],"t":[1.466098,1.436477,1.475877],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[0.003578,-0.039865,0.004876]},{"n":[13,10,8],"u":[0.719541,0.820403,0.917239],"t":[1.469541,1.440403,1.417239],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.003443,0.003926,-0.058638]},{"n":[14,10,8],"u":[0.696565,0.824187,0.921469],"t":[1.446565,1.444187,1.421469],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[-0.022976,0.003784,0.00423]},{"n":[15,10,8],"u":[0.675927,0.827838,0.925551],"t":[1.425927,1.447838,1.425551],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[-0.020638,0.003651,0.004082]},{"n":[15,11,8],"u":[0.678806,0.792675,0.929494],"t":[1.428806,1.412675,1.429494],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[0.002879,-0.035163,0.003943]},{"n":[15,11,9],"u":[0.68159,0.795926,0.879929],"t":[1.43159,1.415926,1.379929],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.002784,0.003251,-0.049565]},{"n":[16,11,9],"u":[0.662556,0.799073,0.883409],"t":[1.412556,1.419073,1.383409],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[-0.019034,0.003147,0.00348]},{"n":[16,12,9],"u":[0.665085,0.767973,0.886779],"t":[1.415085,1.387973,1.386779],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.002529,-0.0311,0.00337]},{"n":[17,12,9],"u":[0.647605,0.770804,0.890048],"t":[1.397605,1.390804,1.390048],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[-0.01748,0.002831,0.003269]},{"n":[18,12,9],"u":[0.631602,0.773551,0.89322],"t":[1.381602,1.393551,1.39322],"pick":1,"why":"move B is not the best on average, but it has been tested less, and the doubt left on it is worth more than the gap.","d":[-0.016003,0.002747,0.003172]},{"n":[18,13,9],"u":[0.633781,0.745768,0.896301],"t":[1.383781,1.365768,1.396301],"pick":2,"why":"move C is the least tested of the three, so most of its column is doubt. That alone sends the search there, even though its average is not the best.","d":[0.002179,-0.027783,0.003081]},{"n":[18,13,10],"u":[0.635898,0.74826,0.853147],"t":[1.385898,1.36826,1.353147],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[0.002117,0.002492,-0.043154]},{"n":[19,13,10],"u":[0.620943,0.750683,0.855911],"t":[1.370943,1.370683,1.355911],"pick":0,"why":"move A has the best average and enough visits behind it. This one it wins on merit.","d":[-0.014955,0.002423,0.002764]}]};
 var H = 250.0, k = 0, timer = null;
 var plot = document.getElementById("uctPlot");
 var axis = document.getElementById("uctAxis");
 var feet = document.getElementById("uctFeet");
 var best = document.getElementById("uctBest");
 var why = document.getElementById("uctWhy");
 var cnt = document.getElementById("uctCount");
 var playBtn = document.getElementById("uctPlay");

 function y(v){ return H - (v / D.top) * H; }

 var ticks = "";
 for (var g = 0; g <= D.top + 0.001; g += 0.5){
   ticks += '<div class="uct-tick" style="top:' + y(g) + 'px">' + g.toFixed(1) + '</div>';
 }
 axis.innerHTML = ticks;

 var lanes = "";
 for (var g2 = 0; g2 <= D.top + 0.001; g2 += 0.5){
   lanes += '<div class="uct-grid" style="top:' + y(g2) + 'px"></div>';
 }
 D.moves.forEach(function(m, i){
   lanes += '<div class="uct-lane">'
         + '<div class="uct-whisk" id="uctW' + i + '"></div>'
         + '<div class="uct-cap" id="uctCT' + i + '"></div>'
         + '<div class="uct-dot" id="uctD' + i + '"></div>'
         + '<div class="uct-top" id="uctL' + i + '"></div></div>';
 });
 plot.innerHTML = plot.innerHTML + lanes;

 feet.innerHTML = D.moves.map(function(m, i){
   return '<div class="uct-foot">'
        + '<div class="uct-name" id="uctN' + i + '">' + m.name + '</div>'
        + '<div class="uct-meta" id="uctM' + i + '"></div>'
        + '<div class="uct-move" id="uctX' + i + '"></div>'
        + '<div id="uctP' + i + '"></div></div>';
 }).join("");

 function render(){
   var st = D.steps[k], topmost = Math.max.apply(null, st.t);
   cnt.textContent = "pass " + (k + 1) + " of " + D.steps.length;
   best.style.top = y(topmost) + "px";
   for (var i = 0; i < D.moves.length; i++){
     var v = D.moves[i].v, hi = st.t[i];
     var won = (i === st.pick);
     var yh = y(Math.min(hi, D.top)), yv = y(v);
     document.getElementById("uctW" + i).style.top = yh + "px";
     document.getElementById("uctW" + i).style.height = (yv - yh) + "px";
     document.getElementById("uctCT" + i).style.top = yh + "px";
     document.getElementById("uctD" + i).style.top = y(v) + "px";
     var lab = document.getElementById("uctL" + i);
     lab.style.top = yh + "px";
     lab.textContent = hi.toFixed(2);
     lab.style.color = won ? "#1d4ed8" : "#94a3b8";
     var shade = won ? "#60a5fa" : "#cbd5e1";
     document.getElementById("uctW" + i).style.background = shade;
     document.getElementById("uctCT" + i).style.background = shade;
     document.getElementById("uctD" + i).style.background = won ? "#3b82f6" : "#94a3b8";
     document.getElementById("uctN" + i).style.color = won ? "#1e293b" : "#94a3b8";
     document.getElementById("uctM" + i).innerHTML =
        "tried " + st.n[i] + "<br>average " + v.toFixed(2);
     var mv = document.getElementById("uctX" + i), dz = st.d[i];
     if (k === 0){ mv.innerHTML = "&nbsp;"; }
     else if (dz < 0){ mv.innerHTML = '<span style="color:#1d4ed8">'
        + dz.toFixed(2) + ' &#8595; just tried</span>'; }
     else { mv.innerHTML = '<span style="color:#94a3b8">+'
        + dz.toFixed(2) + ' &#8593; while waiting</span>'; }
     document.getElementById("uctP" + i).innerHTML = won
        ? '<span class="uct-pill" style="color:#1d4ed8;background:rgba(59,130,246,.14)">the search goes here</span>'
        : '';
   }
   why.innerHTML = st.why;
 }

 window.uctGo = function(d){
   if (d === -999){ k = 0; stop(); } else {
     k = Math.max(0, Math.min(D.steps.length - 1, k + d));
   }
   render();
 };
 function stop(){ if (timer){ clearInterval(timer); timer = null; }
                  playBtn.innerHTML = "&#9654; play"; }
 window.uctPlay = function(){
   if (timer){ stop(); return; }
   playBtn.innerHTML = "&#9612;&#9612; pause";
   timer = setInterval(function(){
     if (k >= D.steps.length - 1){ stop(); return; }
     k += 1; render();
   }, 620);
 };
 render();
})();
</script>
"""))


In [ ]:
class Node:
    """One state in the tree, plus the counters the search keeps.

    state      the position
    parent     the node above, or None for the root
    children   the nodes below, empty until this node is expanded
    untried    legal actions here that have not been made into a child yet
    N          how many times the search passed through this node
    V          the running mean reward, from the point of view of the player
               who MOVED INTO this node
    """

    def __init__(self, state, parent):
        # 🎯 TARGET 1  Six assignments, one per line, in this order.

        # the position this node stands for, straight from the argument
        self.state = ...
        # the node one step up, also straight from the argument
        self.parent = ...
        # no children yet, so an empty list
        self.children = ...
        # every legal move here, none of them turned into a node yet
        self.untried = ...
        # born at 1, not 0, so that log(parent.N) is defined on the very first
        # pass and nothing has to divide by zero
        self.N = ...
        # no results have come back yet, so the mean starts at zero
        self.V = ...

    def UCT(self, w):
        """How attractive this node is to whoever is choosing at its parent.

            UCT = V + w sqrt( log(parent.N) / N )

        The first term is what we have seen, the second is what we do not yet
        know. Only ever called on a node that has a parent.
        """
        # 🎯 TARGET 2  Two assignments, then the return is already written.

        # exploit is just this node's V, self.V
        exploit = ...
        # explore is w times the square root of ( log of the PARENT's N,
        # divided by this node's N ). The parent's count is self.parent.N and
        # this node's is self.N. math.sqrt(x) is the square root and
        # math.log(x) is the natural log, which is the "log" the formula means.
        explore = ...
        return exploit + explore


check_node(Node)


## 2. The four phases

One pass of the search does four things, and each one is a short function.


In [ ]:
# @title One pass, four phases { display-mode: "form" }
# The four phases as a roadmap: what each one is called, what it does, and
# which of them you have to write. One trip round this loop is one pass.

_PHASES = (
    ("&#129517;", "#3b82f6", "1. Selection", "descend",
     "walk down to a node worth working on", True),
    ("&#127793;", "#14b8a6", "2. Expansion", "expand",
     "add exactly one new node", True),
    ("&#127922;", "#f59e0b", "3. Simulation", "rollout_reward",
     "play the rest at random, score it", False),
    ("&#11014;", "#8b5cf6", "4. Backpropagation", "backprop",
     "carry the score back up", True),
)


def _pill(mine):
    if mine:
        text, bg, fg = "\U0001F3AF you write this", "rgba(59,130,246,0.13)", "#1d4ed8"
    else:
        text, bg, fg = "given", "rgba(100,116,139,0.13)", "#64748b"
    return ('<div style="display:inline-block;margin-top:7px;padding:2px 8px;'
            'border-radius:999px;background:%s;color:%s;font-size:9.5px;'
            'font-weight:700;">%s</div>') % (bg, fg, text)


def _step(icon, color, title, fn, desc, mine):
    return (
        '<div style="flex:1 1 168px;min-width:158px;text-align:center;padding:0 8px;">'
        '<div style="width:54px;height:54px;border-radius:50px;margin:0 auto 9px;'
        'display:flex;align-items:center;justify-content:center;font-size:24px;'
        'color:#ffffff;background:linear-gradient(135deg,%s,%scc);">%s</div>'
        '<div style="font-weight:800;font-size:13px;color:#1e293b;">%s</div>'
        '<div style="%sfont-size:11px;font-weight:700;color:%s;margin-top:3px;">'
        '%s</div>'
        '<div style="font-size:10.5px;color:#64748b;margin-top:6px;'
        'line-height:1.4;">%s</div>%s</div>'
    ) % (color, color, icon, title, MONO, color, fn, desc, _pill(mine))


_gap = ('<div style="flex:0 0 18px;display:flex;align-items:center;'
        'justify-content:center;font-size:18px;color:#cbd5e1;">&#10140;</div>')

_row = _gap.join(_step(*p) for p in _PHASES)

display(HTML(
    '<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;'
    'border-radius:16px;padding:18px 16px 16px;margin:8px 0;">'
    '<div style="font-size:17px;font-weight:800;color:#1e293b;">'
    'One pass, four phases</div>'
    '<div style="font-size:12px;color:#64748b;margin-top:5px;line-height:1.5;">'
    'Four short functions, run over and over. Three of them are yours.</div>'
    '<div style="display:flex;align-items:flex-start;flex-wrap:wrap;'
    'margin:18px 0 0;">' + _row + '</div>'
    '<div style="margin-top:16px;padding-top:11px;border-top:1px dashed #cbd5e1;'
    'text-align:center;font-size:11.5px;color:#64748b;line-height:1.5;">'
    '&#8635; <b>and round again.</b> One node is added per pass, so the tree '
    'grows deepest where the search keeps coming back.</div>'
    '</div>'
))


Reading about a loop is not the same as watching one turn. Below is a real search, twelve
passes of it, recorded phase by phase so you can step through it before you have written
anything.


In [ ]:
# @title Step through twelve real passes { display-mode: "form" }
# Twelve real passes, recorded and replayed. Step through them one phase at a
# time and watch the tree grow.
#
# The trace below was produced at build time by running THIS notebook's own
# solved functions, so nothing here is a second, hand written imitation of the
# search: change a solution and the replay changes with it. Nothing runs in
# Python when you press a button, which is why this works before you have
# written a line.

display(HTML("""
<div class="ttp">
  <div class="ttp-h">Watch the loop turn</div>
  <div class="ttp-s">Twelve passes over the position X must not lose. Press
    <b>next</b> to advance one phase at a time, or <b>play</b> to let it run.
    Click any circle to look at the position it stands for.</div>
  <div class="ttp-ph" id="ttpPh"></div>
  <div class="ttp-ctrl">
    <button class="ttp-b alt" onclick="ttpGo(-999)">reset</button>
    <button class="ttp-b alt" onclick="ttpGo(-1)">&#9664; prev</button>
    <button class="ttp-b" id="ttpPlay" onclick="ttpPlay()">&#9654; play</button>
    <button class="ttp-b alt" onclick="ttpGo(1)">next &#9654;</button>
    <span class="ttp-count" id="ttpCount"></span>
  </div>
  <div class="ttp-main">
    <div class="ttp-left">
      <div id="ttpBoard"></div>
      <div class="ttp-cap" id="ttpCap"></div>
    </div>
    <div class="ttp-right">
      <svg id="ttpSvg" viewBox="0 0 600 272"
           style="width:100%;height:auto;min-width:340px"></svg>
      <div class="ttp-leg">Every circle is one node. The number inside it is the
        <b>cell that was played</b> to get there, R being the root we start from.
        <b>N</b> counts the results that have come back through it and <b>V</b> is
        their mean.</div>
    </div>
  </div>
  <div class="ttp-note" id="ttpNote"></div>
</div>
<style>
.ttp{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;border-radius:16px;padding:18px 16px 16px;margin:8px 0;color:#1e293b}
.ttp-h{font-size:17px;font-weight:800}
.ttp-s{font-size:12px;color:#64748b;margin-top:5px;line-height:1.5}
.ttp-ph{display:flex;gap:6px;flex-wrap:wrap;margin:14px 0 10px}
.ttp-c{flex:1 1 116px;text-align:center;padding:6px 4px;border-radius:9px;font-size:11px;font-weight:700;background:rgba(148,163,184,0.12);color:#94a3b8;transition:all .18s}
.ttp-ctrl{display:flex;gap:7px;flex-wrap:wrap;align-items:center}
.ttp-b{cursor:pointer;border:none;border-radius:9px;padding:7px 12px;font-size:11.5px;font-weight:700;color:#fff;background:#3b82f6;font-family:inherit}
.ttp-b.alt{background:#fff;color:#475569;border:1px solid #cbd5e1}
.ttp-count{font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:11px;color:#64748b;margin-left:4px}
.ttp-main{display:flex;gap:16px;flex-wrap:wrap;align-items:flex-start;margin-top:14px}
.ttp-left{flex:0 0 auto;text-align:center}
.ttp-right{flex:1 1 340px;min-width:300px}
.ttp-cap{font-size:10.5px;color:#64748b;width:150px;margin:8px auto 0;line-height:1.4}
.ttp-leg{font-size:10.5px;color:#64748b;line-height:1.45;margin-top:4px;padding:0 4px}
.ttp-note{font-size:11.5px;color:#475569;line-height:1.5;margin-top:12px;padding:10px 13px;border-radius:10px;background:#fff;border:1px solid #e2e8f0;min-height:34px}
.ttp-flip{display:flex;align-items:center;gap:7px;flex-wrap:wrap;margin-top:9px}
.ttp-fc{display:flex;align-items:center;gap:6px;background:rgba(148,163,184,0.13);border-radius:10px;padding:4px 10px 4px 6px}
.ttp-fc img{height:28px;width:auto;display:block}
.ttp-fc b{font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:13.5px}
.ttp-fa{color:#cbd5e1;font-size:13px}
.ttp-fl{font-size:10px;color:#94a3b8;font-weight:700;letter-spacing:.03em}
.ttp-fn{font-size:10.5px;color:#94a3b8;margin-top:7px;line-height:1.45}
</style>
<script>
(function(){
 var T = {"root":"XOX.O....","w":600,"h":288,"nodes":[{"i":0,"up":-1,"a":-1,"s":"XOX.O....","x":338.6,"y":34},{"i":1,"up":0,"a":3,"s":"XOXXO....","x":107.1,"y":130},{"i":2,"up":0,"a":5,"s":"XOX.OX...","x":261.4,"y":130},{"i":3,"up":0,"a":6,"s":"XOX.O.X..","x":338.6,"y":130},{"i":4,"up":0,"a":7,"s":"XOX.O..X.","x":454.3,"y":130},{"i":5,"up":0,"a":8,"s":"XOX.O...X","x":570.0,"y":130},{"i":6,"up":1,"a":5,"s":"XOXXOO...","x":30.0,"y":226},{"i":7,"up":2,"a":3,"s":"XOXOOX...","x":261.4,"y":226},{"i":8,"up":4,"a":3,"s":"XOXOO..X.","x":415.7,"y":226},{"i":9,"up":5,"a":3,"s":"XOXOO...X","x":570.0,"y":226},{"i":10,"up":1,"a":6,"s":"XOXXO.O..","x":107.1,"y":226},{"i":11,"up":4,"a":5,"s":"XOX.OO.X.","x":492.9,"y":226},{"i":12,"up":1,"a":7,"s":"XOXXO..O.","x":184.3,"y":226}],"steps":[{"p":1,"ph":1,"live":[0],"n":[1],"v":[0.0],"path":[0],"focus":0,"note":"The root still has moves nobody has tried, so this pass stops right here.","roll":null,"flip":null},{"p":1,"ph":2,"live":[0,1],"n":[1,1],"v":[0.0,0.0],"path":[0,1],"focus":1,"note":"Took one untried move, cell 3, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":1,"ph":3,"live":[0,1],"n":[1,1],"v":[0.0,0.0],"path":[0,1],"focus":1,"note":"Played the rest out at random: X wins, which is worth 1.0 to X, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXXO.X.O","r":1.0,"mover":"X"},"flip":null},{"p":1,"ph":4,"live":[0,1],"n":[2,2],"v":[0.0,0.5],"path":[0,1],"focus":1,"note":"Carried it back up through 2 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"X","r":1.0},{"m":"O","r":0.0}]},{"p":2,"ph":1,"live":[0,1],"n":[2,2],"v":[0.0,0.5],"path":[0],"focus":0,"note":"The root still has moves nobody has tried, so this pass stops right here.","roll":null,"flip":null},{"p":2,"ph":2,"live":[0,1,2],"n":[2,2,1],"v":[0.0,0.5,0.0],"path":[0,2],"focus":2,"note":"Took one untried move, cell 5, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":2,"ph":3,"live":[0,1,2],"n":[2,2,1],"v":[0.0,0.5,0.0],"path":[0,2],"focus":2,"note":"Played the rest out at random: a draw, which is worth 0.5 to X, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXOOXXXO","r":0.5,"mover":"X"},"flip":null},{"p":2,"ph":4,"live":[0,1,2],"n":[3,2,2],"v":[0.167,0.5,0.25],"path":[0,2],"focus":2,"note":"Carried it back up through 2 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"X","r":0.5},{"m":"O","r":0.5}]},{"p":3,"ph":1,"live":[0,1,2],"n":[3,2,2],"v":[0.167,0.5,0.25],"path":[0],"focus":0,"note":"The root still has moves nobody has tried, so this pass stops right here.","roll":null,"flip":null},{"p":3,"ph":2,"live":[0,1,2,3],"n":[3,2,2,1],"v":[0.167,0.5,0.25,0.0],"path":[0,3],"focus":3,"note":"Took one untried move, cell 6, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":3,"ph":3,"live":[0,1,2,3],"n":[3,2,2,1],"v":[0.167,0.5,0.25,0.0],"path":[0,3],"focus":3,"note":"Played the rest out at random: O wins, which is worth 0.0 to X, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOX.O.XO.","r":0.0,"mover":"X"},"flip":null},{"p":3,"ph":4,"live":[0,1,2,3],"n":[4,2,2,2],"v":[0.375,0.5,0.25,0.0],"path":[0,3],"focus":3,"note":"Carried it back up through 2 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"X","r":0.0},{"m":"O","r":1.0}]},{"p":4,"ph":1,"live":[0,1,2,3],"n":[4,2,2,2],"v":[0.375,0.5,0.25,0.0],"path":[0],"focus":0,"note":"The root still has moves nobody has tried, so this pass stops right here.","roll":null,"flip":null},{"p":4,"ph":2,"live":[0,1,2,3,4],"n":[4,2,2,2,1],"v":[0.375,0.5,0.25,0.0,0.0],"path":[0,4],"focus":4,"note":"Took one untried move, cell 7, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":4,"ph":3,"live":[0,1,2,3,4],"n":[4,2,2,2,1],"v":[0.375,0.5,0.25,0.0,0.0],"path":[0,4],"focus":4,"note":"Played the rest out at random: a draw, which is worth 0.5 to X, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXOOXXXO","r":0.5,"mover":"X"},"flip":null},{"p":4,"ph":4,"live":[0,1,2,3,4],"n":[5,2,2,2,2],"v":[0.4,0.5,0.25,0.0,0.25],"path":[0,4],"focus":4,"note":"Carried it back up through 2 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"X","r":0.5},{"m":"O","r":0.5}]},{"p":5,"ph":1,"live":[0,1,2,3,4],"n":[5,2,2,2,2],"v":[0.4,0.5,0.25,0.0,0.25],"path":[0],"focus":0,"note":"The root still has moves nobody has tried, so this pass stops right here.","roll":null,"flip":null},{"p":5,"ph":2,"live":[0,1,2,3,4,5],"n":[5,2,2,2,2,1],"v":[0.4,0.5,0.25,0.0,0.25,0.0],"path":[0,5],"focus":5,"note":"Took one untried move, cell 8, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":5,"ph":3,"live":[0,1,2,3,4,5],"n":[5,2,2,2,2,1],"v":[0.4,0.5,0.25,0.0,0.25,0.0],"path":[0,5],"focus":5,"note":"Played the rest out at random: a draw, which is worth 0.5 to X, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXXOOOXX","r":0.5,"mover":"X"},"flip":null},{"p":5,"ph":4,"live":[0,1,2,3,4,5],"n":[6,2,2,2,2,2],"v":[0.417,0.5,0.25,0.0,0.25,0.25],"path":[0,5],"focus":5,"note":"Carried it back up through 2 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"X","r":0.5},{"m":"O","r":0.5}]},{"p":6,"ph":1,"live":[0,1,2,3,4,5],"n":[6,2,2,2,2,2],"v":[0.417,0.5,0.25,0.0,0.25,0.25],"path":[0,1],"focus":1,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":6,"ph":2,"live":[0,1,2,3,4,5,6],"n":[6,2,2,2,2,2,1],"v":[0.417,0.5,0.25,0.0,0.25,0.25,0.0],"path":[0,1,6],"focus":6,"note":"Took one untried move, cell 5, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":6,"ph":3,"live":[0,1,2,3,4,5,6],"n":[6,2,2,2,2,2,1],"v":[0.417,0.5,0.25,0.0,0.25,0.25,0.0],"path":[0,1,6],"focus":6,"note":"Played the rest out at random: a draw, which is worth 0.5 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXXOOOXX","r":0.5,"mover":"O"},"flip":null},{"p":6,"ph":4,"live":[0,1,2,3,4,5,6],"n":[7,3,2,2,2,2,2],"v":[0.429,0.5,0.25,0.0,0.25,0.25,0.25],"path":[0,1,6],"focus":6,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":0.5},{"m":"X","r":0.5},{"m":"O","r":0.5}]},{"p":7,"ph":1,"live":[0,1,2,3,4,5,6],"n":[7,3,2,2,2,2,2],"v":[0.429,0.5,0.25,0.0,0.25,0.25,0.25],"path":[0,2],"focus":2,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":7,"ph":2,"live":[0,1,2,3,4,5,6,7],"n":[7,3,2,2,2,2,2,1],"v":[0.429,0.5,0.25,0.0,0.25,0.25,0.25,0.0],"path":[0,2,7],"focus":7,"note":"Took one untried move, cell 3, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":7,"ph":3,"live":[0,1,2,3,4,5,6,7],"n":[7,3,2,2,2,2,2,1],"v":[0.429,0.5,0.25,0.0,0.25,0.25,0.25,0.0],"path":[0,2,7],"focus":7,"note":"Played the rest out at random: O wins, which is worth 1.0 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXOOXXO.","r":1.0,"mover":"O"},"flip":null},{"p":7,"ph":4,"live":[0,1,2,3,4,5,6,7],"n":[8,3,3,2,2,2,2,2],"v":[0.5,0.5,0.167,0.0,0.25,0.25,0.25,0.5],"path":[0,2,7],"focus":7,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":1.0},{"m":"X","r":0.0},{"m":"O","r":1.0}]},{"p":8,"ph":1,"live":[0,1,2,3,4,5,6,7],"n":[8,3,3,2,2,2,2,2],"v":[0.5,0.5,0.167,0.0,0.25,0.25,0.25,0.5],"path":[0,4],"focus":4,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":8,"ph":2,"live":[0,1,2,3,4,5,6,7,8],"n":[8,3,3,2,2,2,2,2,1],"v":[0.5,0.5,0.167,0.0,0.25,0.25,0.25,0.5,0.0],"path":[0,4,8],"focus":8,"note":"Took one untried move, cell 3, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":8,"ph":3,"live":[0,1,2,3,4,5,6,7,8],"n":[8,3,3,2,2,2,2,2,1],"v":[0.5,0.5,0.167,0.0,0.25,0.25,0.25,0.5,0.0],"path":[0,4,8],"focus":8,"note":"Played the rest out at random: X wins, which is worth 0.0 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXOOXOXX","r":0.0,"mover":"O"},"flip":null},{"p":8,"ph":4,"live":[0,1,2,3,4,5,6,7,8],"n":[9,3,3,2,3,2,2,2,2],"v":[0.444,0.5,0.167,0.0,0.5,0.25,0.25,0.5,0.0],"path":[0,4,8],"focus":8,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":0.0},{"m":"X","r":1.0},{"m":"O","r":0.0}]},{"p":9,"ph":1,"live":[0,1,2,3,4,5,6,7,8],"n":[9,3,3,2,3,2,2,2,2],"v":[0.444,0.5,0.167,0.0,0.5,0.25,0.25,0.5,0.0],"path":[0,5],"focus":5,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":9,"ph":2,"live":[0,1,2,3,4,5,6,7,8,9],"n":[9,3,3,2,3,2,2,2,2,1],"v":[0.444,0.5,0.167,0.0,0.5,0.25,0.25,0.5,0.0,0.0],"path":[0,5,9],"focus":9,"note":"Took one untried move, cell 3, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":9,"ph":3,"live":[0,1,2,3,4,5,6,7,8,9],"n":[9,3,3,2,3,2,2,2,2,1],"v":[0.444,0.5,0.167,0.0,0.5,0.25,0.25,0.5,0.0,0.0],"path":[0,5,9],"focus":9,"note":"Played the rest out at random: O wins, which is worth 1.0 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXOOO.XX","r":1.0,"mover":"O"},"flip":null},{"p":9,"ph":4,"live":[0,1,2,3,4,5,6,7,8,9],"n":[10,3,3,2,3,3,2,2,2,2],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5],"path":[0,5,9],"focus":9,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":1.0},{"m":"X","r":0.0},{"m":"O","r":1.0}]},{"p":10,"ph":1,"live":[0,1,2,3,4,5,6,7,8,9],"n":[10,3,3,2,3,3,2,2,2,2],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5],"path":[0,1],"focus":1,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":10,"ph":2,"live":[0,1,2,3,4,5,6,7,8,9,10],"n":[10,3,3,2,3,3,2,2,2,2,1],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5,0.0],"path":[0,1,10],"focus":10,"note":"Took one untried move, cell 6, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":10,"ph":3,"live":[0,1,2,3,4,5,6,7,8,9,10],"n":[10,3,3,2,3,3,2,2,2,2,1],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5,0.0],"path":[0,1,10],"focus":10,"note":"Played the rest out at random: a draw, which is worth 0.5 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXXOXOXO","r":0.5,"mover":"O"},"flip":null},{"p":10,"ph":4,"live":[0,1,2,3,4,5,6,7,8,9,10],"n":[11,4,3,2,3,3,2,2,2,2,2],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5,0.25],"path":[0,1,10],"focus":10,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":0.5},{"m":"X","r":0.5},{"m":"O","r":0.5}]},{"p":11,"ph":1,"live":[0,1,2,3,4,5,6,7,8,9,10],"n":[11,4,3,2,3,3,2,2,2,2,2],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5,0.25],"path":[0,4],"focus":4,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":11,"ph":2,"live":[0,1,2,3,4,5,6,7,8,9,10,11],"n":[11,4,3,2,3,3,2,2,2,2,2,1],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5,0.25,0.0],"path":[0,4,11],"focus":11,"note":"Took one untried move, cell 5, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":11,"ph":3,"live":[0,1,2,3,4,5,6,7,8,9,10,11],"n":[11,4,3,2,3,3,2,2,2,2,2,1],"v":[0.5,0.5,0.167,0.0,0.5,0.167,0.25,0.5,0.0,0.5,0.25,0.0],"path":[0,4,11],"focus":11,"note":"Played the rest out at random: O wins, which is worth 1.0 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXOOO.XX","r":1.0,"mover":"O"},"flip":null},{"p":11,"ph":4,"live":[0,1,2,3,4,5,6,7,8,9,10,11],"n":[12,4,3,2,4,3,2,2,2,2,2,2],"v":[0.542,0.5,0.167,0.0,0.375,0.167,0.25,0.5,0.0,0.5,0.25,0.5],"path":[0,4,11],"focus":11,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":1.0},{"m":"X","r":0.0},{"m":"O","r":1.0}]},{"p":12,"ph":1,"live":[0,1,2,3,4,5,6,7,8,9,10,11],"n":[12,4,3,2,4,3,2,2,2,2,2,2],"v":[0.542,0.5,0.167,0.0,0.375,0.167,0.25,0.5,0.0,0.5,0.25,0.5],"path":[0,1],"focus":1,"note":"Walked down 1 step from the root, each time into the child with the best UCT score.","roll":null,"flip":null},{"p":12,"ph":2,"live":[0,1,2,3,4,5,6,7,8,9,10,11,12],"n":[12,4,3,2,4,3,2,2,2,2,2,2,1],"v":[0.542,0.5,0.167,0.0,0.375,0.167,0.25,0.5,0.0,0.5,0.25,0.5,0.0],"path":[0,1,12],"focus":12,"note":"Took one untried move, cell 7, and hung a new node on the tree. The tree grows here and nowhere else.","roll":null,"flip":null},{"p":12,"ph":3,"live":[0,1,2,3,4,5,6,7,8,9,10,11,12],"n":[12,4,3,2,4,3,2,2,2,2,2,2,1],"v":[0.542,0.5,0.167,0.0,0.375,0.167,0.25,0.5,0.0,0.5,0.25,0.5,0.0],"path":[0,1,12],"focus":12,"note":"Played the rest out at random: O wins, which is worth 1.0 to O, who moved into this node. Not one of those moves was remembered.","roll":{"state":"XOXXO..O.","r":1.0,"mover":"O"},"flip":null},{"p":12,"ph":4,"live":[0,1,2,3,4,5,6,7,8,9,10,11,12],"n":[13,5,3,2,4,3,2,2,2,2,2,2,2],"v":[0.577,0.4,0.167,0.0,0.375,0.167,0.25,0.5,0.0,0.5,0.25,0.5,0.5],"path":[0,1,12],"focus":12,"note":"Carried it back up through 3 nodes, adding one to N and folding it into V at each one.","roll":null,"flip":[{"m":"O","r":1.0},{"m":"X","r":0.0},{"m":"O","r":1.0}]}]};
 var IMG = {X: "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAIgAAACWCAMAAAAyuDCLAAAA/1BMVEUhKUzO2uUxhbYZGDfxdjMvb5mM4OxSsdJYUmArSm36jzF10+T19vRqamuYnKpPLjRWkaz7yzpPUmafWjf60EVfc4qkpaVfYW8vMT04k8JpSzrfpUmYZUcTE2auuMVMNkPSc0OukUjL1uHO2eI1OFJ1d4Zqa5WGh5EAAP/X1beqijwSYmI9QVm/v//DubaJfYcA//9vkZj//wCGf4ZucoOBOjG3xbfCtaeBgX28x8f/AP9/hX8A/wAAAABJqs39/f0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACQfr59AAAAQHRSTlP7/P7+/v7+/vj7/v4KBv7+/v+c/f7+HWQE/v7+/gP+/f7+m1eoWCBNAQ/9A68E//8BIQEkk/8WEzUXASgBAP7+3dBW8gAAFPNJREFUeNqtXAl7o8jRLmgDDWoOgwDrlj2y58zut8km+S7J//9fpaq6uaQG5J3p58lkzdG81H0hON+xdvi/f73mp9Mpryv6++ktPn9g4cXqNQdaYR6r8/np9na4c5sf0Kz8TR9+UvcjUTX0Vk4vE6sPAiEYdYivUmZBEGQlbZTXTJj4f+8CcT6/0v3Zwse10DvcQoF5HDUIKAO/WVnIr3Wqv/LZO3AgU8pFez9iwQ3zWg1vnwHydK7MNotFu1cGIKXA91IsPZPrv84KNwj4fr0D/otQkERDKDC9C5IDwgYG/oPcCfBfEM+bLW5Wn3fTkqKeEEfYo4ZefkBklfW5exOYZgu+TdbsEVyaFVxAbJYbiUQ5z1G0w0GvoW8P8K3KVmxngcTnrx2OHgzaKiMkyw3yWu3iqS1yES4MjP4GlwUzmGhq2AMTm8QoVIHP+wxgaCRy+bjcpJM0ic+vAAvm6+JyvRalkFt6E40ExokaA0mZfZdL4AlE8riUiORpFEdFFGUkF8sWodhuDiA1EpigB++ysO9yuYTiGWmCSOoxLSYB0ZwNrDuQpKGgya+0AUy8jChvpaNbWkyWG9YdO47aCOrIDhcX0ihaE02+jAHZKRLqyV2CDJmD6xmgstEkPr+RgNCrjOxwWXmiICQ5Xgxj0o4EySZ3ITHZEpIthOqsbm3QV9rBLmENECZJJJCmT2DH8RnSlF9nfBcjJign61uBjZ92+C7lNI6L64YCgUT0JmB3+yFsRUjbBBM42Jo8ourwK11vwgIyiQOBMG8iEniwEqSGdCO8mW2YOeulFtj/G4pJVdfsYfzJN0EgsGaSgLLLiITjEXVmBkhAmoMr3ZKYdPFJrHQQhKHDYnID13U1kDXUYFfdw0YDmXqfi6cDgpCccU9MSG3BwxUizDAL7gBSQA52/S+ihxkgJCAQ4uNcIvCzhNeOOZJxMBaEcgeQSEiwG8Ti4YGEdQKIp2HoBVv0f1Vr16AFQlDgEoxpbwsECWvTmZN4eHhIYUJ70U90MBDIGq1J3gD5O4DbPzmGhGlZjFPkfA4RSLTluGpUXVocrkeyT9ak1WHZowidDUc5EwoNBAXNBgQAgbC02oEErgj1O4chS6v49vi4MaZeoawLXnSKkQgvGOMMvXIUbaysoTdC1kQHoNjQ7jYBqYo0108DHRE8o/fafSERE2WGS6MEZk42xpn0gdYRpFVGfsCReMOuws4Yj1XX8zLthylIMrHJrvG5FN9eMg91y7OShAiChpVwoPrWI1qzjfC8gJFYJARSy+wS6CBUB0loYcnUU/ywaE0hXkBQIAxsBHENQVAxYitFalQqPF2Q/11YnR3D6OsQer/l47MApUIiZM+0M5RbiiCMladfmF7ZZuK/kwM355HEtpgovOJ5QN4PiYI6nIsbn4scsoQiJCEHTRA0rFant9MWjaSktJPEsKTJDjAVRUvyyCEsmOiBTgZBC8XKGC0hD0cyhqPel67AgDIbCxXx4Y2v8TArxkCYNGfD8TaKqF7emKdx2YZoCYnIFj6NxCPA3IuOLHh2V4OCIssEl0RpdDnlQoqggJSUKyfJHs+MOT3WGMN/1F0yQDAawh9Zrzg8sruaxHk3C58YciyNvERxKdsTDp7xZhiDKlNPxqxyw1eR5tykV/j+yXt/7dm+IgEwa3IGZxKL03NdHRJpSRVSjaYT8U5JKFrNWdxkEjB8Gi4pyjLMyiuARBUUmyESt68xD0J7hgGQL3E/F9AstGmOBQchCfzsFgcjufJU2slELWOeLJnermdeWagj0pwBc4LQhgORgBUHIhEDw6o1RtM7akO7HpAvZ6V6SDDFKiIt1AOSoEG3Pg7fXEjrCZQTNxgKSKe5YHIiDUQpbT3g1CZtnHQeIzZrQ82Bkcfh8xz7CUfClaAazX2ANsKEjicVar/s0kcKXQ+t5rRmDQmyHzwELYl5focQD26THqq9aMwJCwhmTY3mtjF3R5Hvr8R6CRRTDL0wueGWOUE40E9HUj7Bh5yOIM6hPWq41kiJJohsNRcDyw4IpSHIEwbidAEfM6fQN4SdmEA2ZMAhTVOR4rESevjS5qj+s4QBY44GRy99B2YB2em6AqJ52UiPYqnRzFy3Zg2jxP0QyO/47imJqZQDZTZHDas0bwYCchT90gpRhHmbwJ9QvmuSxLqwtYsbHe6YgyLid6bd4UcCPTN559vpKMuLPvqbAeIzEPL9reZuDoOMuQWCSLR9kCcU3O9abtDA8m0YSRvmBB44LZCDdPBdKWillwej1enWcaQ5ujVAHKDQaOBzr4pemjU+XV0KNhB7zpSo+J5XWoeb0IQ0h6xZy3mUAuPve0CujvaADBhTAPx7WAKPYwVavqW+T+ZUX5UJehTFOsyidSAdHlKEJJKjeBDr9NABSQ0QPNoA8RGIa0x742L+e1gCj7mUwZqAToH4n4CKedOEhTo3yqbFBGWkx5oGiNziI2V5BY+PtjIyMO0P4roGiDJSvalcW20HIxp8CqbmRmrU+a31wxyaYPiJWtMg2Qp+JhLyN9xAQqMwzdHEOAOSpExHy+vo2pI1QKoT2p5ckuo6xlYlgk2TDycKkShxMzrMYoLEankj0vUaCSB/k3hHY+F/a44m8tCIiAS3l09hHiOvi27wAzCoo2jPd1pnliSaUWxR2ggWXwPjUbKsnf6i+Vyv5Zq09729//f2qGNwoGXtCSq5un9fFyJBa/+e9M3sk+yNMTRmj3LI1Fgg1GG3x5v3JOWlITS8MUfFwXlvOeP1o3aAzzc1NzgZj+A0LivR0YbfmhtmTtH64QB1rEPi/C6lNP4tE1kXq+LR5g8kCLCgrhuXa6n9wZvR3ffWh5pdoS2edrE0BekLDH96SAZxqzZIVzGAgwi9LgaxCKrWGoxCBh7b0LeEfu7XRAQC48FFiFb+Fgm+dgbh7WFmzGrVWrI1Bsu39WGyI1UOYdY6kEQ2gV938d+/q741CcjeXCEhxcioeTI8jpc5exH2LNnA5Q4ogrFhdQJokhFHmvfr0y8+fw21NWExweh5z8/onucjPQKqz/cx0iXkjFhA2nw7tnYzQDtaVbe6ayRG5Go3rPDrV+Jw7RJiFtVf+DT281RrTPonfMn0QEGNWotq7e+Aad6dT31BcxIohza4sybU1Qoo0yv95mEJiNBkUQFixLC/wYdWwetbssNonwmaIn7VaA8TNNu/h/lN+WatrQkxJ+D2ryzLUl6VdTkrpjOSzKR34WKfsWTcPvgyBYTbb1pM9LtcCQkFbEobJC5p6VpQrxowLBToyhaeoP/uLBnGqF/H2l3QrxPJxHf8vbYSZFj/6F/51uQXVAoOmsKI/ue2ZqFPBJz4NxZkTGGuupy78z9O/IZsVUpxw81dIyYpzHQtLllX2CKfawhCFjWOn/SKr6YUYJBtVjVpAycJZUjyra7EhLbEIG+ma0F8aXI711uFoi1QXZUN35S1AU1UyyGjkJ6UMey6ww2Qk/gWRUuYo0hGvf+sBeKJZ25BIGc+v9Y5r/pzpdqH3vZ9q6eK2YMmmZP+XJ2f4i89DaZK2TPFJf1iiY0kGXhBrzvUFtj6S+af1Xm8E66qzxw98H6SmvfEUFyU5WweHzf9vA+vCa+hBAGVM7Ms6FXcv1FlWkDIxVkedsh0CY6mA8Y74TUi8VwqUNGVDR8rLrlzmaxXRc2IjUG/3qjLqt0xjog21PlzV65WpsBMTZAxynV75RYIyjWNODASNlsyr1/jP+IYvQ0SeH1VuGFTygYjaGvewwLeiqRkvYzQByN1+qCJNCEPKox2wn8wkoCLmC1PidPfKNPyb0YndB+Cm2dwU71bMXOW7Pzwv1d66YtoGAX+HCvm/T+VfMHTRMHEyDTFlkZArnWG6QC6MYK3BNZuptg8D3pN5IJYxRc+qae9vHlWO24Qur3VCKq1m0T0ps5Ir9h821ddFk0Xh7MvIqJLsAlJNTU/ErZQPKItzUZwVcAfK0WTjLDeWxorFAgso+dnpIvUjrmkcrC3IiTBSAPaWA1VS2j7mNwIkRO94JWh3GgfEfSYBxWFu2ADLRYK4iIca0C3A2imkRwaSxZOzj6gHE42vUnIWO+SXnHN5eEa+J+JGaMvpN7Va56fDp0lm3N3k3MAbGDJoPQyNE9P+VQwPbyl/08SjqUeOAr+IpDAGNhHoqy8AuJNsaY1bwpgTlDvI4nWPFxbk/Nz5kVAwnkgOkSk5tT80MI8c8AwR1ImH5LtCSmEykY6WNeDAumSPFbo+z+FQ8+tFEwSFLc9pUAUXFPXadKOtLM1IvpZQe3EpGEOemJo6rTZBeR5BzM4vqL6k6D+pIB0zEkNc0TTX6JqYIVp08xobw4FC2r5Ewpzq8OIQyZNvbYU41NX/Q6wsWSLn2WMQUIUZmFtSgUotSc1N71Zw6jL/evMEYijy5r3mM1hvh9PTm9WpnVJraPFDGO0rwnu0RzZ72/IUlcvYdqCFKaFOiuobbhwD5KyV7jwRcIVEJi0IHe4uq5faCKGyxzlvEFLkKrl1ZSwVh+xIG4XQ80iocyv18dwMpFPAYkp5/7GTjubxaHlowmkgjuYs+9XQGBKRoyAPG7vsCC6yh5SwZkD9Q8xxykldWbGLCuV76TW3HkLYuixpvTcu0NgyayVXe0tLOF1DEiruWJi9Ko/K9SUHjhlmAjUzGABmjWn7W2BhB/quxWI+q4kxyD3aK7bb8MUXDG7gySmvOzsyeUkYmSM1KTbd/n+lTscfbhLTIg5iQlaqVA2AoQZozV3FkcjqISjWDNz7hATCurBpxDN9x1nPzbIgj7XaG5gnzG6ZcyaS8ooJ1Svv0OHmTlN6ZHbICNjpMLUQe4UkFRX6hIhab78DuZwOgzZ3vcpSHu1GzSy7Rwtw2JOc/uCSkmCFNvoeBdz6DbK2IVYb8enwAH4w4jRWcWhZdc9FFZIBxBJoXXYvSOmpxx0uQS70yPOSA7bgxnGcD+ZBzCpgZKYFgUJy7yYsOnRkfSSJiMsCRaFZRwewozGaIVhSxa1CW1Cf95hYHUdWOeghd3Es5eJIpoC9xd3CkjR+dPfqU81LyYBVxg2j1EUPX6jmVwbkBNwyT2bmyXvGsq9bBYFtqCpsdW0qQ+4+cojOzR6bbMjBkgB2cxwvXZ1BZvTZDDhU8zrcGDmjJtK8CiQaAZIiwMFdT0cOkKBPW5ghjmr3ihaMcaaHMwM6WJGcdGSRSSoV0NHCRw2W9bh8XCtM8g8VmsHwmPxk0B4pE1b1OgINzNOgEjWYjJc0wq3Nl2cz7YIjZqr7L3KUSArHYKwoB4B4HYijQflJ5gTuL1B43Qsr1H8BG5UTXh+j11/dLBMn1EYup009W5/dIIHBWB0rjaFxWTZTocgawgt44mCBppMkLSaCGJM+4T6jWD/ZimlxmrmT6RSJgQRoScsveiQkKRah1ejJF23U5M7+6gxBorbCInuX1y7XTexMjXH3FsRISEhST4aHV5ZVa7tJ3ExYDRCgyP5j4U33CVo0yg9BUpiYBtQpOkIvGRr/LBrIynrfkuQsZg1h0MUpSIjB7u6rul6TR9X0PNsI5SJnkqQuvs8gNJuYiTkYPrLI3kNMgetnhiW47u0Um/CTtazTUo6pv0tDyM7tCpDoxPfx9skNEFJlBNa3G72EHrsgZ9mHW7lgRE8J2EEiRlHS+l7i6mPSlFMJJL/eIuE5i96OFz7UGkzU8QNQitNTT7WfFM6lYTnGPcdDxAOYBBb9LsYQYTEPtpK3R4XRpCEJIKkc3k7SgKTH4OD3ErBfRVGwemz9t2pIIKsViPjvnuaXTG8gWuq4rusN9H2wE00NfvhMV5B34PLLQXbuvdEnTIaiY2Qt4BA+GtMxz527K3IYID+uYDQXfVhIKWLw+D75+li3hMShVqiB8mNJ5qVbT6HgTwHj3QxBPt0MUorIUGd4C30m7j80Y846E/bq0F7G2Z/M4Bn/Y7b7bYgH0emMCWanoBt5NgAtASORShTILpC20Izi354YPfl/s/1Yz3/i0by+BDpcAphSLK9NDzcK3RcLRq04hZ2FWsW9xvg9T/6zfj7fsCAWjbmdyHE4aB/uuAVj/+hB1THRtNJbVY861mb9joNBfxZ169visXz5tcl5n9bgrtHqs5P+jcq8G1UXX/OeRzC9UYn0DWQDPLX+vP1sNdTfP4rP3LRfMmg1Ffe0dCZhqxG7KqOBFzdsKeFWfbuCV/oaUczBjODLNNkic1Qy+7pFeTzZrvGXJ6+YB4b1ifbugowDSyeN88HdLBfZp5wJ5BBjL/cYH72kIbBHJALGnL6HHBD0fGvBhLnQDjQQMOErNKso7fKgL9LjJ7hNX77tUBiSjb6QPbv7+Nq0wDZojWZ+92UDwEhj12j0ysYSDhJkZKMPANB50ZV7tEfXPg4EDJNJ0GfFVBJRHj8UeL7mPqiVb+EKcX55T7RUOJfAgTJUfGUukxQOYsH8j7pdkx9E7HlC6JUvjtSJpjq1GqKKPABctTk/0rHKWH/Duj6yAOOCUmJwRNeEBXUF0rek/KAuH4FReLzVxoXL5Dh5fseJOslAUnGgfDMl5SABEzpRmHs/U8AUVTk4nHx9GGzlv6h4KdMAaEwjq9Jy70oNgWUyb6c+I2QWaenqr/R3JWCZia7iIpUfz/N2dFgerVbpR6w5B8FQOqlusmbUAGCNlQfAqLU38wNlcobYXDM5+QaBk022oGA+aiNV1q2HDNRGW6t7gOi9PyZUp8+vdBYQ2vL92kHY78fpQg4GaSGKkXY+cLSeXn5pEGoSs0DoUvUp5eXZogXwnYAWFKcVqQiJDUGOw4noVGiDDi4jNb7TqtNBZ7QnM9ndQdFPr309t1DCdRE55dN/5nS3Pi7L4X0nTEk9HGpg/Ip0n8KfWOSgQxldwlhUdNAkCHDbX00HYnUYXAoy8ThD1fHBKSRV1EiGf2sNDfKcv8u5eCaT2oynUCg17vqz00cM4PuJPimchKGoQrad6e70cmuZeprPJlgqerTDRJBvPF9P0nKUD/gjsXfqSAF+cYM/8icj1BEy8jtnmDyApndhcJpJt+aZCIsByL18ulfczLC4qyG8soymyT4cs6Hl7+/uU9rzR2/dRVXjdi+vDi/dH3MjtAPAHUXkllDQD8FiW7/1MqEqir1IV8T4w1V3FdswsSgXu559AtdjTep3o43IGj9B3ST6KhuhGa8AAAAAElFTkSuQmCC", O: "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAJ0AAACWCAMAAADzP3onAAAA/1BMVEXc2+FrFROdIh7TMTDpVVLeZi309faSXV3xoUuymJuoUCRyHyDsjTbLq62TY2F1c3OjXV0hDQWxoJ2LUVB0NDPk4ubj4+Z3T03Gu8P/AADItrZzQy+wkY+7usR8TEx7Q0M/DgfcPkH/f3+BOThsNjabfICEQD6r5uarj4+8wMlwPECqAAAA//+pg36jgX94QT6qqv9/fz9/f/+EP0B//38A/wC6xLp8PkDQu7ieg3vMv8wAAADWKir3+PvjMzLhLi0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABbHldKAAAAQHRSTlP9/P7+/v4M+v7+/QP+/l8DEv8hmKCeXPj+ASL/Sv5njwT+ArNo+7AJk/7+AwFN/2kDBAL/AgEapng6FAD+/v7/aEo+BgAAEfpJREFUeNrFnAlj2jgWgCUZW7HBxgUakjTpOe3c1967dsz//1f7DkmWbBkMtBnNlKY4wMe79SRZNFeN9ZL+yupqu9NSKvgj9W5bPWbmF5bLj1e8vbiGrcaHh2r7SdJQPPgfcrutqgeCXK7/AjoU26bSjJUs/MGQ+LitHkjGL0yHn1d9IrRFfJAUkbDaWDm/EF0Ggqvww5PFsWHVvK0v06+4VKkVqm5xchgBSv0Ir8pegg4El22HbJ0/0hQfP9CFVElVFEp+2ZyvXnGRxVWBTrvp8cHIr7gpQL9Zs7z/xnR1KLgoVOL9THxS3dzAi6qmuf+mdMtm88mxpRMiS5LEIyQ8mRrx1d+Qrm5qFlx5TKWJJ7+kRbxEgnZBfNvvz4l94lybY7iyPGpuPeYB5Ni2idVuIdWmWX4jurpZM9wptsTTMdAZvILw6vl44jzJAVw7Aw4khho9MB3iIZ/Bk/PxzqH72DzMkxzgHZLDoYX/WwPX46VSrufa3hl098tsF4FLeAxkRzz0YOAQL7F4m5l44pxYsh3C9VQBXnvAiJIYrXp4rZQrsr1sXlgWZ3hENYBDIvhoLEPazhcfEpliT/VwKEeIezeIt51nemK+R2wgonpwSNO6WlP5wgOJSTdUG+KBbm+UrGbhifl6/YKiu/HgWuUKYnBlkx04ECtbKFMR6tEBXkp49RzTE7NFB3rtejhAUK5WJwZfQh5deA2lCkEZPSObUVCJ2f4qZeJn1sQHsBKyntANLvZ8VrdoeuuvRbdml/A8Ivh8Q5e4EV5Fxfd45Leg2xlhRcyEy1RgdL5w/AkZpy3yzXCeNhJeOkd4YmY02fqis3DSG72gKMap8FKgWxZeAX5bfw269T2ksN4lEvwANaLyaZDPezoQHic0LKey5degoyyROpcg0cXQYP6AMwhprZBoWHKe8DChzRTeHLqsyVB0C9/qAjTyykKDYd4sCsvt5diB284Xnpifw7w6XRn1tQ6ugOs3aYFgGgf+QtKXAL5q24Vz2/qE8MQsxRaQw/zKBNOoVRvJJV2kNG+QuXimITThRUdqhLc6mW7FvAyrhtMvLtuM5FBwKQouZzIi1Ea845FwFT8j5ok5oqsoTQxrur4EAKPD0lIbuZHkYkWAU23KfpGeEt4srwh8wtZ1WAPovBTiLqc+lNQ+m4YLQpR4SUWFZ/3iSs1+RMV24dw1oRqg7GVVAoX9Vw5Xnt0VkUf4wC9ItcUJvxCzFIvBbgiXu8+nUVom7a7wJXKQId3CqnZ3VLVz6HYDswO4XlKWTgjG0r3g3JV8gJekM1Ub0GV1HQ3FXp6w8W4Mh3jiWUevDPAS57WF/O8x1YpBOzMSin8dKhbgygiCEBBS3JXgwki5NiCf8FoRSKmq8a+T8WRgWZ6InOOKwfAqPDu3TUm1+rRmP9YZYNQQGHbfD3pYy+aTGsQTJT2XFD6L06sYMuaD2Y81PAWGtz4tu/dQm2t4/91wqjkyu6Hocr3XuUHQ7nl4dg9pzdNtSJcwXSEfj6iW6TZVhhj4Zupd8NtrE+2iLkEBI6eUsGc6YVOZ2L+SlHajwks8w6uO02EDQsoq2xBdDrnvflCfDOi0r1fxXCLGK0nysXqFC2/oaYdXhn5hI97qaMQTaPaqFDkYAPgb2u+ueU/TsCwzhWcb0CV+HKYHiWJ6BdKTPvf+VSi9wC96t1DHIh5qdqc5OWr9TALIaIrYUAsfnEImAV1rgwYKLkc5vXpFILmw3GRvkp/dTxmec4sjdQrKbkdUAjInG0hFKx8ZL3I1oNjUd4rWGRfaHIjmjaHTd7aCQqT9K/N0NOT1dEdTLWuWXn5n1KAl9tXRFrcZu2zqm51P9xuq9JXh2GMoRkMU/rNOtXrkFsrOLpbTdNkaQsmzCab81yMylWWuAG8zQUf+qnuMQu5RduQoPd2bN3G6mdlC3NOKV2neoDSWh/aNnFnzSHVxkClKV37kjmNfQLTkb+nRSf1G3k3QpdI47b+Pya55eMy2Fo/ePpdbCi536CDj8kmSK9iyiUE0Fk7ss4C3N8+CKnqvkFMh5UguE9kXrrD5LVgomlR0V0JwwSzbDWoA7egEcFBc0/kbyYnMmAg9XeS5U+wg3iWW7mgRJXZFSZWt7OHg7a3oMNyN6KSXoACBBll/zpaBmPZpPVVEpX7Amwopgt7wmTzCZW3SD8BhJBoGY6NaO+5QcmD7AHf3lBv3Ijx6VosJxVJIMXQPR+jsG+b2M20ax0YCFsZAFy6UKOnXRp/fkGYB9OlJG/sVrI59kGbHdC7gTdJVVhKu4OEfQHTNe7AIjZoNF5da942C8XSXe8F3VN8NinePbjMZjgWYvRG/0ytprNS7XzDfjug6mGQ7EQUDtFx6uhzUxtN0j9Oyo6JT+5/GblfK7ygOadWN6Cbw0GXzKN4Ibi4drfHvQj6cQefyLQl8THeg5mEZUV5Js1kdhRvStT3ddCoTPNmpd36YQNFpeNGaNBulU0OI3E4icbIdXCllpKPiy+4oHe972QWWnuc/GU9HunRIx52n3Pt9DUnB1naBpZSad9NcTocl8IMfxZ7yvCz5VegVcTrqo9DA6OHmGiw+vqRpX5QM16OCXspqDh1V6A7vydZ5fzJdF6VL08I2oXT5HI5S91dKXKaakN0K6X494hVuevM2MCXMY2vIZO8gGsfoINKn6Sol93iODRfZ80m6lOkeT9PRtJr0YeHkullyrugiXkF06UrK51MDS6cxnUK6FdEdi8YeXvbW2jrAgday+5ry7GDZ2tAViOe1e/xxp7XrNUbplEoN3bEFvWEfZUva/cxLILtfmnU1oDt0Pt1KxenIUYSlaw+LMZ2i6nN1vAoYbHCCQp0b5mRauw1NZ0PRzaGDsKLt/FFiHRHS2dWrIqUKql7P699lTbbdwYRnkZiGdf0AOgjpOkOnkK6QoozCSZCf+RGzjU93UDJYBprePTjsLlIb4C16GQgfUwKu8oSKHdCVEdPL/e7jkA7h7C6qFBF3VTYR8cR4+9oSwwivxSXIN9QsGF5iAh6EFEiu2uu2Iw42yvonSup0hKLDCINhNOUtXHI/scFHTDVj1SKxa5ndWLOJoQOn1SQq6kJhkCuHiYNKsHREZ9aZD7iJC96riipXxFeyGS/h5dTEKpV8InTalG0/9xc8c/E8pOtCn1A9HowO1y+iK3oTXe2fcVXRrrQmgcey8FpjeKkNxyA0HHkphv5BJVgbCq9fbU7sFGhzv57dc1/y/kR0DpCisTcjO59uMqSEZneIRDuzFEQSXCyinTJxZBfgjhYKQYlMZ+B62UnjtCfyWDKma43PYxmhcEMXZt2syeavV9RUM5MGFlanhu7Q8iqZZ3iTg1pY3Vh28GrMY2AZCle2FirmGEdWU+65ZoaxYJ9wdFHDO6bYLpbHkA5ED4kOhafNPGb+Wg96UbbZyg/W8A6ealtTphw3PKfYw4gOawAwW+BPUbXy9zPp8AwApV4jvF7BRngnVStYdIeIVyiEW0GglO2CQkN1Nh2uTmVM54cU+LSEcvkprwXRpb1iDz4dKPaGFKsWTLc5y+68DRW2M9t7hl3sp2SWHxddehjToWLRJbQ0dFJGtiLPXAFdBHSd8cGTwtO8iNXGnIJzNC3tqgnFzqCjRcaFhetcUDE5iYQ3YXk5iW4U7Ey0uymUyTIQ8zDc/XLZvgBQbR/yDKNNmcUKhVdOR5NRsGPFFmBzqm8jxKeNM+hqq9o+pnQHD29qdiEoEEfg2CRod0iPF8kUs+hYtSkHZJtx2970CtTtGK/kDXuLQ5ROyZ/81Uns89cX0VEDFFXLUmM6a3k9XjmyOYKbEB3Wp8JfO/39Qs3Cl7IBuVds10dYsvBUBQUytp0S2nUWiK7/RtKHewK67y6la5bZJ+MXQTVgPow2gNJWHs3FHXd2wOZunOQOw+qp9CVX5nl8tWze/rumdn4R8B14DyrtBaSmiuIdWiS4XnKHMV249n1W5R6daCwWw3zR43VJsDkwSVNkM3BGykbWprkWGJ3Yy22W3V++s7Ie5ouez5xXSHh/NrCmqTlU43IeMx5sZafsuoaj218c72xE1mPL47TWhf1Hy9YdrNwOve1xZecWz6xmf9TRFZ/ZO3pplxtPt2N4w3NknfMH5nMOjltm00HvNpf7+GrZ3L3aXsLw8sUEX+rYWlN3+nBqdaP0QLGfddRp5+9zz5TNtiPpWUC/CAzp/EMCKS7K5p+91aKfIODFCvdzzgjUSvXFQIzQhUGPjnzVAdJW6BU1i5XfQv9R3Mm319D1uu2OkAWhZhB3qG1Ey+1Ih13MJ4P3I++VyK4/m5K4fsW03CI/W6OTK+7G7ivAy3W/3nJNROHuVCYNXndcq+OnEwuXmknshnoN3BzCptU1ucKa3r8UNX9mKneQji0cFjR4aGvz9i0ewdV76pC9v/o8GRQrl+CZgwM80yE4bDiRIrPHarfDs8kzu4tz8D6cwWcOlAVwHDzW2Mc8tjHxgnOMhLegE1Bz7M6DUynvPwkiWw2Iy/XXOIvH0vt+a/DGgO3wB9NBJMHB3J/h3n6jc4yMx4d7P9gDeHEJtg7NaLVY8b6imWe1LqTDGTuddl/0RwSjnuCxGTjs6ZxxxLK57FT5e3s2OgkHVUyd/wynCCzsTSTZnXN69sIT+TC3q+mIxZAvHHZijWw3OC+S2zPvunDZ/QLwTH7VH7OLg9lFZmKDmT8KrjnvlgGX3gmCbgTxxZ0x6k8CWjx3LoTkVlAv4iytXkPX3NdLPNrgnfBxbP0aWIH2xkvguCv37HtBXHGPj01dY/MoPFbWz8wKDw1Xws4WXHPlHUig4AMVglPq4GgbcBEYkBE53eNjeclHXEX3TvLKAG50zHV4Ks/+i9CaOmtelu4eW1NMZw8blTl3AhwkNQ3/knvLUPuCTkbHeoulpas+Li9Xj7jK7BKmi7cWmW7X/CV0Pzc7lXSUDaIrUBQI4630l/AK3CdNm/H0xC4UZ3gvT0e9i5QUm0/025muvuyGQdfRkdmlU2bnu8UVhicuN7svoNhpOnALnF6nV7mFuM7skimzw4VZPCiaymsM71I6Oj94lA7PU5BqN82LR2Mv2uXTW8euNbzLamOY5dU7olOTS3i0sR3p3r2HX79/Ybsz0W6SThCdAsMzwn4pus0WBtAdEO7N81G6hZLb3bZ6qRoFt2By57zjnb2Tuylk0u9fBd/4+AJ09hQ3th6OOQUtHdMyFd3awJ4D+8Z0f5Kz4pY1OmSvpjfL5Cy7D2YLanZBOXC+7Gqgo45cgmI5smWhRDrJdxVAuvW3p1svmwewuAOvLaqJ8ql3C3NwYaF2DR2I+JZ0dMtAZSbZim/eMSm732hOpOyse/sA3+0UX5Zl+CfLLqBb8tajXORKOTo9KTrJqRZcQpelOTzHHU/sryPDOvsbPG7oJ+JqbLcg+z47s6vdNA9bYMtxK7FWrd1UGmejEorWQhWdltI4aYOwV1PnP5toWWT/yDzRnUFnbhmodE6neBGPCzgR3USOLov7pbpE4eFQ2vwNgqZWipNddntLD7e3r2EIfsBXww+38DtnrPVkn6itQ7KQf6cw1o7o+hMuSIcLVK3C42ma2gbUsfhfRjSMEZ6ICf4N3HPp7vFmVQVO7/FQSamNY4R0wbsTHeElqtwrXGAucMdZ/lrMHK9n0+FKCrcIleq8m52ofvvd4LvjRh6zvNdi1qPNeoh3N5dOZLNXjyvcuMTvr8K9G7ED+Eyn3BZ5SHxpavHmocGbzqSjrsQqXdzw+/NOUt5DBj4bf/dc0hc58NqtKm4sXvy419AAX4PDzN4XsJHuy4PtfUhMyzUtJg6Q4XK6IjsASXfpQq3cq6dfYZz1ljwZ48tcugpDr/v2VECRj0Cg5bM95d3dwKBK3uVTcE/K6nV8mooCyWuH5KeN+XTpquhv61WwFPCoIo27mKnTppnUNPLswJ2YPReqz0dac+awAVnMTBOZ+RxPvciGcEfMiG4twz3Q/oXupNdtZqW0of/g/8uqgE3zqIzEUtfT1A7MxPnXE4D+K92p1deY0LI/sia7ukaBLP1a+23h/efcGYzVBGcnzk0+aa79l+ZPqFH8/T+yr1hBZbfV9t1ut3uHN/X2c7XVT4NN2H825nmTRzmJ/ueHH97ttvDSHx5vzfXN5vRH/h+BeMSwZBVc8AAAAABJRU5ErkJggg=="};
 var COL = {X: "#3b82f6", O: "#f97316"};
 var PH = [null, {n:"1. Selection", c:"#3b82f6"}, {n:"2. Expansion", c:"#14b8a6"},
                 {n:"3. Simulation", c:"#f59e0b"}, {n:"4. Backpropagation", c:"#8b5cf6"}];
 var LAST = T.steps[T.steps.length-1].p;
 var k = 0, timer = null, pinned = null;
 var svg = document.getElementById("ttpSvg"), phBar = document.getElementById("ttpPh");
 var board = document.getElementById("ttpBoard"), cap = document.getElementById("ttpCap");
 var note = document.getElementById("ttpNote"), cnt = document.getElementById("ttpCount");
 var playBtn = document.getElementById("ttpPlay");

 phBar.innerHTML = [1,2,3,4].map(function(i){
   return '<div class="ttp-c" id="ttpC'+i+'">'+PH[i].n+'</div>'; }).join("");

 function boardHTML(s, size){
   var out = '<div style="display:grid;grid-template-columns:repeat(3,'+size+'px);gap:4px">';
   for (var i=0;i<9;i++){
     var m = s.charAt(i);
     var col = m==="X" ? "#3b82f6" : (m==="O" ? "#f97316" : "#9ca3af");
     out += '<div style="width:'+size+'px;height:'+size+'px;border-radius:6px;'
          + 'background:rgba(128,128,128,0.15);position:relative;display:flex;'
          + 'align-items:center;justify-content:center;font-family:monospace;'
          + 'font-size:'+Math.round(size/2)+'px;font-weight:700;color:'+col+'">'
          + '<span style="position:absolute;top:3px;left:6px;font-size:10px;'
          + 'font-weight:400;color:#9ca3af">'+i+'</span>'+(m==="."?"":m)+'</div>';
   }
   return out + '</div>';
 }

 function render(){
   var st = T.steps[k], ph = PH[st.ph];
   for (var i=1;i<=4;i++){
     var c = document.getElementById("ttpC"+i);
     var on = (i === st.ph);
     c.style.background = on ? PH[i].c : "rgba(148,163,184,0.12)";
     c.style.color = on ? "#ffffff" : "#94a3b8";
   }
   cnt.textContent = "pass " + st.p + " of " + LAST;

   var live = {}, j;
   for (j=0;j<st.live.length;j++) live[st.live[j]] = {n: st.n[j], v: st.v[j]};
   var hot = {};
   for (j=0;j<st.path.length;j++) hot[st.path[j]] = 1;

   var out = "";
   T.nodes.forEach(function(nd){
     if (!(nd.i in live) || nd.up < 0 || !(nd.up in live)) return;
     var up = T.nodes[nd.up], lit = hot[nd.i] && hot[nd.up];
     out += '<line x1="'+up.x+'" y1="'+up.y+'" x2="'+nd.x+'" y2="'+nd.y+'" stroke="'
          + (lit ? ph.c : "#d8dee7") + '" stroke-width="' + (lit ? 3 : 1.6) + '"/>';
   });
   T.nodes.forEach(function(nd){
     if (!(nd.i in live)) return;
     var d = live[nd.i], lit = hot[nd.i], foc = (nd.i === st.focus);
     if (foc) out += '<circle cx="'+nd.x+'" cy="'+nd.y+'" r="24" fill="none" stroke="'
                   + ph.c + '" stroke-width="1.8" stroke-dasharray="4 3"/>';
     out += '<circle class="ttpN" data-i="'+nd.i+'" cx="'+nd.x+'" cy="'+nd.y+'" r="17" '
          + 'style="cursor:pointer" fill="'+(lit ? ph.c : "#ffffff")+'" stroke="'
          + (lit ? ph.c : "#cbd5e1") + '" stroke-width="2"/>';
     out += '<text x="'+nd.x+'" y="'+(nd.y+4)+'" text-anchor="middle" font-size="12" '
          + 'font-weight="700" pointer-events="none" font-family="ui-monospace,Menlo,'
          + 'Consolas,monospace" fill="'+(lit ? "#ffffff" : "#334155")+'">'
          + (nd.up < 0 ? "R" : nd.a) + '</text>';
     out += '<text x="'+nd.x+'" y="'+(nd.y+32)+'" text-anchor="middle" font-size="9.5" '
          + 'pointer-events="none" font-family="ui-monospace,Menlo,Consolas,monospace" '
          + 'fill="#64748b">N = '+d.n+'</text>';
     out += '<text x="'+nd.x+'" y="'+(nd.y+44)+'" text-anchor="middle" font-size="9.5" '
          + 'pointer-events="none" font-family="ui-monospace,Menlo,Consolas,monospace" '
          + 'fill="#94a3b8">V = '+d.v.toFixed(2)+'</text>';
   });
   svg.innerHTML = out;
   svg.setAttribute("viewBox", "0 0 " + T.w + " " + T.h);

   var showId = (pinned !== null && pinned in live) ? pinned : st.focus;
   if (pinned !== null && pinned in live){
     board.innerHTML = boardHTML(T.nodes[pinned].s, 40);
     cap.innerHTML = "You clicked this node. Press a button to follow the search again.";
   } else if (st.ph === 3 && st.roll){
     board.innerHTML = boardHTML(st.roll.state, 40);
     cap.innerHTML = "Where the random playout ended. <b>No node was created for "
                   + "this board</b>, or for any board on the way to it.";
   } else {
     board.innerHTML = boardHTML(T.nodes[showId].s, 40);
     cap.innerHTML = (showId === 0) ? "The root, the position we are choosing from."
                                    : "The node the search is working on.";
   }
   var html = '<b style="color:'+ph.c+'">'+ph.n+'.</b> '+st.note;
   if (st.ph === 4 && st.flip){
     var strip = '<div class="ttp-flip"><span class="ttp-fl">LEAF</span>';
     st.flip.forEach(function(f, j){
       if (j) strip += '<span class="ttp-fa">&#10142;</span>';
       strip += '<span class="ttp-fc"><img src="'+IMG[f.m]+'" alt=""/>'
              + '<b style="color:'+COL[f.m]+'">'+f.r.toFixed(1)+'</b></span>';
     });
     strip += '<span class="ttp-fa">&#10142;</span><span class="ttp-fl">ROOT</span></div>';
     html += strip + '<div class="ttp-fn">One result, read by each owner in turn. '
           + 'Every node belongs to whoever moved into it, and the two of them want '
           + 'opposite things, so the number flips at every step up.</div>';
   }
   note.innerHTML = html;
 }

 svg.addEventListener("click", function(e){
   var t = e.target;
   if (t && t.getAttribute && t.getAttribute("data-i") !== null){
     pinned = parseInt(t.getAttribute("data-i"), 10);
     render();
   }
 });

 window.ttpGo = function(d){
   pinned = null;
   if (d === -999) { k = 0; stop(); }
   else k = Math.max(0, Math.min(T.steps.length - 1, k + d));
   render();
 };
 function stop(){ if (timer) { clearInterval(timer); timer = null; }
                  playBtn.innerHTML = "&#9654; play"; }
 window.ttpPlay = function(){
   if (timer) { stop(); return; }
   playBtn.innerHTML = "&#9612;&#9612; pause";
   timer = setInterval(function(){
     if (k >= T.steps.length - 1) { stop(); return; }
     pinned = null; k += 1; render();
   }, 950);
 };
 render();
})();
</script>
"""))


Simulation is given, along with two small helpers, because none of the three holds a
decision. Read those first.


In [ ]:
# @title Given: most_visited_child, rollout, rollout_reward
def most_visited_child(s):
    """GIVEN. The child the search spent the most passes on."""
    best_child = s.children[0]
    for c in s.children:
        if c.N > best_child.N:
            best_child = c
    return best_child


def rollout(state, rng):
    """GIVEN. Uniformly random legal moves to the end, creating NO nodes.

    These states are sampled, not remembered. Nothing here is added to the tree.
    """
    if game_over(state):
        return state
    else:
        action = rng.choice(legal_actions(state))
        new_state = env.step(state, action)
        return rollout(new_state, rng)


def rollout_reward(state, rng):
    """GIVEN. Play the game out from `state` and score it.

    Whose reward is it? The player who MOVED INTO `state`, which is the opponent
    of the one to move now. That is exactly the point of view V stores.
    """
    mover = other_player(player_to_move(state))
    final_state = rollout(state, rng)
    return reward_for(final_state, mover)


Two details in there are worth pausing on.

**The rollout creates no nodes.** It walks to the end of the game sampling random moves,
and remembers none of it. The tree grows in `expand`, one node per pass, and nowhere
else. Confusing the two is the most common way a hand written MCTS goes wrong: the tree
balloons and the statistics become meaningless.

**`rollout_reward` scores for the player who moved in**, not for the player to move. That
is the same point of view `V` uses, and it is what makes the flip in `backprop` work out.


In [ ]:
# @title Given: checkers for exercises 2 to 5 { display-mode: "form" }
def check_best_child(fn):
    print("\U0001f50d  checking your best_child_by_uct ...\n")
    passed = True
    parent = Node(EMPTY_BOARD, None)
    parent.N = 100
    kids = []
    for cell, n, v in ((0, 25, 0.90), (1, 25, 0.10), (2, 2, 0.50)):
        k = Node(env.step(EMPTY_BOARD, cell), parent)
        k.N, k.V = n, v
        kids.append(k)
    parent.children = kids

    got = fn(parent, 1.4)
    if got is Ellipsis or got is None:
        bad("it returned %r, so the gap is still empty" % (got,))
        return report("Exercise 2, best_child_by_uct", False)
    if got not in kids:
        bad("it returned something that is not one of s.children")
        hint("return the child itself, not its score")
        return report("Exercise 2, best_child_by_uct", False)

    scores = []
    for k in kids:
        scores.append(k.UCT(1.4))
    best = kids[scores.index(max(scores))]
    if got is not best:
        bad("you picked the child scoring %.3f, but %.3f was on offer"
            % (got.UCT(1.4), max(scores)))
        hint("the three scored %s" % ", ".join("%.3f" % s for s in scores))
        passed = False
    else:
        ok("picks the highest UCT, %.3f" % got.UCT(1.4))

    # The negative-score trap the pseudocode calls out. Two children, BOTH scoring
    # below -1, and the better one second: a seed of -1 leaves best_child stuck on
    # the first child and nobody notices.
    p2 = Node(EMPTY_BOARD, None)
    p2.N = 2
    worse = Node(env.step(EMPTY_BOARD, 0), p2)
    worse.N, worse.V = 40, -5.0
    better = Node(env.step(EMPTY_BOARD, 1), p2)
    better.N, better.V = 40, -2.0
    p2.children = [worse, better]
    if fn(p2, 1.4) is not better:
        bad("two children scoring %.2f and %.2f, and you did not return the second"
            % (worse.UCT(1.4), better.UCT(1.4)))
        hint("start best_uct from the FIRST child's score, never from -1 or 0: a "
             "UCT score can be negative, and then no child ever beats the seed and "
             "you always return children[0]")
        passed = False
    else:
        ok("survives two negative UCT scores")
    return report("Exercise 2, best_child_by_uct", passed)


def check_descend(fn):
    print("\U0001f50d  checking your descend ...\n")
    passed = True

    # A root with untried actions left must be returned as is.
    root = Node(WIN_NOW, None)
    if fn(root, 1.4) is not root:
        bad("a node that still has untried actions should be returned unchanged")
        hint("only go deeper when len(s.untried) == 0")
        return report("Exercise 3, descend", False)
    ok("stops at a node that still has something to expand")

    # Fully expanded root: it must go one level down.
    root = Node(WIN_NOW, None)
    while len(root.untried) > 0:
        action = root.untried.pop(0)
        child = Node(env.step(root.state, action), root)
        root.children.append(child)
    got = fn(root, 1.4)
    if got is root:
        bad("the root has no untried actions left, so descend should have gone down")
        hint("the condition is `len(s.untried) == 0 and not game_over(s.state)`")
        passed = False
    elif got not in root.children:
        bad("descend returned a node that is not a child of the root")
        passed = False
    else:
        ok("descends past a fully expanded node")

    # A finished game with nothing untried left. This is the case that needs the
    # SECOND half of the condition: X has already won, but there are still empty
    # cells, so a node here can be fully expanded and over at the same time.
    done = Node(env.step(WIN_NOW, 2), None)
    done.untried = []                 # as if expand had used them all up
    try:
        got = fn(done, 1.4)
    except IndexError:
        bad("descend crashed on a finished game with no untried actions left")
        hint("`not game_over(s.state)` is the other half of the condition. Without "
             "it descend calls best_child_by_uct on a node with no children, and "
             "s.children[0] raises IndexError.")
        return report("Exercise 3, descend", False)
    if got is not done:
        bad("a finished game should be returned as is")
        passed = False
    else:
        ok("stops at a finished game even with nothing untried")
    return report("Exercise 3, descend", passed)


def check_expand(fn):
    print("\U0001f50d  checking your expand ...\n")
    passed = True

    s = Node(WIN_NOW, None)
    before = list(s.untried)
    child = fn(s)
    if child is Ellipsis or child is None:
        bad("it returned %r, so the gap is still empty" % (child,))
        return report("Exercise 4, expand", False)
    if child is s:
        bad("it returned the same node, so no child was created")
        return report("Exercise 4, expand", False)

    if len(s.untried) != len(before) - 1:
        bad("untried went from %d to %d, expected %d"
            % (len(before), len(s.untried), len(before) - 1))
        hint("s.untried.pop(0) both takes the action and removes it")
        passed = False
    elif s.untried != before[1:]:
        bad("you did not take the FIRST untried action")
        hint("pop(0), not pop(). A fixed order keeps runs reproducible.")
        passed = False
    else:
        ok("takes the first untried action and removes it")

    if len(s.children) != 1 or s.children[0] is not child:
        bad("the new child was not appended to s.children")
        hint("s.children.append(child), then return child")
        passed = False
    else:
        ok("hangs the child on s.children and returns it")

    if child.parent is not s:
        bad("the child's parent is not s")
        passed = False
    elif child.state != env.step(WIN_NOW, before[0]):
        bad("the child's state is not the result of playing action %d" % before[0])
        hint("new_state = env.step(s.state, action)")
        passed = False
    else:
        ok("the child holds the right state and points back at s")

    done = Node(env.step(WIN_NOW, 2), None)
    if fn(done) is not done:
        bad("expanding a finished game should return it unchanged")
        hint("guard with `if game_over(s.state): return s` FIRST; a finished game "
             "has no actions to pop")
        passed = False
    else:
        ok("never expands a finished game")
    return report("Exercise 4, expand", passed)


def check_backprop(fn):
    print("\U0001f50d  checking your backprop ...\n")
    passed = True

    root = Node(EMPTY_BOARD, None)
    mid = Node(env.step(EMPTY_BOARD, 4), root)
    leaf = Node(env.step(env.step(EMPTY_BOARD, 4), 0), mid)
    fn(leaf, 1.0)

    if root.N == 1 and mid.N == 1:
        bad("nothing above the leaf changed, so it never walked up")
        hint("recurse on node.parent while it is not None")
        return report("Exercise 5, backprop", False)

    for name, node in (("the leaf", leaf), ("its parent", mid), ("the root", root)):
        if node.N != 2:
            bad("%s has N = %d after one backup, expected 2 (it started at 1)"
                % (name, node.N))
            passed = False
    if passed:
        ok("all three nodes on the path counted the visit")

    if abs(leaf.V - 0.5) > 1e-9:
        bad("the leaf has V = %.4f after one reward of 1.0, expected 0.5" % leaf.V)
        hint("V = (V * (N - 1) + r) / N, applied AFTER N has been incremented")
        hint("a node starts at N = 1, V = 0, so one reward of 1.0 gives (0*1+1)/2")
        passed = False
    else:
        ok("the running mean is right at the leaf, V = 0.500")

    # The flip. This is the one that decides whether the search plays both sides.
    if abs(mid.V - 0.0) > 1e-9:
        bad("the leaf's PARENT has V = %.4f, expected 0.000" % mid.V)
        hint("the parent belongs to the other player, so it receives 1.0 - r")
        hint("a reward of 1.0 for the leaf's mover is 0.0 for the parent's")
        hint("without this flip the search plays well for one side and helps its "
             "opponent on the other")
        passed = False
    else:
        ok("the reward flips on the way up, the parent gets 0.000")

    if abs(root.V - 0.5) > 1e-9:
        bad("the root has V = %.4f, expected 0.500 (flipped twice)" % root.V)
        passed = False
    else:
        ok("and flips again above that, the root gets 0.500")

    lone = Node(EMPTY_BOARD, None)
    try:
        fn(lone, 1.0)
    except AttributeError:
        bad("it crashed on a node with no parent")
        hint("recurse only `if node.parent is not None`")
        return report("Exercise 5, backprop", False)
    ok("it stops cleanly at the root")
    return report("Exercise 5, backprop", passed)


### Phase 1a: picking the best child 🎯

Plain loop. The comment about the starting value is not pedantry: a `UCT` score can be
negative, so seeding the best score with `-1` quietly returns the wrong child.


In [ ]:
def best_child_by_uct(s, w):
    """The child of `s` with the highest UCT score.

    Plain loop, no max(key=...). Start best_uct from the FIRST child's score,
    never from -1 or 0: a UCT score can be negative, and a seed of -1 would then
    never be beaten and you would return the wrong child.
    """
    # 🎯 TARGET 3  Two small holes in a loop that is otherwise written for you.

    best_child = s.children[0]
    # the score of that first child, which is best_child.UCT(w). Not -1, not 0.
    best_uct = ...
    for c in s.children:
        # does this child score higher than the best seen so far? c.UCT(w) is
        # this child's score, best_uct is the best so far.
        if ...:
            best_uct = c.UCT(w)
            best_child = c
    return best_child


check_best_child(best_child_by_uct)


### Phase 1b: descending 🎯

Now walk down. `descend` recurses, which is the readable choice here because the depth is
just the length of one game.


In [ ]:
def descend(s, w):
    """Phase 1, selection. Walk down to the node this pass will work on.

    Go deeper only while there is nothing left to expand HERE and the game is
    still running. Otherwise stop and hand `s` back.

    Both halves of that condition matter. Without the first, the search dives
    past nodes whose moves it has never tried. Without the second, it walks into
    a finished game and then looks for children that are not there.
    """
    # 🎯 TARGET 4  A condition and a recursive call.

    # go deeper only when BOTH of these hold at the same time. One of them
    # alone is not enough, so the condition has to test the two together:
    #   nothing is left to expand here, which len(s.untried) tells you
    #   the game is still running, which game_over(s.state) tells you
    if ...:
        best = best_child_by_uct(s, w)
        # descend called again, on `best` this time, with the same w
        return ...
    else:
        return s


check_descend(descend)


### Phase 2: expansion 🎯

This is where the tree grows, and it grows by exactly one node per pass.


In [ ]:
def expand(s):
    """Phase 2, expansion. This is the only place the tree grows.

    At most one new node per pass. Take the first untried action, play it, hang
    the resulting node under `s`, and return it, because that is where the
    rollout will start.
    """
    if game_over(s.state):
        return s                       # never expand a finished game

    # 🎯 TARGET 5  Four lines, in this order.

    # the first untried action, taken AND removed in one go: s.untried.pop(0).
    # Use pop(0), not pop(), so the order is fixed and runs are reproducible.
    action = ...
    # what the environment gives back: env.step(s.state, action)
    new_state = ...
    # a new Node holding new_state, whose parent is s: Node(new_state, s)
    child = ...
    # append that child to s.children, so the tree actually grows
    ...
    return child


check_expand(expand)


### Phase 4: backpropagation, and the flip 🎯

A reward of 1.0 for the player who just moved is a reward of 0.0 for the one before them.
So the number that goes up the tree alternates: `r`, then `1 - r`, then `r` again.

This is the single line that makes the search play both sides properly.


In [ ]:
# @title Before you write it: whose number is it? { display-mode: "form" }
# A short reminder of the one idea this exercise turns on, put right next to the
# gap that needs it. The same two characters as part 1, on purpose.

_WALK = (("X", 1.0), ("O", 0.0), ("X", 1.0))
_FACE = {"X": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAIgAAACWCAMAAAAyuDCLAAAA/1BMVEUhKUzO2uUxhbYZGDfxdjMvb5mM4OxSsdJYUmArSm36jzF10+T19vRqamuYnKpPLjRWkaz7yzpPUmafWjf60EVfc4qkpaVfYW8vMT04k8JpSzrfpUmYZUcTE2auuMVMNkPSc0OukUjL1uHO2eI1OFJ1d4Zqa5WGh5EAAP/X1beqijwSYmI9QVm/v//DubaJfYcA//9vkZj//wCGf4ZucoOBOjG3xbfCtaeBgX28x8f/AP9/hX8A/wAAAABJqs39/f0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACQfr59AAAAQHRSTlP7/P7+/v7+/vj7/v4KBv7+/v+c/f7+HWQE/v7+/gP+/f7+m1eoWCBNAQ/9A68E//8BIQEkk/8WEzUXASgBAP7+3dBW8gAAFPNJREFUeNqtXAl7o8jRLmgDDWoOgwDrlj2y58zut8km+S7J//9fpaq6uaQG5J3p58lkzdG81H0hON+xdvi/f73mp9Mpryv6++ktPn9g4cXqNQdaYR6r8/np9na4c5sf0Kz8TR9+UvcjUTX0Vk4vE6sPAiEYdYivUmZBEGQlbZTXTJj4f+8CcT6/0v3Zwse10DvcQoF5HDUIKAO/WVnIr3Wqv/LZO3AgU8pFez9iwQ3zWg1vnwHydK7MNotFu1cGIKXA91IsPZPrv84KNwj4fr0D/otQkERDKDC9C5IDwgYG/oPcCfBfEM+bLW5Wn3fTkqKeEEfYo4ZefkBklfW5exOYZgu+TdbsEVyaFVxAbJYbiUQ5z1G0w0GvoW8P8K3KVmxngcTnrx2OHgzaKiMkyw3yWu3iqS1yES4MjP4GlwUzmGhq2AMTm8QoVIHP+wxgaCRy+bjcpJM0ic+vAAvm6+JyvRalkFt6E40ExokaA0mZfZdL4AlE8riUiORpFEdFFGUkF8sWodhuDiA1EpigB++ysO9yuYTiGWmCSOoxLSYB0ZwNrDuQpKGgya+0AUy8jChvpaNbWkyWG9YdO47aCOrIDhcX0ihaE02+jAHZKRLqyV2CDJmD6xmgstEkPr+RgNCrjOxwWXmiICQ5Xgxj0o4EySZ3ITHZEpIthOqsbm3QV9rBLmENECZJJJCmT2DH8RnSlF9nfBcjJign61uBjZ92+C7lNI6L64YCgUT0JmB3+yFsRUjbBBM42Jo8ourwK11vwgIyiQOBMG8iEniwEqSGdCO8mW2YOeulFtj/G4pJVdfsYfzJN0EgsGaSgLLLiITjEXVmBkhAmoMr3ZKYdPFJrHQQhKHDYnID13U1kDXUYFfdw0YDmXqfi6cDgpCccU9MSG3BwxUizDAL7gBSQA52/S+ihxkgJCAQ4uNcIvCzhNeOOZJxMBaEcgeQSEiwG8Ti4YGEdQKIp2HoBVv0f1Vr16AFQlDgEoxpbwsECWvTmZN4eHhIYUJ70U90MBDIGq1J3gD5O4DbPzmGhGlZjFPkfA4RSLTluGpUXVocrkeyT9ak1WHZowidDUc5EwoNBAXNBgQAgbC02oEErgj1O4chS6v49vi4MaZeoawLXnSKkQgvGOMMvXIUbaysoTdC1kQHoNjQ7jYBqYo0108DHRE8o/fafSERE2WGS6MEZk42xpn0gdYRpFVGfsCReMOuws4Yj1XX8zLthylIMrHJrvG5FN9eMg91y7OShAiChpVwoPrWI1qzjfC8gJFYJARSy+wS6CBUB0loYcnUU/ywaE0hXkBQIAxsBHENQVAxYitFalQqPF2Q/11YnR3D6OsQer/l47MApUIiZM+0M5RbiiCMladfmF7ZZuK/kwM355HEtpgovOJ5QN4PiYI6nIsbn4scsoQiJCEHTRA0rFant9MWjaSktJPEsKTJDjAVRUvyyCEsmOiBTgZBC8XKGC0hD0cyhqPel67AgDIbCxXx4Y2v8TArxkCYNGfD8TaKqF7emKdx2YZoCYnIFj6NxCPA3IuOLHh2V4OCIssEl0RpdDnlQoqggJSUKyfJHs+MOT3WGMN/1F0yQDAawh9Zrzg8sruaxHk3C58YciyNvERxKdsTDp7xZhiDKlNPxqxyw1eR5tykV/j+yXt/7dm+IgEwa3IGZxKL03NdHRJpSRVSjaYT8U5JKFrNWdxkEjB8Gi4pyjLMyiuARBUUmyESt68xD0J7hgGQL3E/F9AstGmOBQchCfzsFgcjufJU2slELWOeLJnermdeWagj0pwBc4LQhgORgBUHIhEDw6o1RtM7akO7HpAvZ6V6SDDFKiIt1AOSoEG3Pg7fXEjrCZQTNxgKSKe5YHIiDUQpbT3g1CZtnHQeIzZrQ82Bkcfh8xz7CUfClaAazX2ANsKEjicVar/s0kcKXQ+t5rRmDQmyHzwELYl5focQD26THqq9aMwJCwhmTY3mtjF3R5Hvr8R6CRRTDL0wueGWOUE40E9HUj7Bh5yOIM6hPWq41kiJJohsNRcDyw4IpSHIEwbidAEfM6fQN4SdmEA2ZMAhTVOR4rESevjS5qj+s4QBY44GRy99B2YB2em6AqJ52UiPYqnRzFy3Zg2jxP0QyO/47imJqZQDZTZHDas0bwYCchT90gpRhHmbwJ9QvmuSxLqwtYsbHe6YgyLid6bd4UcCPTN559vpKMuLPvqbAeIzEPL9reZuDoOMuQWCSLR9kCcU3O9abtDA8m0YSRvmBB44LZCDdPBdKWillwej1enWcaQ5ujVAHKDQaOBzr4pemjU+XV0KNhB7zpSo+J5XWoeb0IQ0h6xZy3mUAuPve0CujvaADBhTAPx7WAKPYwVavqW+T+ZUX5UJehTFOsyidSAdHlKEJJKjeBDr9NABSQ0QPNoA8RGIa0x742L+e1gCj7mUwZqAToH4n4CKedOEhTo3yqbFBGWkx5oGiNziI2V5BY+PtjIyMO0P4roGiDJSvalcW20HIxp8CqbmRmrU+a31wxyaYPiJWtMg2Qp+JhLyN9xAQqMwzdHEOAOSpExHy+vo2pI1QKoT2p5ckuo6xlYlgk2TDycKkShxMzrMYoLEankj0vUaCSB/k3hHY+F/a44m8tCIiAS3l09hHiOvi27wAzCoo2jPd1pnliSaUWxR2ggWXwPjUbKsnf6i+Vyv5Zq09729//f2qGNwoGXtCSq5un9fFyJBa/+e9M3sk+yNMTRmj3LI1Fgg1GG3x5v3JOWlITS8MUfFwXlvOeP1o3aAzzc1NzgZj+A0LivR0YbfmhtmTtH64QB1rEPi/C6lNP4tE1kXq+LR5g8kCLCgrhuXa6n9wZvR3ffWh5pdoS2edrE0BekLDH96SAZxqzZIVzGAgwi9LgaxCKrWGoxCBh7b0LeEfu7XRAQC48FFiFb+Fgm+dgbh7WFmzGrVWrI1Bsu39WGyI1UOYdY6kEQ2gV938d+/q741CcjeXCEhxcioeTI8jpc5exH2LNnA5Q4ogrFhdQJokhFHmvfr0y8+fw21NWExweh5z8/onucjPQKqz/cx0iXkjFhA2nw7tnYzQDtaVbe6ayRG5Go3rPDrV+Jw7RJiFtVf+DT281RrTPonfMn0QEGNWotq7e+Aad6dT31BcxIohza4sybU1Qoo0yv95mEJiNBkUQFixLC/wYdWwetbssNonwmaIn7VaA8TNNu/h/lN+WatrQkxJ+D2ryzLUl6VdTkrpjOSzKR34WKfsWTcPvgyBYTbb1pM9LtcCQkFbEobJC5p6VpQrxowLBToyhaeoP/uLBnGqF/H2l3QrxPJxHf8vbYSZFj/6F/51uQXVAoOmsKI/ue2ZqFPBJz4NxZkTGGuupy78z9O/IZsVUpxw81dIyYpzHQtLllX2CKfawhCFjWOn/SKr6YUYJBtVjVpAycJZUjyra7EhLbEIG+ma0F8aXI711uFoi1QXZUN35S1AU1UyyGjkJ6UMey6ww2Qk/gWRUuYo0hGvf+sBeKJZ25BIGc+v9Y5r/pzpdqH3vZ9q6eK2YMmmZP+XJ2f4i89DaZK2TPFJf1iiY0kGXhBrzvUFtj6S+af1Xm8E66qzxw98H6SmvfEUFyU5WweHzf9vA+vCa+hBAGVM7Ms6FXcv1FlWkDIxVkedsh0CY6mA8Y74TUi8VwqUNGVDR8rLrlzmaxXRc2IjUG/3qjLqt0xjog21PlzV65WpsBMTZAxynV75RYIyjWNODASNlsyr1/jP+IYvQ0SeH1VuGFTygYjaGvewwLeiqRkvYzQByN1+qCJNCEPKox2wn8wkoCLmC1PidPfKNPyb0YndB+Cm2dwU71bMXOW7Pzwv1d66YtoGAX+HCvm/T+VfMHTRMHEyDTFlkZArnWG6QC6MYK3BNZuptg8D3pN5IJYxRc+qae9vHlWO24Qur3VCKq1m0T0ps5Ir9h821ddFk0Xh7MvIqJLsAlJNTU/ErZQPKItzUZwVcAfK0WTjLDeWxorFAgso+dnpIvUjrmkcrC3IiTBSAPaWA1VS2j7mNwIkRO94JWh3GgfEfSYBxWFu2ADLRYK4iIca0C3A2imkRwaSxZOzj6gHE42vUnIWO+SXnHN5eEa+J+JGaMvpN7Va56fDp0lm3N3k3MAbGDJoPQyNE9P+VQwPbyl/08SjqUeOAr+IpDAGNhHoqy8AuJNsaY1bwpgTlDvI4nWPFxbk/Nz5kVAwnkgOkSk5tT80MI8c8AwR1ImH5LtCSmEykY6WNeDAumSPFbo+z+FQ8+tFEwSFLc9pUAUXFPXadKOtLM1IvpZQe3EpGEOemJo6rTZBeR5BzM4vqL6k6D+pIB0zEkNc0TTX6JqYIVp08xobw4FC2r5Ewpzq8OIQyZNvbYU41NX/Q6wsWSLn2WMQUIUZmFtSgUotSc1N71Zw6jL/evMEYijy5r3mM1hvh9PTm9WpnVJraPFDGO0rwnu0RzZ72/IUlcvYdqCFKaFOiuobbhwD5KyV7jwRcIVEJi0IHe4uq5faCKGyxzlvEFLkKrl1ZSwVh+xIG4XQ80iocyv18dwMpFPAYkp5/7GTjubxaHlowmkgjuYs+9XQGBKRoyAPG7vsCC6yh5SwZkD9Q8xxykldWbGLCuV76TW3HkLYuixpvTcu0NgyayVXe0tLOF1DEiruWJi9Ko/K9SUHjhlmAjUzGABmjWn7W2BhB/quxWI+q4kxyD3aK7bb8MUXDG7gySmvOzsyeUkYmSM1KTbd/n+lTscfbhLTIg5iQlaqVA2AoQZozV3FkcjqISjWDNz7hATCurBpxDN9x1nPzbIgj7XaG5gnzG6ZcyaS8ooJ1Svv0OHmTlN6ZHbICNjpMLUQe4UkFRX6hIhab78DuZwOgzZ3vcpSHu1GzSy7Rwtw2JOc/uCSkmCFNvoeBdz6DbK2IVYb8enwAH4w4jRWcWhZdc9FFZIBxBJoXXYvSOmpxx0uQS70yPOSA7bgxnGcD+ZBzCpgZKYFgUJy7yYsOnRkfSSJiMsCRaFZRwewozGaIVhSxa1CW1Cf95hYHUdWOeghd3Es5eJIpoC9xd3CkjR+dPfqU81LyYBVxg2j1EUPX6jmVwbkBNwyT2bmyXvGsq9bBYFtqCpsdW0qQ+4+cojOzR6bbMjBkgB2cxwvXZ1BZvTZDDhU8zrcGDmjJtK8CiQaAZIiwMFdT0cOkKBPW5ghjmr3ihaMcaaHMwM6WJGcdGSRSSoV0NHCRw2W9bh8XCtM8g8VmsHwmPxk0B4pE1b1OgINzNOgEjWYjJc0wq3Nl2cz7YIjZqr7L3KUSArHYKwoB4B4HYijQflJ5gTuL1B43Qsr1H8BG5UTXh+j11/dLBMn1EYup009W5/dIIHBWB0rjaFxWTZTocgawgt44mCBppMkLSaCGJM+4T6jWD/ZimlxmrmT6RSJgQRoScsveiQkKRah1ejJF23U5M7+6gxBorbCInuX1y7XTexMjXH3FsRISEhST4aHV5ZVa7tJ3ExYDRCgyP5j4U33CVo0yg9BUpiYBtQpOkIvGRr/LBrIynrfkuQsZg1h0MUpSIjB7u6rul6TR9X0PNsI5SJnkqQuvs8gNJuYiTkYPrLI3kNMgetnhiW47u0Um/CTtazTUo6pv0tDyM7tCpDoxPfx9skNEFJlBNa3G72EHrsgZ9mHW7lgRE8J2EEiRlHS+l7i6mPSlFMJJL/eIuE5i96OFz7UGkzU8QNQitNTT7WfFM6lYTnGPcdDxAOYBBb9LsYQYTEPtpK3R4XRpCEJIKkc3k7SgKTH4OD3ErBfRVGwemz9t2pIIKsViPjvnuaXTG8gWuq4rusN9H2wE00NfvhMV5B34PLLQXbuvdEnTIaiY2Qt4BA+GtMxz527K3IYID+uYDQXfVhIKWLw+D75+li3hMShVqiB8mNJ5qVbT6HgTwHj3QxBPt0MUorIUGd4C30m7j80Y846E/bq0F7G2Z/M4Bn/Y7b7bYgH0emMCWanoBt5NgAtASORShTILpC20Izi354YPfl/s/1Yz3/i0by+BDpcAphSLK9NDzcK3RcLRq04hZ2FWsW9xvg9T/6zfj7fsCAWjbmdyHE4aB/uuAVj/+hB1THRtNJbVY861mb9joNBfxZ169visXz5tcl5n9bgrtHqs5P+jcq8G1UXX/OeRzC9UYn0DWQDPLX+vP1sNdTfP4rP3LRfMmg1Ffe0dCZhqxG7KqOBFzdsKeFWfbuCV/oaUczBjODLNNkic1Qy+7pFeTzZrvGXJ6+YB4b1ifbugowDSyeN88HdLBfZp5wJ5BBjL/cYH72kIbBHJALGnL6HHBD0fGvBhLnQDjQQMOErNKso7fKgL9LjJ7hNX77tUBiSjb6QPbv7+Nq0wDZojWZ+92UDwEhj12j0ysYSDhJkZKMPANB50ZV7tEfXPg4EDJNJ0GfFVBJRHj8UeL7mPqiVb+EKcX55T7RUOJfAgTJUfGUukxQOYsH8j7pdkx9E7HlC6JUvjtSJpjq1GqKKPABctTk/0rHKWH/Duj6yAOOCUmJwRNeEBXUF0rek/KAuH4FReLzVxoXL5Dh5fseJOslAUnGgfDMl5SABEzpRmHs/U8AUVTk4nHx9GGzlv6h4KdMAaEwjq9Jy70oNgWUyb6c+I2QWaenqr/R3JWCZia7iIpUfz/N2dFgerVbpR6w5B8FQOqlusmbUAGCNlQfAqLU38wNlcobYXDM5+QaBk022oGA+aiNV1q2HDNRGW6t7gOi9PyZUp8+vdBYQ2vL92kHY78fpQg4GaSGKkXY+cLSeXn5pEGoSs0DoUvUp5eXZogXwnYAWFKcVqQiJDUGOw4noVGiDDi4jNb7TqtNBZ7QnM9ndQdFPr309t1DCdRE55dN/5nS3Pi7L4X0nTEk9HGpg/Ip0n8KfWOSgQxldwlhUdNAkCHDbX00HYnUYXAoy8ThD1fHBKSRV1EiGf2sNDfKcv8u5eCaT2oynUCg17vqz00cM4PuJPimchKGoQrad6e70cmuZeprPJlgqerTDRJBvPF9P0nKUD/gjsXfqSAF+cYM/8icj1BEy8jtnmDyApndhcJpJt+aZCIsByL18ulfczLC4qyG8soymyT4cs6Hl7+/uU9rzR2/dRVXjdi+vDi/dH3MjtAPAHUXkllDQD8FiW7/1MqEqir1IV8T4w1V3FdswsSgXu559AtdjTep3o43IGj9B3ST6KhuhGa8AAAAAElFTkSuQmCC", "O": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAJ0AAACWCAMAAADzP3onAAAA/1BMVEXc2+FrFROdIh7TMTDpVVLeZi309faSXV3xoUuymJuoUCRyHyDsjTbLq62TY2F1c3OjXV0hDQWxoJ2LUVB0NDPk4ubj4+Z3T03Gu8P/AADItrZzQy+wkY+7usR8TEx7Q0M/DgfcPkH/f3+BOThsNjabfICEQD6r5uarj4+8wMlwPECqAAAA//+pg36jgX94QT6qqv9/fz9/f/+EP0B//38A/wC6xLp8PkDQu7ieg3vMv8wAAADWKir3+PvjMzLhLi0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABbHldKAAAAQHRSTlP9/P7+/v4M+v7+/QP+/l8DEv8hmKCeXPj+ASL/Sv5njwT+ArNo+7AJk/7+AwFN/2kDBAL/AgEapng6FAD+/v7/aEo+BgAAEfpJREFUeNrFnAlj2jgWgCUZW7HBxgUakjTpOe3c1967dsz//1f7DkmWbBkMtBnNlKY4wMe79SRZNFeN9ZL+yupqu9NSKvgj9W5bPWbmF5bLj1e8vbiGrcaHh2r7SdJQPPgfcrutqgeCXK7/AjoU26bSjJUs/MGQ+LitHkjGL0yHn1d9IrRFfJAUkbDaWDm/EF0Ggqvww5PFsWHVvK0v06+4VKkVqm5xchgBSv0Ir8pegg4El22HbJ0/0hQfP9CFVElVFEp+2ZyvXnGRxVWBTrvp8cHIr7gpQL9Zs7z/xnR1KLgoVOL9THxS3dzAi6qmuf+mdMtm88mxpRMiS5LEIyQ8mRrx1d+Qrm5qFlx5TKWJJ7+kRbxEgnZBfNvvz4l94lybY7iyPGpuPeYB5Ni2idVuIdWmWX4jurpZM9wptsTTMdAZvILw6vl44jzJAVw7Aw4khho9MB3iIZ/Bk/PxzqH72DzMkxzgHZLDoYX/WwPX46VSrufa3hl098tsF4FLeAxkRzz0YOAQL7F4m5l44pxYsh3C9VQBXnvAiJIYrXp4rZQrsr1sXlgWZ3hENYBDIvhoLEPazhcfEpliT/VwKEeIezeIt51nemK+R2wgonpwSNO6WlP5wgOJSTdUG+KBbm+UrGbhifl6/YKiu/HgWuUKYnBlkx04ECtbKFMR6tEBXkp49RzTE7NFB3rtejhAUK5WJwZfQh5deA2lCkEZPSObUVCJ2f4qZeJn1sQHsBKyntANLvZ8VrdoeuuvRbdml/A8Ivh8Q5e4EV5Fxfd45Leg2xlhRcyEy1RgdL5w/AkZpy3yzXCeNhJeOkd4YmY02fqis3DSG72gKMap8FKgWxZeAX5bfw269T2ksN4lEvwANaLyaZDPezoQHic0LKey5degoyyROpcg0cXQYP6AMwhprZBoWHKe8DChzRTeHLqsyVB0C9/qAjTyykKDYd4sCsvt5diB284Xnpifw7w6XRn1tQ6ugOs3aYFgGgf+QtKXAL5q24Vz2/qE8MQsxRaQw/zKBNOoVRvJJV2kNG+QuXimITThRUdqhLc6mW7FvAyrhtMvLtuM5FBwKQouZzIi1Ea845FwFT8j5ok5oqsoTQxrur4EAKPD0lIbuZHkYkWAU23KfpGeEt4srwh8wtZ1WAPovBTiLqc+lNQ+m4YLQpR4SUWFZ/3iSs1+RMV24dw1oRqg7GVVAoX9Vw5Xnt0VkUf4wC9ItcUJvxCzFIvBbgiXu8+nUVom7a7wJXKQId3CqnZ3VLVz6HYDswO4XlKWTgjG0r3g3JV8gJekM1Ub0GV1HQ3FXp6w8W4Mh3jiWUevDPAS57WF/O8x1YpBOzMSin8dKhbgygiCEBBS3JXgwki5NiCf8FoRSKmq8a+T8WRgWZ6InOOKwfAqPDu3TUm1+rRmP9YZYNQQGHbfD3pYy+aTGsQTJT2XFD6L06sYMuaD2Y81PAWGtz4tu/dQm2t4/91wqjkyu6Hocr3XuUHQ7nl4dg9pzdNtSJcwXSEfj6iW6TZVhhj4Zupd8NtrE+2iLkEBI6eUsGc6YVOZ2L+SlHajwks8w6uO02EDQsoq2xBdDrnvflCfDOi0r1fxXCLGK0nysXqFC2/oaYdXhn5hI97qaMQTaPaqFDkYAPgb2u+ueU/TsCwzhWcb0CV+HKYHiWJ6BdKTPvf+VSi9wC96t1DHIh5qdqc5OWr9TALIaIrYUAsfnEImAV1rgwYKLkc5vXpFILmw3GRvkp/dTxmec4sjdQrKbkdUAjInG0hFKx8ZL3I1oNjUd4rWGRfaHIjmjaHTd7aCQqT9K/N0NOT1dEdTLWuWXn5n1KAl9tXRFrcZu2zqm51P9xuq9JXh2GMoRkMU/rNOtXrkFsrOLpbTdNkaQsmzCab81yMylWWuAG8zQUf+qnuMQu5RduQoPd2bN3G6mdlC3NOKV2neoDSWh/aNnFnzSHVxkClKV37kjmNfQLTkb+nRSf1G3k3QpdI47b+Pya55eMy2Fo/ePpdbCi536CDj8kmSK9iyiUE0Fk7ss4C3N8+CKnqvkFMh5UguE9kXrrD5LVgomlR0V0JwwSzbDWoA7egEcFBc0/kbyYnMmAg9XeS5U+wg3iWW7mgRJXZFSZWt7OHg7a3oMNyN6KSXoACBBll/zpaBmPZpPVVEpX7Amwopgt7wmTzCZW3SD8BhJBoGY6NaO+5QcmD7AHf3lBv3Ijx6VosJxVJIMXQPR+jsG+b2M20ax0YCFsZAFy6UKOnXRp/fkGYB9OlJG/sVrI59kGbHdC7gTdJVVhKu4OEfQHTNe7AIjZoNF5da942C8XSXe8F3VN8NinePbjMZjgWYvRG/0ytprNS7XzDfjug6mGQ7EQUDtFx6uhzUxtN0j9Oyo6JT+5/GblfK7ygOadWN6Cbw0GXzKN4Ibi4drfHvQj6cQefyLQl8THeg5mEZUV5Js1kdhRvStT3ddCoTPNmpd36YQNFpeNGaNBulU0OI3E4icbIdXCllpKPiy+4oHe972QWWnuc/GU9HunRIx52n3Pt9DUnB1naBpZSad9NcTocl8IMfxZ7yvCz5VegVcTrqo9DA6OHmGiw+vqRpX5QM16OCXspqDh1V6A7vydZ5fzJdF6VL08I2oXT5HI5S91dKXKaakN0K6X494hVuevM2MCXMY2vIZO8gGsfoINKn6Sol93iODRfZ80m6lOkeT9PRtJr0YeHkullyrugiXkF06UrK51MDS6cxnUK6FdEdi8YeXvbW2jrAgday+5ry7GDZ2tAViOe1e/xxp7XrNUbplEoN3bEFvWEfZUva/cxLILtfmnU1oDt0Pt1KxenIUYSlaw+LMZ2i6nN1vAoYbHCCQp0b5mRauw1NZ0PRzaGDsKLt/FFiHRHS2dWrIqUKql7P699lTbbdwYRnkZiGdf0AOgjpOkOnkK6QoozCSZCf+RGzjU93UDJYBprePTjsLlIb4C16GQgfUwKu8oSKHdCVEdPL/e7jkA7h7C6qFBF3VTYR8cR4+9oSwwivxSXIN9QsGF5iAh6EFEiu2uu2Iw42yvonSup0hKLDCINhNOUtXHI/scFHTDVj1SKxa5ndWLOJoQOn1SQq6kJhkCuHiYNKsHREZ9aZD7iJC96riipXxFeyGS/h5dTEKpV8InTalG0/9xc8c/E8pOtCn1A9HowO1y+iK3oTXe2fcVXRrrQmgcey8FpjeKkNxyA0HHkphv5BJVgbCq9fbU7sFGhzv57dc1/y/kR0DpCisTcjO59uMqSEZneIRDuzFEQSXCyinTJxZBfgjhYKQYlMZ+B62UnjtCfyWDKma43PYxmhcEMXZt2syeavV9RUM5MGFlanhu7Q8iqZZ3iTg1pY3Vh28GrMY2AZCle2FirmGEdWU+65ZoaxYJ9wdFHDO6bYLpbHkA5ED4kOhafNPGb+Wg96UbbZyg/W8A6ealtTphw3PKfYw4gOawAwW+BPUbXy9zPp8AwApV4jvF7BRngnVStYdIeIVyiEW0GglO2CQkN1Nh2uTmVM54cU+LSEcvkprwXRpb1iDz4dKPaGFKsWTLc5y+68DRW2M9t7hl3sp2SWHxddehjToWLRJbQ0dFJGtiLPXAFdBHSd8cGTwtO8iNXGnIJzNC3tqgnFzqCjRcaFhetcUDE5iYQ3YXk5iW4U7Ey0uymUyTIQ8zDc/XLZvgBQbR/yDKNNmcUKhVdOR5NRsGPFFmBzqm8jxKeNM+hqq9o+pnQHD29qdiEoEEfg2CRod0iPF8kUs+hYtSkHZJtx2970CtTtGK/kDXuLQ5ROyZ/81Uns89cX0VEDFFXLUmM6a3k9XjmyOYKbEB3Wp8JfO/39Qs3Cl7IBuVds10dYsvBUBQUytp0S2nUWiK7/RtKHewK67y6la5bZJ+MXQTVgPow2gNJWHs3FHXd2wOZunOQOw+qp9CVX5nl8tWze/rumdn4R8B14DyrtBaSmiuIdWiS4XnKHMV249n1W5R6daCwWw3zR43VJsDkwSVNkM3BGykbWprkWGJ3Yy22W3V++s7Ie5ouez5xXSHh/NrCmqTlU43IeMx5sZafsuoaj218c72xE1mPL47TWhf1Hy9YdrNwOve1xZecWz6xmf9TRFZ/ZO3pplxtPt2N4w3NknfMH5nMOjltm00HvNpf7+GrZ3L3aXsLw8sUEX+rYWlN3+nBqdaP0QLGfddRp5+9zz5TNtiPpWUC/CAzp/EMCKS7K5p+91aKfIODFCvdzzgjUSvXFQIzQhUGPjnzVAdJW6BU1i5XfQv9R3Mm319D1uu2OkAWhZhB3qG1Ey+1Ih13MJ4P3I++VyK4/m5K4fsW03CI/W6OTK+7G7ivAy3W/3nJNROHuVCYNXndcq+OnEwuXmknshnoN3BzCptU1ucKa3r8UNX9mKneQji0cFjR4aGvz9i0ewdV76pC9v/o8GRQrl+CZgwM80yE4bDiRIrPHarfDs8kzu4tz8D6cwWcOlAVwHDzW2Mc8tjHxgnOMhLegE1Bz7M6DUynvPwkiWw2Iy/XXOIvH0vt+a/DGgO3wB9NBJMHB3J/h3n6jc4yMx4d7P9gDeHEJtg7NaLVY8b6imWe1LqTDGTuddl/0RwSjnuCxGTjs6ZxxxLK57FT5e3s2OgkHVUyd/wynCCzsTSTZnXN69sIT+TC3q+mIxZAvHHZijWw3OC+S2zPvunDZ/QLwTH7VH7OLg9lFZmKDmT8KrjnvlgGX3gmCbgTxxZ0x6k8CWjx3LoTkVlAv4iytXkPX3NdLPNrgnfBxbP0aWIH2xkvguCv37HtBXHGPj01dY/MoPFbWz8wKDw1Xws4WXHPlHUig4AMVglPq4GgbcBEYkBE53eNjeclHXEX3TvLKAG50zHV4Ks/+i9CaOmtelu4eW1NMZw8blTl3AhwkNQ3/knvLUPuCTkbHeoulpas+Li9Xj7jK7BKmi7cWmW7X/CV0Pzc7lXSUDaIrUBQI4630l/AK3CdNm/H0xC4UZ3gvT0e9i5QUm0/025muvuyGQdfRkdmlU2bnu8UVhicuN7svoNhpOnALnF6nV7mFuM7skimzw4VZPCiaymsM71I6Oj94lA7PU5BqN82LR2Mv2uXTW8euNbzLamOY5dU7olOTS3i0sR3p3r2HX79/Ybsz0W6SThCdAsMzwn4pus0WBtAdEO7N81G6hZLb3bZ6qRoFt2By57zjnb2Tuylk0u9fBd/4+AJ09hQ3th6OOQUtHdMyFd3awJ4D+8Z0f5Kz4pY1OmSvpjfL5Cy7D2YLanZBOXC+7Gqgo45cgmI5smWhRDrJdxVAuvW3p1svmwewuAOvLaqJ8ql3C3NwYaF2DR2I+JZ0dMtAZSbZim/eMSm732hOpOyse/sA3+0UX5Zl+CfLLqBb8tajXORKOTo9KTrJqRZcQpelOTzHHU/sryPDOvsbPG7oJ+JqbLcg+z47s6vdNA9bYMtxK7FWrd1UGmejEorWQhWdltI4aYOwV1PnP5toWWT/yDzRnUFnbhmodE6neBGPCzgR3USOLov7pbpE4eFQ2vwNgqZWipNddntLD7e3r2EIfsBXww+38DtnrPVkn6itQ7KQf6cw1o7o+hMuSIcLVK3C42ma2gbUsfhfRjSMEZ6ICf4N3HPp7vFmVQVO7/FQSamNY4R0wbsTHeElqtwrXGAucMdZ/lrMHK9n0+FKCrcIleq8m52ofvvd4LvjRh6zvNdi1qPNeoh3N5dOZLNXjyvcuMTvr8K9G7ED+Eyn3BZ5SHxpavHmocGbzqSjrsQqXdzw+/NOUt5DBj4bf/dc0hc58NqtKm4sXvy419AAX4PDzN4XsJHuy4PtfUhMyzUtJg6Q4XK6IjsASXfpQq3cq6dfYZz1ljwZ48tcugpDr/v2VECRj0Cg5bM95d3dwKBK3uVTcE/K6nV8mooCyWuH5KeN+XTpquhv61WwFPCoIo27mKnTppnUNPLswJ2YPReqz0dac+awAVnMTBOZ+RxPvciGcEfMiG4twz3Q/oXupNdtZqW0of/g/8uqgE3zqIzEUtfT1A7MxPnXE4D+K92p1deY0LI/sia7ukaBLP1a+23h/efcGYzVBGcnzk0+aa79l+ZPqFH8/T+yr1hBZbfV9t1ut3uHN/X2c7XVT4NN2H825nmTRzmJ/ueHH97ttvDSHx5vzfXN5vRH/h+BeMSwZBVc8AAAAABJRU5ErkJggg=="}


def _tag(text):
    return ('<span style="%sfont-size:10px;font-weight:700;color:#94a3b8;'
            'letter-spacing:0.04em;">%s</span>') % (MONO, text)


def _chip(mark, value):
    return (
        '<span style="display:flex;align-items:center;gap:6px;'
        'background:rgba(148,163,184,0.13);border-radius:10px;'
        'padding:4px 11px 4px 6px;">'
        '<img src="%s" alt="" style="height:32px;width:auto;display:block;"/>'
        '<span style="%sfont-size:15px;font-weight:800;color:%s;">%.1f</span>'
        '</span>'
    ) % (_FACE[mark], MONO, MARK_COLOR[mark], value)


_arrow = '<span style="color:#cbd5e1;font-size:14px;">&#10142;</span>'

_parts = [_tag("LEAF")]
for _mark, _value in _WALK:
    _parts.append(_arrow)
    _parts.append(_chip(_mark, _value))
_parts.append(_arrow)
_parts.append(_tag("ROOT"))

display(HTML(
    '<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;'
    'border-radius:16px;padding:16px;margin:8px 0;">'
    '<div style="font-size:15px;font-weight:800;color:#1e293b;">'
    'Before you write it: whose number is it?</div>'
    '<div style="font-size:11.5px;color:#64748b;margin-top:5px;line-height:1.5;">'
    'Say the playout came home worth <b>1.0</b> to the player who moved into the '
    'leaf. Walk it up. Each node belongs to whoever moved into it, and those '
    'owners take turns, so a result that counts as <b>1.0</b> here counts as '
    '<b>0.0</b> one step higher, and <b>1.0</b> again above that.</div>'
    '<div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;'
    'margin:14px 0 0;">' + "".join(_parts) + '</div>'
    '<div style="font-size:11px;color:#94a3b8;margin-top:11px;line-height:1.5;">'
    'Miss this and nothing crashes. The search simply plays well as one side and '
    'helps its opponent as the other, which is a far harder thing to notice.</div>'
    '</div>'
))


In [ ]:
def backprop(node, r):
    """Phase 4. Carry the result back up, flipping it at every step.

        N  <-  N + 1
        V  <-  ( V * (N - 1) + r ) / N          with the NEW N

    Then recurse on the parent, WITH THE REWARD FLIPPED. `r` is worth
    `1.0 - r` to the other player, and the parent belongs to the other player,
    because V is always read from the point of view of whoever moved in.

    Get this flip wrong and the search plays well for one side while helping its
    opponent on the other, which is hard to spot and easy to write.
    """
    # 🎯 TARGET 6  Three holes. Both formulas are written out in the docstring
    # just above, so read that first and then transcribe it here.

    # one more result has come back through this node
    node.N = ...
    # the running mean, using the N you have just increased
    node.V = ...
    if node.parent is not None:
        # the parent belongs to the other player, so what it receives is the
        # flipped reward, not r
        backprop(node.parent, ...)


check_backprop(backprop)


## 3. Put it together 🎯

Every piece is written. The search is the loop from the top of this notebook.

Two things to be careful about, and the checker tests both:

- **`expand` returns the node the rollout should start from.** That is why the second
  line reassigns `leaf`. Rolling out from the node `descend` returned instead would
  never use the node you just added.
- **`backprop` starts at that same leaf**, not at the final state of the playout. The
  playout left nothing behind to walk up through.


In [ ]:
# @title A perfect player, and the checker { display-mode: "form" }
# A perfect player, used only to grade your search. Provided.
_SOLVED = {}


def minimax_value(state):
    if state in _SOLVED:
        return _SOLVED[state]
    if winner(state) is not None:
        v = 0.0
    elif len(legal_actions(state)) == 0:
        v = 0.5
    else:
        v = 0.0
        for a in legal_actions(state):
            candidate = 1.0 - minimax_value(env.step(state, a))
            if candidate > v:
                v = candidate
    _SOLVED[state] = v
    return v


def optimal_moves(state):
    best = 0.0
    for a in legal_actions(state):
        candidate = 1.0 - minimax_value(env.step(state, a))
        if candidate > best:
            best = candidate
    out = []
    for a in legal_actions(state):
        if abs((1.0 - minimax_value(env.step(state, a))) - best) < 1e-9:
            out.append(a)
    return tuple(out)


def perfect_mover(state, rng):
    return rng.choice(optimal_moves(state))


def random_mover(state, rng):
    return rng.choice(legal_actions(state))


def graded_positions(max_marks=4):
    seen = set()
    out = []

    def walk(state):
        if state in seen:
            return
        seen.add(state)
        if game_over(state):
            return
        if player_to_move(state) == X:
            out.append(state)
        if 9 - state.count(EMPTY) < max_marks:
            for a in legal_actions(state):
                walk(env.step(state, a))

    walk(EMPTY_BOARD)
    return [s for s in out if 9 - s.count(EMPTY) <= max_marks]


def agreement(states, M, seed=0):
    rng = random.Random(seed)
    hits = 0
    for state in states:
        child = mcts_move(state, M, W, rng)
        if chosen_action(state, child.state) in optimal_moves(state):
            hits = hits + 1
    return hits


def play_match(M, opponent, n_games=20, seed=0):
    rng = random.Random(seed)
    wins = losses = draws = 0
    for g in range(n_games):
        state = EMPTY_BOARD
        if g % 2 == 0:
            mark = X
        else:
            mark = O
        while not game_over(state):
            if player_to_move(state) == mark:
                state = mcts_move(state, M, W, rng).state
            else:
                state = env.step(state, opponent(state, rng))
        champion = winner(state)
        if champion is None:
            draws = draws + 1
        elif champion == mark:
            wins = wins + 1
        else:
            losses = losses + 1
    return wins, losses, draws


def agreement_with(states, M, reward, seed=0):
    """GIVEN. Exactly `agreement`, except you choose where the reward comes from.

    Same tree, same four phases, same budget. The only thing that changes is what
    comes back at the end of a pass.
    """
    rng = random.Random(seed)
    hits = 0
    for state in states:
        root = Node(state, None)
        for i in range(M):
            leaf = descend(root, W)
            leaf = expand(leaf)
            backprop(leaf, reward(leaf.state, rng))
        best = most_visited_child(root)
        if chosen_action(state, best.state) in optimal_moves(state):
            hits = hits + 1
    return hits


# ==== display helpers, so the three result cells match the rest of the notebook ====

PANEL = ('<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
         'background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid '
         '#e2e8f0;border-radius:16px;padding:18px 16px 16px;margin:8px 0;">')

TRACK = 240.0


def panel_head(title, sub):
    return ('<div style="font-size:17px;font-weight:800;color:#1e293b;">%s</div>'
            '<div style="font-size:12px;color:#64748b;margin-top:5px;'
            'line-height:1.5;">%s</div>') % (title, sub)


def show_search(state, root, sub):
    """A finished search: the board, and what each move earned."""
    best = most_visited_child(root)
    kids = sorted(root.children, key=lambda c: -c.N)
    top = kids[0].N
    rows = ""
    for c in kids:
        if c is best:
            ink, col, weight = "#1e293b", "#3b82f6", "800"
            tag = ('<span style="' + MONO + 'font-size:9px;font-weight:700;'
                   'color:#1d4ed8;background:rgba(59,130,246,0.14);'
                   'border-radius:999px;padding:2px 9px;">chosen</span>')
        else:
            ink, col, weight, tag = "#64748b", "#cbd5e1", "400", ""
        rows += (
            '<div style="display:flex;align-items:center;gap:9px;margin-top:6px;">'
            '<span style="%sfont-size:11px;color:%s;font-weight:%s;flex:0 0 46px;">'
            'cell %d</span>'
            '<span style="flex:0 0 190px;display:flex;align-items:center;">'
            '<span style="height:13px;border-radius:7px;background:%s;width:%dpx;">'
            '</span></span>'
            '<span style="%sfont-size:11px;color:%s;font-weight:%s;flex:0 0 52px;">'
            'N %d</span>'
            '<span style="%sfont-size:11px;color:%s;font-weight:%s;flex:0 0 56px;">'
            'V %.3f</span>%s</div>'
        ) % (MONO, ink, weight, chosen_action(state, c.state),
             col, max(4, int(round(c.N * 190.0 / top))),
             MONO, ink, weight, c.N, MONO, ink, weight, c.V, tag)
    display(HTML(
        PANEL + panel_head("One search, and what it settled on", sub)
        + '<div style="display:flex;gap:18px;flex-wrap:wrap;'
          'align-items:flex-start;margin-top:14px;">'
          '<div style="flex:0 0 auto;">' + board_html(state, 44) + '</div>'
          '<div style="flex:1 1 400px;min-width:340px;">' + rows
        + '</div></div></div>'))


def show_agreement(rows, total, sub):
    """One bar per budget, against a perfect player."""
    body = ""
    for M, hits in rows:
        pct = 100.0 * hits / total
        body += (
            '<div style="display:flex;align-items:center;gap:10px;margin-top:8px;">'
            '<span style="%sfont-size:11px;color:#475569;flex:0 0 58px;">M = %d'
            '</span>'
            '<span style="position:relative;flex:0 0 %dpx;height:15px;'
            'border-radius:8px;background:rgba(148,163,184,0.15);">'
            '<span style="position:absolute;left:0;top:0;height:15px;'
            'border-radius:8px;background:#3b82f6;opacity:0.55;width:%.1fpx;">'
            '</span></span>'
            '<span style="%sfont-size:12px;color:#1e293b;font-weight:800;'
            'flex:0 0 50px;">%.1f%%</span>'
            '<span style="%sfont-size:10.5px;color:#94a3b8;">%d of %d</span></div>'
        ) % (MONO, M, int(TRACK), TRACK * pct / 100.0, MONO, pct, MONO, hits, total)
    display(HTML(PANEL + panel_head("How often does it agree with perfect play?", sub)
                 + body + '</div>'))


def show_curve(series, total, sub, floor=80.0, title="More compute, better play"):
    """One or more agreement lines against budget.

    `series` is a list of (label, colour, rows), each rows a list of (M, hits).
    `floor` is where the vertical axis starts, which is stated on the chart.
    """
    x0, x1, ytop, ybot = 58, 528, 26, 188
    out, budgets = [], [M for M, _ in series[0][2]]
    span = float(len(budgets) - 1)

    def place(i, hits):
        pct = 100.0 * hits / total
        return (x0 + i * (x1 - x0) / span,
                ybot - (max(pct, floor) - floor) / (100.0 - floor) * (ybot - ytop),
                pct)

    grid = floor
    while grid <= 100.0001:
        y = ybot - (grid - floor) / (100.0 - floor) * (ybot - ytop)
        out.append('<line x1="%d" y1="%.1f" x2="%d" y2="%.1f" stroke="#e2e8f0" '
                   'stroke-width="1"/>' % (x0, y, x1, y))
        out.append('<text x="%d" y="%.1f" text-anchor="end" font-size="9.5" '
                   'fill="#94a3b8" font-family="ui-monospace,Menlo,Consolas,'
                   'monospace">%d%%</text>' % (x0 - 9, y + 3.5, int(grid)))
        grid = grid + (100.0 - floor) / 4.0

    for label, colour, rows in series:
        pts = [place(i, h) for i, (M, h) in enumerate(rows)]
        out.append('<polyline fill="none" stroke="%s" stroke-width="2.6" '
                   'stroke-linejoin="round" points="%s"/>'
                   % (colour, " ".join("%.1f,%.1f" % (p[0], p[1]) for p in pts)))
        for px, py, pct in pts:
            out.append('<circle cx="%.1f" cy="%.1f" r="4.2" fill="#ffffff" '
                       'stroke="%s" stroke-width="2.4"/>' % (px, py, colour))
            out.append('<text x="%.1f" y="%.1f" text-anchor="middle" '
                       'font-size="10.5" font-weight="700" fill="%s" '
                       'font-family="ui-monospace,Menlo,Consolas,monospace">'
                       '%.1f%%</text>' % (px, py - 12, colour, pct))
    for i, M in enumerate(budgets):
        out.append('<text x="%.1f" y="%d" text-anchor="middle" font-size="9.5" '
                   'fill="#94a3b8" font-family="ui-monospace,Menlo,Consolas,'
                   'monospace">%d</text>' % (x0 + i * (x1 - x0) / span, ybot + 20, M))
    out.append('<text x="%.1f" y="%d" text-anchor="middle" font-size="9.5" '
               'fill="#94a3b8" font-family="system-ui,Segoe UI,Roboto,sans-serif">'
               'passes allowed per move</text>' % ((x0 + x1) / 2.0, ybot + 40))

    legend = ""
    if len(series) > 1:
        legend = ('<div style="display:flex;gap:16px;justify-content:center;'
                  'flex-wrap:wrap;font-size:11px;color:#475569;margin-top:4px;">')
        for label, colour, _ in series:
            legend += ('<span><span style="display:inline-block;width:14px;'
                       'height:3px;border-radius:2px;background:%s;'
                       'vertical-align:middle;"></span> %s</span>') % (colour, label)
        legend += '</div>'

    display(HTML(
        PANEL + panel_head(title, sub)
        + '<div style="display:flex;justify-content:center;margin-top:12px;">'
          '<svg viewBox="0 0 560 240" style="width:560px;max-width:100%;'
          'height:auto;">' + "".join(out) + '</svg></div>' + legend
        + '<div style="font-size:10px;color:#94a3b8;margin-top:6px;'
          'text-align:center;">The vertical axis starts at %d%%, not at zero.</div>'
          '</div>' % int(floor)))


OPPONENT_ART = {"a random player": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAKAAAAC+CAMAAACFzfT0AAAA/1BMVEVaIhiiVzTckE81BgPOcEP18OygaVPIcDzbxbaTMi7OsKD39/athmVyUEytlI5wbW11RTM9Hx1vUE2onZxtHh5XMC1yVVKKcW6eZGP7w3mllJLe0c7/AADay8dUMC3MtbOKbmrUgD5kQj7/f3/DPTVtbRuZg3xnOkK0y8uUhH3Uwb7Z1K+wl5P//wDRwb3EvcKdaZWQe4TCuLWnNkJbbYgAfwDMv7tjQT8AAH83DQ0AAP9VAFUA//8/f38AAADml04AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABmjK2AAAAAQHRSTlP8/v7+/v7+/v7+/gv++f4G/QSfIQOnZV0Q/lJhAaprIJH/owL/A1H9DS28GZ4Bbv8c/2D//wKIfQKTAQMBBAD+mJmadgAAIDdJREFUeNrNnYdi4ziSQAvBoEiKEpVltztOns23l4P4/391lRAoiZLt7tlZbq/HlhUeq4BKKMBw+lbX83fH7XZ52qwB1pvT8lu9LXybtzmu9JsW8HJw+GaE3wKwZbr24/HDp09rqCtTA7Sn1fGfBLDd4pfNfg0OL/mC11OLjy7/GQARoj2gVgnLmBovkJ/WT6T63x1we2pJdhCstT5dArnb//4S3J4OhGcb66uqyoQGoCbEp6/W89cBrk4fwJkLPF+hDEnZDtbtafv7AS5P+yS92o8vJkTEHufz7wWIVhkc8yHPGZ+vgAgDCvGXr9Ly1wBuW3ChsaTckYIFUERIhE+nze8CuDw9OWC+6hIQdQ5C+JOD41eMw68ag+A8j7/qUoBJhEzY/qMA2+XqtOLrtFpuNzhDLNPdAkQtw/7tEwVepdPxtUYLI/blCh9bawEMKMJ/lATR5+74Wu/3TyAC9FcvNIWFCA9vdnqvAaRQjyMBdmR0mVApzSVglQGDW7/Z1MArjDLS1SEChGAYE/V4VYBjwLfr+KWAW+SDwE5WLsSi+CBI6FJf8mXAvweKD5e/sQR/BvAe0pVkh5CGf6rP+DIgDcLNWwfhCwE3pycMlXlaQnnVtcwSM5bjJeDhN5Ygud1zOo1M6++//74Sq4I/VAnx+wIwwIe3WsKXj8EdfnwtUis5CbGuGdHX/JMR212XgDiNf+NJwpELxE8sEXn8obgqccCueNY/EpAId6MPz9YwPVx9zyMvPasA3P0fe0iyNvj/57b99oZ6eWr3YqcjoUt8oucsXv25BIxDZZm+ab8B4HJ5XG63W44O2NXtd5rAFeIbSzHR4ySq/16ngGa9/rJbH1rOn4/H9sVZ6Q3A56vzrn0KnZu4SsZr94E/fdioDWXY5dcAcj6+PPzx8AnvfbcLoevLiXGNL2bt6Qn0bd+bgNd8bq38Kk5+tI0vCGTh5pAbD7NsWM6v3uhVPmgM+kL7SJeVy4CpXW+NYUcOsNvcFyJM8i3lg+nuYW7j1aTv7ECu2Exf3pYXe0WExveiSEMR1y3P6tcDbsnshRlddhbCLAmBwMprAjIEfmK8k4FfhYC1AVsRLT4l1sFu+5gpwLYjPr5z28+VSz+UwYbEyo/RRybReRPocdTxEDDoNtYE/DWYwUDdW4ov8MFgXAiS2r8ekDNy5BMtGQTET6JPtb3hDw5moFE2DBTg0Oi3A/3Xch2EJ8PAX61x9EtnDA5cQ78JdGee5I4iDBStYVr6w6sliBnvfGaNqIjVRfpAUTj6ZIfDncaTI0z8DEvpHUEOCOFwcA0ekYB+DGBAxh5L3MhwphsnQGN6EuL+xmy+CthSofTRlsN8YDlY+TiishAQMzjkJyHRBCZAfALC8f3wMz2QSFmCDm/Ug+jES6yLrwrzQITLV0pwiRq21QjQsdpQIvQFFBDFxICGyQbmGhjQREB6nEj4zXwQQNKNAOJ86hxMF7WnAXl+lBJEGDYNiEaaRkCSKn9FDv6WVcxDwHmaA/RvKMegLQCNAOLY7jDY2b4G8Pl05CF/oWId9TT0VImGh6azPRAc/R90/jDAEEiOQYwKU10D5JzldWNwI4BMWMnwxhmCFgQ/1+NXH3Baky0Z0IgMfCeD2qD8H/1XGk0F9AmQfBBOFPg4VWC6DcholdpAtXz5s29cCevMmwQx4TaNQTHrJME3A6I94G+8eChyUvTFsiU+u3q6WJms4YkrKO4YcPkKwD8pIL6Tcdc/xKklvnrVSBig1rzv/C5ofApg8ouvHYPPGTDd5cXHDHbqV6ZCQDP5WwLkfPrrAQ2793EUIFmlLQBT+cikEWgEsByUVRUXKsjbmSRBIEAqjTy/VsWPCTB+hvc89iU6EcBRZFPFyCEC5imCA9coH47pQScJx48MuJsMaaYnySObfQIcbK1CYBsixApYTtxq0DBxBJiniyBaejwBkq3hrHT1al9sI6BnP0sfW5GAavbHVyRoq2hZFJADlyHkkh0TMqBPgPS8zn16PaATOw0U3oFEAoYI6KbJirwA0A/Ml+Z9j+E+jex+iAmAAoL78DpXh7ezc2L9CJBcMPrmoSI/4kgQgwJW4+D6EnAIjg0f2UxUpiOt0/yyWYIY0IArwpl2u9ou7wesS9EqA7I1RMOH0Y18M9wGHBIgxoV4P9HHife2EhpadN8CSBLMJeLlWT18Kpo5renNGLCyAtZ7ywHwcAdwKAApwlaF0psRbQQkIx4B81oPJpOH9WFzP6KmUQgRkJRGopNoawrQD+eAeF8OYsrJb+L6CCihTAT8ePpOk7Unzpyp66G9mXauTv8FPbAx1WhG52LUIwP664A2AqIAw5ABvUZoQwwUIKo4ye+JhrikAXclCI8c3cXAXwhTlPISwEG0kAAtvwFPXSepfQ4H28gXHvnD3Lrd/u8NCbaSlSSHxeOJLdkUYJExKyAleTBkFVsBDEBRlw29MJKJkNRzy/J7lDzbaZQ9AXg8bfCppUfl8U4aGybGoIjqUbLmEnBIgF4BB83wRMWgKQnzYSqkXt+J7YGJNTm21FUGpAHONZqRBEdVBhNS/UEMugBGoeo4HjgLKAw1RpFu125Fv5Cjp1q4pwz1mrOmBCizmL1CBhzzWSDTxqWlIIDVUABWrGEyMwo49DqJTXCuFfmFhwU+JYZl7KFv+GKfgyYtE4jfshHHR7vM/wnG9qkWFujVFc1iHbXkB8eGmvLEWgNqd3xWvtmscwmQ4uxbwYKqwVp978EmT0eAqjpnEmBRNwyYTXkZdDGDMcnVJcAY8aOZoRr9rAnzxQLTZAkU2QNOzeIdAvgESIMnSOikhD7obEAYlSHaL5erl/giebqOVLJ8AEOUoC+CdYpXeQ3BhQVdLEOoJYa4Dcg2Szy+0ZyO3sgPmlyKzEKaxpQAi+HAwecJTOq9UtlUm3MVkCymB9cxIcoAUxnS/D0JDhJ2OgBI5TYuMWiSy7cNpYGW+pupqZwkQyEVgzVqEEB08A5iSuKg48V7er/FInTsZ8id4OSeAlwDJ8QMKCUYHy2xBnqyhIj/fOGCU4mQrYw8GEvXPvoUSXeCqzMg8VHghUNiEUKHSg5wy8yMAS3nOVUsXUrxLEoGdeptFmCQzMqUpViubaZ6awQ0Lid1ysf2Gf7WQQcY3cJy2lBTPwLXPBhwIINifa4WDDr4KJ+gcmostSKexvfiwEduZkgpHw2XEpA9tvLhG3QdoFBJfjdc3SkIoJew3o5crYgx1vSpVmOjpHCUPzw0QQkngsUzQNDxl37sugUnottb4dYnN4+A46JLDg3wLgNbreCjKh3i8dWRLRqVZYLX2ok/AzznQ8C/dVF+0znJB7R74p2uXGwIDRp+uhpxbQSoj9DFBWxrrhZoQgl4IT8SoXPLmOZNAjoBvFbEYpqQadQ/44BNfA8LN/bU43CMAHuX+arsf9k1Z767Ehwm1kfQHBYwavLCAgpqO6q/SVUhrgBZ8osM5C74gObv6vQCwM+cel1TMc5oWGRxgbrnDnAQznQUAhfo6msvJxUPBMN8RVkJarV/qztJEzWhCCDUV8tX6F2aOB+6h86oj6Grk4cDS9D6yxfXNEkoFFP5FXwiv+Ppbl4sgJUk7oLkR0tvOCEU8F+7nxYJkB1+IcGyyvrZJkMtgOSx51f4lqMyyDSgiYBlNTcP+WhR/rXr/tJlFTs4mzmXk8RnwFAomPn6c75JwEMJKCXn0VSxaTosFg/AsRfN4hAfbSRMtGYEKNUQjmYohAuP9o78pgHXEl+pBO0Qi0C9Hy5syiLGMxF6gd5KzIzJsY4dJDYaJGDF2Gr+mCdwnB/L8yoSXM/p9pxhFhLk4SUdb1HHMhv+8pcmmjzLjzWdk0iPSjN9kZXwMlBFKhY+ll8VJw7zXemSg6uFmY+UTDxqmZbFx4GL6SEndsF1jVhBsFqH49VM/PFxNnuUBQtTxIscXTMn3zCPvyRAkR8tH79Agst2DSpBCRYoHWPrNQyysiRJCJqUrvuJVsVSMjrERYYhV1fljnLhznLsHCj8rhRR5UeR4fmqHVyfIYTxmAD5Q1SNQfIgsXHimHvMtqMevc0TNqYIzuRkJobhwXL5XUVY8/zFNKR35ysmcKXE3wKvBT/GrM5wNpyiUhcTdi7Bsv0bBVZe4sH4CiHUwh0nWSSrxxC4x2vER3KEL/fMDJcGS8BBDFY2MA7GGTvViXyu03jDnXrplkhiqmm6uyATDnIRIfEJ4XjlEy75/sqvzIBec+IpQPFxRbmaqiZFLMNaRql5FbaJSVTsrCE+6aWSUtfhpqvDPIrjoAxodPDEz+szYHIOofBr4M4DLYhlJ8tdBVJLx2seJPOK8ovFuJ8LQrgwgQdaFDaQACsFHC4BaQ7Z8/q5O+eTYQgxnsDvZpaXUGIt3cXeNCkXkpInoxmeIUGWqScAY1WSZjQXeMuYAJ/bn/PxBBbzySXXmbWyxoNz5DN6PiIEwwqWhTHqXD9OAP6PtJdzGWVKgjwdfVpaiPbHsxnv7TU+aaiie+vtY+yqSMFNcEJocp6crSFc+BCQjE0MNQFKXdTnzxO7ZqP7ygO+D1rAlKW7WG6n1wdpHehnsfEotU8Z9SEmRZ6ho20yE4BrykVkklhfmpn4cTbWuVJEH1Iv1GMhPh/jNKkpihzRkVe+yk1oVKBnkRFjBiyHIVz0+kqMW0vSVBGgl8Wi6EhkTp/Fo9ziViRGXAKOVjC5OgfUwS5TF0jglssO+KYNODAlYVQyXK6AiXcUQAkWUhHN84KOONRxwiYhSS4KR8BUUBRUHY5xpLK3FhHOFLDinzvYXcmLf4gC9GNAsa2p5TeWgEeXuOXsThTQ5tgCUR+1YhKszV05QSJVioqU0OtMXl4CfqElxKo6B9RSluw+zBXp0VonBSZVETMIYCpqEx/LmaRIH5Fncc0lwvCglVVDa/eyCN+OAeMIrOwFIA8VdkplyRzOluryFAmyDIH31TupG7H8Yv0qmNhTj59QawEJMwepNVCgGNI2HhhPYUN7qKyOQVPUZrQcWbivi2WI4XytzrKr4YzG8AKISIgANV82PiUljkeoVu+8erznEnDJlfOBWiQioCxos2T8hX+VlaaLChY39VCHoYxBP+gihk19IHEFnrRZJUAIOreFMMSwBoopvCfvIQLMgCCLidV5Z6hIMJclR6QswSrRB9Fvr10eqEqf+lqjfa49WVRxOVQBlRW87zJg+8xLD5WYeXzenAGn+4d4Bky2xvDkzCWXPlQQu1AM7ZyJW2mL/hqfowfUjYzCokZNgX7/KJ1f3I4ogNeqW2xsrZZ2L7ujdE3FlG1KAeLiB6/QBZ8qSvkmfa4vcSuILIJCGQdq+ZmrnKTiijte9EVn3WIXnWPnFbr0HRq/ZjZkPm4DUcWOGpRSiZMGrU7ktE5ypKXuR7YsMtEUcFLDRo0zwDVNWzaMcYJiiGA5LGL19qlfnVYDypJUqv+w5sUWQhLgF8qkK89GEA4YNNBc9GefLX36xmgDaD2Jj148a3g+kzoe83WUNaVw6www6tjb6JHhzEhrrNGu3Xwg3yDGgIy77uGsxAtI7xanZdXlNdIdUFaKY6XnDpWODV5eTxn1rseZoxvjyNLAmY0hQEBP3fHKQKVrdf6iAuzjgvbIyMTObwUkVvQhFENriszpEVBIebXsWCVI8rc8TSA1TvMClQpw898EWAkgD4ghjY9BkBUwzQQjqXDZ/ebZifSukaKv6Sk9crKO6aS1EM4Aqe8+IoqOQTX8C3eTVRxruY5XmjisAQ6AQMuZcSW/GgHm6aKrcQoor0Anp/EYLc84mctxP8/ZEOexpIgVx4l7SCvsgTu7WYCHCKiKgj6ZLmYx0ZNIx4rNi7DS5DGYXtpSdCVXGu15GSoaa/F4aedHIcNiVwCKcA1lM5lRG9PyaifFNQIoMVJunxlic1k0q/wZINllBtRl60AxQJA9lmYMCGnFb2RrfFHWZMC/shHUTuQaqNtQAL0OdalcxNYFDghg8LHGGYNSXmHkIlgCRDfcM5j0jFwCZsMURj2e5EkyYNSwRpS0JX0M6LWpLPJx/6AADtRIoSutQ+Dl7RIQXFowNpHwAnC9ph8ESXUrig4KiEENh7tWjeAFYCXVMxN1mAGpNJzan2J3VgIczEiHIOblAvDDafMFf+xCwViqeCkdoSZIoLVut2J1WLcKKPWVzIeAFSNIq0m6pJKogJQP6ppyikoLQJMBZXe1IGZGstQgiw5kpaOGqe5wDkgNHDJLE4llQMqKzjZNYOKks1hTKTvkuGsEaBLgEqP7pzWvNMZuctmZJXaw3fGKj2qY2urPAbm30TlnzgAZofQyRgGlsceoeDMg20HpVoaUwNPW2WWxR10Z8btP5Ook1ucoOIiRSZ45A9Lq8BUJMuAwBWh1SPgSkFoCEqAQSnpEiO3hSzm19y2FW1RxS7E+9yM9k2PhEwoyoJZg8hjUFkb0WMPnsYptBhTxxlCA9/IwID5YnwNqX6icRNBh6v5pI1kdGhmYc5wgkdbp+MwtrCWgZzsgZQJtxBdnVo1VHCeJGUmQAkTJEQHonmhEm+g/VMWxwza1B7ayOxJioEDJCANuKF9eOQHkcEv4nHSzgQZs4MXbQnyE1wulfDrw5h0BNLqrghpByA2hj4V4Yg6AxLCj7dvtcrvlH7erJUtwKTusKt7tgW8h3YPIXGXAKDoPUWAeSLpVpVvUrGR+RudRBGReGSJe2oC5+OgraetiwNBf21/etn+K1a2tDEGvgOvTDwj4LyNArVdwPyHIKqNK8PNnr8GD1lcZKALKL+WNEJA3ZrGK1TlRjmXubIAH9nM8BMkByoDNgBzy8816iqzVo5C4RYIsNZBYh5Na/nCaByJfI/tbqkp2CJEA6Zccdat7ubO/HCXYslujuN7ogQhZxR4kAaGUx/g4mzOgeEHNgpzyJcAY6UQbTTOPAfmFGbC7vXVS50glc0RWomiSVCYBYpKTkg8ZSEEARVnxTDCaEfSUzxGQbsSlLb+OZkcVAb02ON4D5LI0DUEjVkZq/WRmEmBlap+zI64nIVSUIBdzYilIHsiAVa5NOMfZFwH6DBjuAqJCaVDRWlYlEnzWfXWUkgigrqnFnA05FNDoeT1WKn7xCZ9rkFU+HhSVSakorR5a+qVPgLwB9Rbg6nSgUiDScUbMW2N4wwtn3uyQ4pkyNtX0eCoWgOcXCKD+9rMUYYyogWpOrwHcnj7xHOFoUBpvddua1EzP9nvlSM3TKm/RrpPOViOfWUcVp8vjYxFwEOP/UkCexFZzj0/bbWyylVFnfVFdKCsJVXWxWy2txVzsCqMEsdaYgXt5RoDu5iyW3UFWUg/3Cc3OCJDePFw/ksdfdOwUvUm+nt73Z84l2N7avvsj9LylW4sKB3IzGVDuXsuu7K3jEVFagOQxaUdn5yl+PthFyyFed+Z7U8zi+4AxlLFKuG/FPfu4amF4kss8ZGMm/s6IY6Xd68VTNfjxfFpYwcmZOP/P65NeDMhmOp1kZOhogS2lnaEYP5V0+1X01dWG22MdbZD0aNwc7cnmjb1GAhSIfcmsaemJkRJF7nNjWRSA02NQhmDSTFwn20l5h1VEBpYXkYMU3diC1LSYRyEo+VQXBJDk7vUhqdEZSY0Ml7RGfLIVggDTSSXXDTXFHz4zEsWSD/3CV9cFoOc0jIv1LBtUdrRChreyU3s2zmTP+/IJ0LMton3kJtBSJ/iSjyVYB6oOr29FM08SnOQrUGnh9B+OV6gEMBCO8XHfeqC9BJ7iJXZynDdxXs7C4vVpbqzQV/AJlE4DjlHOy/XgO8eGwZ6XlEpADlqPknR5dXPcJO7kdAnZr85r3mw6AwVgScWcDTnedO+9TizNPfxVQLh9KBesHXV0F8c9eSPVdSchqcxtNThIT1sp4ywQr0Y6ZT06ox0oMjKpgZkWFEBzYXcmQYq/OLc83jqRBDo207FmqP6k5XAh1xeTdoR3dHoVYZN9rEq/ITGk51mjob0/q7XTiKADr+Dp5okpUE5ifXdZCuXzg/IZLeb2JeNAgPWf3IqJ0VB19grdpjdqWL0DWElpgkW4lQNMXT7CZhQqvP4qpghofzzNnv29U0aBw3DyBjzWnJMQ/8tpKSfiyLk1bcuFk+JooMt97VP0IXdYxo0SQeurfFLKnRNxKGnkqpGcj7aOBQkduG272bTx7MHDmg4fnDpQ6Pz4o8tt+6oCdTZ4479gbnHv7Cg+coUSfS+zY89vUvNSaLvKK7XHbXGQSknh4KFpFvGimk/XdVduojwtSaqWm+KQq5uAnlepPNuXVSsijGlCu8QrZfwrqpNsSl07cEXD90PZKj9mWh8O+8MnqrngtVsfNi8914pXfejgPkogyKYf5E2njtXk7keTzy0rOubHl7a8xevS3a5edpweehI1UpZizP1/0uSliGY5tWt2DcUiF2ir7TURjs6/go/H42qr54utVqsXH/ZH+7HQYPpY0aFTnH788cbJs21hEsn/Lh6mRAh5OgDp5m3n+wGPKfZqDHi4bZY2smrrE+CUhmnPizQEajawfisg9XRz4FGLCPen7fGHSQXQomOwmgHUNJkmARcOisQJps+iuKdiOQiePL7U4T/ckiGvivKqbSWAYRLwIQHGmsobVcwFaTAhOd5bh+rzwnyMwepwYwjyvia6B69r5G+VIJvhNbmquNpye7S0sU1Numum+R7mjjaMSWp4O2q+dzAYCmwvTiGOllvq2JHHqimMvwO4YECfNoS/HZB4Wu4+5/TrzuHMaxysOCKqcE+CDw5iEAcAXwUoTufjYT1aFJjcCkM77jDguyfBBy3LSivK7usANTCgZZR9u7x5EuYTrb5L8nIfkKsSvLj/tYB4ffcS93M8/ugUkJv4mpuAQZbQKz526bn9SkDV9M0BiL88csYugP0tMyOAKdk+ve1vCLz2TPT28AnMzMbRfxPwJxd7OO6UN74Z4IpOYuUFYD0b6DZgR4Bc/pIQf/f0Bhm+BpBPX5jP57JhmbuXbvjiBBgr/XwU4vI3BNyefqSTBuaE6GoBRIFOInLHZ+Ayf5iHHiNpaoHf/GaAS8rlZzNZa+6dNrnD5DzhOo3hs8VmoadUhQ4avKgELo98iP5qtf3u6wBXpyX1Ac75ClSt8rwoHiYCmhmV5DwfimPnhl7T0e7/bnwK3fLCiLwNkMz4gdrsIl+P6uNT27spwIb2eKEAh+Dm+iomhLNMvd3v5Rz9w8fiSNY25WrwQjwKeOxsngSoKVOYjrhwIgXpluls0JcF3ZybTl5qd8Cb3zo92FaS8SjJY/sCwFZaCfCNm5KvofcNfNxAmALsA0/4eRShyJBPCxLEbQtdTqk7tZUrkevmJRKUuKZ9oo00NvQmCB8X2AIfuYNTs5sItvB5jNh0YNPQ6GkHLZ0BzIgthFlJ+DMruT3spPKwvyfBLT+b/khPj+ILcigyfiTMtXWkQ+vhYAJQunTmPd5NL4R8S9RKBZ8kb/8ZVMOqZGqcbj/gmzcPs2ZB5fL2jgQpvHGkXRtEegzYQ68q66mCexUQJ08nr6Hn8Ov6Xhtieq18LNuidkOnSbf/tmq/wHxG4XjD27z304DPGmfTQdR9n0b5fE4nK8QhBTSoptxIp4QIxID6qlhLIsR2XSCuN0vqeWoeFgveKL8ITVhDMa+32x+W+aTybbtDEn7mbNGRM0B1oQxYxQmQ5mVzB3CuL6CfenYrOp/pT021x8OHdbf+sD/IJ2+gWcxwYM6aEBYNfoLM9yKY1IoT7Y2A2QOf+Dzj4x7kfQOPpJ4+riPZAtmZZnzRK8j/diLyKH8gUTYzuuHFoqEzLp7K8n67/LCkJLhxc3zKbI5Pabod6GRoN0+H/X7/RFWnLZ+sjsnowyzMG0WcJULBkqFElhDtjD6puB64+i+ILEhyOtDpHTf0Vo3+0Y3tkf6c12ZH88ftDujuG/q8gLOk6VjFz6fNLo8EtAHHDZ+X0uBMms/Th6PYFLCjuckDG7UQSILnfA0C8jSEODH4YPCH9Hv6uQH3x3fvuMVovyOLCIt5R8cMzFF1Df4arS0BLtEMkyowgpIVIDSk7bs/OofjIKqMvi4g8vWxj26OD3fXAfFh/OU8M84LPhQha1ls6Pt3OxyY+C3KrQ9oQBsB7PCbALw4PLfz2WPcqGXg3Xs2EwixKEdWxzNaTtZasDTJuOKHXCgYARc01+ll9BQgmffxd/hWBIi/iidbAMKix+znD2SwcdbR6/ChBnYC2OdTPWjBc8Z2DMdpH+9/LjEWnxNA1HN+jM9VCLOrFx3xFRllQIRyHolr63j8PpDEZiTRBaseAXtHgA3bQa0elQfXox0nwGbOSkrHfXT0cSQCKkV3vEjRLZqH64ANGSeOJ0RibP76TkeJLL03OsEa9NcUAKFeF41D40ryCyT7HXmSLVrL1AzZ8+aAGQEuHsg40Lvnj13gT81CRNeJyWhmE4QzPrsHnzgXy9PYuf7xAVIKW5EQJUgyDIvZnA9cofM8pQHziX3xM9vzuKGGThNhw0fn8bAymsVIN6rYBZlntgiTV0OMJMZeEcuLwgO2ADNRcTPrAe8n0GQ8HMmq0B+7S8HCRw0gYLf+5R3HmzKG0I5kvMVc+8mZbtbcpIuMfDIEvhFKrHx6oCULiuFo7qIEcdQhtJxcxmbkDy1X+66fs0CzmE/EgNGijHwfFrN7ortADNK8RXqNauAzJTrmQycQ0ARihEATAzrYL7dSjCyKR9tV8nQckr0TSNZoniSi2IfZqy5GXAQ4/9sWFFTx9f7fgYKXgDMJx3Uf4HD64fm7WyF/y9EsQb7XDEjG/MMLFXt1MD6QaZHb7Tu90waN9DuKSnv8EYcqHc+xGBchpg7p3Gwk5P65bRPmQ6J90zWOdt6/f/9OOmnRsz6helC7c7at8OP9E3sE8g/tJqdZ7R/e4SVD5uzDpgzh+En8xIf3f36n7lfXNTYfudPpIM6TervHBd57SRPe5Kb9ePYIShWvP7+n69dff73AVKZff+VnMNM7flnxNt/94SO+c5tzC27m7w/tWX3t/wHz276VDHyNIgAAAABJRU5ErkJggg==",
                "a perfect player": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAPkAAAC+CAMAAADAzFl2AAAA/1BMVEUAAADql0ykUi9aIxfNcjk2CQWLOyeeYln325jRekLfgzz07Nz39/Zwbm7OqaRwHx90V1StlpRvT0tzRS75x3WrnpuIb2tZNDBwTEyujGObY2FVLyujkIyueoZlQTw5Y5Cm099vYRn/AADHt7JgPUSWgn0vN1NHbZjKxbpjpc+jl2dlQT01T3VhiqeGbGiltMb/f3/MuMj//wDHtq/CfYUA//+0x9VVqv+Rf4J///+gh3//AP84DwuahYIAAH+Rf5EAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADwD3OfAAAAQHRSTlMA/v78/v7+/v7+//4KBf4EZv6Z+v4lWqz2/RdfV/6o//4MAST9WP/+HP4XaP/9ov0C/gFM/wESA1QCqAG5iwIO5uqGqgAAI3dJREFUeNrlnQl/27iOwC2CpCxFvu9cbdppOzOda9+19+73/1ZLXDxk2Umaztbp4/vNS+PYsv4kAIIASI1G36ZNRlsAuN+MutE/VyPwegWwGf2ztQ7galwZmH//Yzwpf30PZlxVFvbl+66xU75D+uyfHZiqag0sv/cxv76Zd6N36feH0T0YawG2s6xD3qH2L7+jMZ91y2DHgzGb6Ni/G3U7qImcXpxk9h6g6KPXLeVzqNcmEM30hdlo6Q5Bzyvuj0kye2j1Pn8/g76DMRIFyEm3DNIcBtetELyqXBjhh9Fmt9vMJpNbVH4Lu++HfAlkzbow1jdBmn+kwV0TuXHBxt2GF3fhj+Fla9Hqzb4jr8UYcNsO2VDsdyzr2FZu+7/04iaMPKk5vC+nwNds4cJIOziE/2634EMfgKsVHFUd+8XAbunANGHwvye37l1wXILtqgM7QhqHep+h42vgHFgkH303I87tHsak1Y7kPgcPL9JrYZYzgfz+f74r7tlkC4e1bVGtg6wf0NLzeBtGB2eqAF7X35sjfz3aQD1G8nHFSh56oFpXYbFG/OEXBDdN7bbku38/KzN2UgJ5tV5XNKEFuxYGHkTgq9YieeiKzXem5uitjivLnGv6jyawqOvcKnRdv7c2h2rc2qDWouGEfkgzG5q5prow/63rvoKi3wZFb00wZMHUEblJY44/DZLXX2PVOvkaa/xu0pHBmb1U9/4zKPqKlygrAWffRi18WLd5NHDzl6q5fnwS1ggvvkj3VXQPSNHDSJu1gItR57nNAho490ID95dwt9vlctuJ8/gCOe+67T3A8vbFAtTdu/G6bQKfC+iBvFY5Z0lHT44MXPeyAd8GZzD4S8vb282XX2lDAQV0OR3Q2vJli/TgwLSk0fWa7XvmxpHvXr/QtIcb7JbgVodDjfjwZV4R9x5xV+Nx0Mj7l6FPgi9jguuW0aZGczn5MS8w7dhneMtTNic1xrR/w4DQc5dXo9/Qy0RotEdjcLvuJXrzjn0ZZAzduVqtamrGrFa8NEVhR9919jJBr9lNqsfhZw1w8/zLdAG8GtO8S+M0XrnVCwV+B4dAuaoPoSNDW0eBpxFn/Nsv/YoZCzovhZqGr92E8do+8665+3JVHNfu2QmRSdZm/75Mnou21pq8AVxPrq9n3B7yT3dPuWM40IAbE3vUhxd3/3h6QHOGl6mrQ+ZcCvr1E+bAMzGp8dgGnabRsNY2oVlq6rvWsH20M9+dchHDSI3ZWkazWVMUZPeM+WJCKr5e19p7tJisAvpJPZycmhS5/fh+8+Me1KzBUaOX65X7j9u/bh4eHq5D+/nnzfv373+UC5z9Jozu4h3jTXongmVX0zWOOzzZL8QVpWO5qeq2QD+4UxNF6I/NfLfb71eJxvXx3BNaehsMtD35KJOBeYONEmq2cKNsoXiFJf8TQ5qY+FHBUUGn6LD82A51+my0uYeT90u2nBrGHs62XOtXYvZd0XfY90ccSzFKFmC6rlTXadiausabfhz93azb4WXqNO3m6GTmukFpE/M11rYeF21NTf5drdfVE1sj3p78AoMhyh0Ey4GSLXEuC8nKhZnduSfNlsswNxyM8VUh6IpO8fGhsPKBmArnbJ36Du0X/7T0nxi1tqUX6A4rE02dsakFq6DvppeRfVJwUNij4ijmtAduVjWhB9t8/ZhZX0pgtLEK4PgyjfMcMxuwZNG+GNKw0NrWpq4DS3HlwBBuyfB/5KnjPYY/skVm42fovU1BnveE9UHii2k/kgcdXzP4iu+4kZ5vSFQnjyYFUBrZstUSGzX6swmX/6+BKYXUzDZM3pfXgIY0tBgTcoh6ROQtzesVxacMDXBOXjcoDVlPBPRePn7nkFyUXWNc4WPYDUFi6mnQke78iCezngm6onNGYGDMwa3HNU3UZkBTAWMtpkrkGnJlclsFckNhufBH34QfviAnKTL6Gyoz9IcwWDi8swI8TuzYeVNwj8xt97lZD59tFFl6wBzn+iecJQueSZAUewxOwgwcZWs4+BIWZZ71Hv9oqUfpLawGvTHv/QjkdZaYFaHD1+lurcxrTfRoUIhq8kUm51aSRfajgohu8VIOO7Y/Q/wRBGWFa5GqzZT7S1pr0amT2Tih0/8DdwaSe2dqyDlwPpcVfngjezJhSjjofGAEvcPAxTD4j+QQtFVSVvANj7YnC4KX7/uCk9G1M0zeFHesDXEaMTnmbCs+ps2L5eO/oxlwmJHY5QPQ0a05ylyt1i3aZ5A0tXxl+ERYAj+cIieHgMyCEfSmVPFw4f3xxyb0MYu5oxOeDMUa7Zm/istiW3PyPeHihskxiFGMwIyytZ6UkWb/AA4FOKEPe2EiMyLraGJV1Q05hUCJQDSru27WM4sPQg7g/anhDGqjOny6OdvCqTeFS2N2HYc9CGAYwesUbETD7MBzfg7vPEx94sKpzLC8D/szs24Vs9o0CxWZTrpouHypYUw+U3LTi7Nksp/Iy9dVEfhbbRXFuq81pG6s6wGy3rsP8TYeuvugAI6ylTTwOMGJ7JrY3yjvg64cVnAEE91idYMlc2uraNCRnFLAtlpjzcO74oO3gbxlclx69jx+6cAmE6RTjcmzN+HFGrZxwcIxOcleII8U1ySsOJshO98uDXmbg6uR6waGnHKc6AUZSXlwr7Hd8DT2DcWLS+vO5JbJEbv5EvLFIHmVkePkIuQe6iJwNac7b2T5S1O7WEtPtkVEhgZ9MpQEoYD4mOfEuGgmi8m/NbT+Me6mJyy34YuFnDRLPYh8yfMY+dvF8Zsa/WK8eEGOip6RLx33d0N5eL3pgAsyhucH/R47Lnxr21aNp/nLR3K0q6bRRP+8R/4hJ6cAso4Xm0hyZaqz5NOrxdGbTFyfBmomtyLtpu6R16mviFTJvRXFZ3Q4Dq1ImteGIbU05moTazaYDmS12oS1/4di3ROmlGLMQdGtgNPE+GxyclIV3VcFuS/Ir3FOY/NC5DRUUKtZdL6n6cehtxU6MTasHFG5QXspKJXoikVuchz7Hy3ILVAFBM8qK/m3QJ1W87e/1/qmRnMPQDMB/kR3H5fag+TkyAQxaJqKFdOhOaYoh4+CHwe9v2bjosS2ygRNOom8k/R5Wh0Pkwevx7RkCFKS1KnoJ/LpEHm9ODT5mxoEl3gSTrGBOeh4Np/n5DSfqzYTNt50Q6PuXO5i5FNC1m3JZ21ZwD1+UNTdxQKm66NOY/LgoyG51L0A17pptjRCtYvf20H0pugezTCrxgAZtxPkiL4TdjbkNPA1Bp0LcBzJ3dGUTuRslDBSQp9ietUQmi/nR4H7GU0qObmiZ4OfoN6+PTOf65vwk2l+IIk/R463NGelJGAZJPZtila71XFYBTBGa1PSI/iLUcZrJA8XXm6Ow39M3lryedpY6EWzg4MjqMXiCeQ4gNk6t0JrQQu2RF6KbTA9HbGjmxnYsbEUlP503Q8vTIS8QS8sOoy1Kjs6fjC/xrTsbCCcMQ8rPMuOZ2RwPJWWUBwFfpy8yT9ZUda9IIcjcg5FXz8gPlAMV8kN+GzZV/fXmg+ja17V+XwKwIQkNa5OR8p3o0HysDJsC/KqB57Iz3qvQh7MiynJ8YVIjn7acfJReDbkxtUyXCy7ROWFvDTuOKmZMU8mMSaO41bXe1Xv2bvh7NaMQs4Vk9ussBGg+mJyqHpjDhm5M0PkOO6zsI7s5tEPyPWcht4PLFWXQOs08oEAQ7WkNOJILLuzabPPOLw9cvri1ReSw2Pkpj6ZcMbygfmuyPN4tPUkzJ5c93LcVpSesV6NIgcZ5je/bbebxypObsK0cDCFjyZ6br4GeYMW7snk3N5vP9983qaUloy9709rEy21D8ZZHX2AOnlr5/OkNxTnthk5eTAF+vOkHQrT3lpya9BJRC13j5G/S0VJzILjbVshL1zQn0cfKHrTgOXQN961r909XvzxijAcc1MEcth1y4scn0sOWVyypQuRe0wT7hPGvLsezR42SxdDeA2t1nvBnJiSa42vyGunfEjoniduq7iBigu+hNySirfHnsyTyC1ZiSQtLdbQOdMKedsMeDInQ8mm8WjbLIVzhXze912RHN1LZEdZPRW8OUGejTmPdRvH/snkRt9kCkVx4TqtFXKdz59CPgkDgPJrJYxtPEfjjmKP1gdhZ3JP5DdPKzHpkYOLI71Ka5esiqNovaSq+u1G0VHnHbutzyT/GYMta4OL2yqG9OrejD4Ps/mYVqhEjqrhT0dpe9akuwG6fyEnzzWf3JT8iJKKrlJBHHvssraHVFmBI94ekz+aE9+M/hoGtGka/pgEM+u8TG5G8ZhxFewfJkGfTI5mRIJBQt70dBRzz2AiuVn1h53BDzr3cLpZY746J7MOhlsryWez88VuG644lbLaNOhpXuM5DcdcyT1o50weqwjCGpbfOLjN5FD6bhKMQhVeKd2RetNKg//YRp2IY255/UjkbSK/fWzCjeSmJEdxn8UF9iqQt5gZI00H/yg5dna3XbKzxEt7WWBCySZWL1ho9ZD6457Xz0BuDazJfsGcXSLHb90tl9v3Z9hZ2omctUXFPbPuLOxs3rB5UYjTqhS+7XoJoBU9GXkMNrfDfGkhw2+s0x88LUb7HdO28qOtdMUSS4kC/va8hRtzPUaGnpyZbvQjOnDrjFxDcHCKfIbVwC7491QWU0fy5vSE5aJI87BjZI3qutW9xOVxYySZWpC32pNxzNkt414/s7NvqSn1mJ4trDuutcIKNSPnihUkz0xcd431iWpRPlCNpLREfnrC1ug9xDUBkicpN5G5P+Z5isq0EnVOkQYXK58kIKz3iOFk2febMtOFdZ+NaJfoOCfHdG+NcinZw64oS6F16aqiIiceT7q4PZUFXUmPhKt6CRKhRhNDpuKmKS3bQIvx9jyKQIVifwyUJc7dKm5/bTkRbzJnRtyYJpFzbQkL0zxujO822/lWiuW51HKsOmg0cZmnB6uh6glcAbOWksBb/ne9mF5NF5hEHpKZbMxl3IjcStCsbmquTww3uNzt7qlWcIaBFywPjC4ERssiOYs7hVQCNpFXzG5J1UE0HWNcuLHBsUXpsCr2wGVEV9OC/BHf1MHiKrTF3pFRFC91fyVt4R5zb9uMnOy+VfRtt0yKRNHC2eizg3FynqgajcnvxC/v7jmTKJYEax+Y3BtxZro5LfGxJpE2SmiN5OEtZUYiua5vsdaxVwxBRQg5owoHxBfxZRR4c6o2VtKETC5zHpssyeyv6gNvZ6BQCq9FMnIWd0/iTpVSHRYOWiFvpd6l1akvvAULO6XQbzw+oDGWMvJ6sagz8tMK6jEmq4zTBTJqkg5ICKCmP+1xSWa8P1k2ESOQsii0VdT1eqoGd1qT1eOan0Te5OTAOyB496DF8p6Wc4pKjqmkScZNzciOiRhHjeT2uOxFNTS8icHxthZXe13ILGisw2vUKa4sKug3MGrhQse9zXYF6DyjLjFOS5B298deUnKgcnItjrFVTt7opME7BbKKqTFeuCkVWL3XM3oaxRo15mqh5DLWou6YGTyaz/PKOiUPv7CmUaQCaKvrWu+ykQDoeIgcY7A1V8ftJHsc1vws7Ow8iLY56s91b4G1L9DNE+bzSqwbkiNmIt8vpiTK1AWB3JwjbyP54e3vQs6BtLaK44PFjxinHlenyGmrEK1WqBjcVsfkdIHxkeVe12SGrZV1abZiOU+O4RU0lxm5RnhJFSL5YB/SokvI6wWl4qyRKEhZH20arXjPyK03So4hxg1vKaIZDyXeiqscydG7CxcJOr3OxsFwoRz2eTbmhQ6YIhQh5Iop5HukhXzMra3SFr5s3YP+fD7mfKeYMLe5IsbvBldWkpuc/A7gZz7ZhchNlZdvmkaLRcIfcV9HT+SzfTJwTF4uMQmlZvvGYd0rCcUbkgNs3DFeyM2qWM5K3KIgVy0f9H7oLIsqT1rzPCjkUyDjTyVdGHgsyatYJoPbCmN3au1TkwfXVdp73CuTv3algx6EfWrYP7fTYNHR5vG0BlakPQvEtakMI5G3abNfVWVTSqOibfSonrfaF/o+dIP+hoqOFQMNeYaGnCQpU6qMThhUej/OdpZJN4t/EnqiT76S+iKuXoi7gKY8q+GIX9Wtrkx0ir9a1Auc6xoTd6QbdXq1/iSR23gz+RKnSdOpUQP3dqFq3iTyYOL2mDcfc/FRSW4r2SbgQ5cM7b0wUj9i++RZvQDV0cVBXywW6q0ZrSBvK+kPRyJilByDWs6qt681qIlcagqBlrqlZbVSCzjmMpRFXAFa3lCg9ZAFOSn820WbbBSl0Ic3mHAcu+qTFxHIKlZKqQqwuht3iL2Jhm11GMvST8mN9l8Cz8hRW1Fnoany4nCRguqIHLmaSO7RuGfkVt5xtahK8jTKTS9AHF44knYoUkNa/cDT7Zh21ASjU/hX4f3raIjjfC7oGXg+5kHWAj2KvYlqDvQ/zCw0RqW9Zj/T0OZASqiCJJmY3DB5MHBNe3j7lrapNVXU82rIuWi4oFzG3KZyhzLQBtCPOeL13OkVXZ6VdL3kXBpza6UoJgvckQdDCfPGY7V39pUN7Ypkckvk+5yc6m8tklvU30YqbnruSXgDWotGy4Z65P3oapkdTMFo2amRqjyPyAm9TMhm5EaqeyktwmYt+iw4sPltsKwLudfgBDr2TC6rhRrJm0ZCN26I/CCBHa6jsTm56cdWrWQQjtbquCTLyh1VvXMRI7/HDHmvJtZBxb1eMUlORXQGXAHOeTUmF/d1H0xBnZFXuN6NQSuPVVW4dbahCQwlwda/1lp8rbViiRx6q5kT5Fwkmg1vTCYdkVdDKxaj98ZzmOTZbSx2Tdkp+e6mIL9DaV8GU9DIBjqRd5O2CNOohpeaRnYO4//zai9afiZvlRyqIWk/SjA01ubgqq45OS24nDlFjtnxuBrWW4zCiuJOXlfDc29OHnz+sFij5XlObuPeaK9fUq6SGR4713ipLM7I3RPJs7bKykoyN1w2dhR6npGjm+TBHN+fkFO8shFwJvda07pA75Wr4vHTvncFnDWcGbo0y5Qcbibk9gQ5PEKOeYNDNUBOs5otklXgEzmBG938MbR1CFccqKa6zd3Ed+OKZUPhqrUaiiNyOI6viCZxqb/H8sqcvG/hNK8GRVeYuLQOU8Ehi5Vl5GjZbXJoeuTeZdWe+R3GX2Xbi0qwzd58x4UwGJMRX7DcCpuDF0ciAB9pKCsaV5L3FF0dd9AsUXAuVqkIaXVYZ2tAk5GrnOcuYSIncDY14E/sADPQ29qrnl49xUKY0ehaxL2HLjVkuXrIFiJVMjGwDnLyXhIZVjF/3saXo8O2zs/Z4LlcyRMwGnjbI8eu8+lW/Imtb5ABpSH3KOxzrXL2Sp7PkEY0H8XbF6OeldgBB88jeT5GWMvjUpV37gzydvShHTy27/xnJ2uJhctreT0dowHl/q7+ZkGTdvvlBVJhXlsbk6OnnmvEkmv42MvG//Q1fXLWzOhpOHO2WsSWcRubnP+k3doLrZJLhsoXN3EUlj8ilw94+BuQsPMhDWZQWdIcqYlLJ3lMX5wdATl5lW8zTAPXI499OUhelh9ImKJHnmDz+wFvhnaIZpW/vtZDUjCjaCoY3ibaZOyxdlYmtTzfjxUN2RYULXXu1UD2w3SZs12Qo9vd9IJRA+Q6EP4oxp8bdL1hfRdEYcc6Z3eAZtBIkHc02IU5fI+cymehF4A8qgrrbVCkN7X8ptb0nH+61gB5dNR798rbgAvylHdORRMPYdBXUJlT6LwV2pQdEDraZ2MuptQ+XuT3pErAoR387QB5lMFctcX9NAP7mo0mx7NyOKh6Bi75qhp8S4bfG1/I3FclP7HBCTO8fXJIpevxjkvouJDLaw3gQ1YVhkbO2OFNwNDbH05fV5i4/w/yqrVFnQzm3WnPhm7ZsIrKvVH62nTLXrZo7uNuajo1CCMapXz47GCj3lVoLew9lf/TrKaOQ9VIWLpfBCCb8NrUjpYZ1shBmPmfGlZenZBLctqwwr5GJvW0eyE3e0Qtu5rqfuHTKqwajLo9w7v+5SIyhSB+WOwB13uwuTq3q96cqSbJ31SdfYPmz7nsAjdt+CP2eKPShICmeSbPt191uLOTXdUzW9+1CTeCA2ZreT4/2k7+1Vp/och1Ar8xeq0iSCMSUX2UcC/yy4rQK3CnWsBKfdXkHBUzpc+6EZPQdzjgu270GQpyPkbnhWdtPEaODuuHf2BhWrgNI+wg5J7FPb9lVVoivz86bIti2KpTsa98xh0vVocvrPX0qjkI+YsE+cnSzuSU+8etQ/ta96d5n2u3iEF8WW+8V82LflyY14RNN6nV0cAV4x7+dreoeaMmXWTH5ObsARpovNRADDb8q74JitejTXUak8Fft+Gr398Le12IpDrzsn0r/Q2Lg8qTaHhPKx9SVsdNarQEKu6O/3rHO7rmoi//quSnT9AQs20HIke5GZdzJqw95XrLfC5jPpFjG3V/Wk8la/aii1PYjk4fwvr//F0yTYM/Os/t7k76JNvJtaNdJhIJaF4yn9vB+fy4TkZL0CdUnIfwe761ssVF1j4N5qRfITzRc20hO+GOfuyLi/HGPeKepdPZVqDkz/Fkmv7bB8jJO1ChyMj38EGLFLvrJeR81H7Q29+lQwa5VG4yUL28wR1u+O4lVjt2m/kyOmnZmXx41nFXnCC7wzO+hLwtVqFZqqhPbjKrdoLcykXMIPltepjRqNvuaG3oiqHbzakGtrueL+l0weV8M7jzjs5U6jaT4jhkKqnM7NRuKRWgXXkiH/q+HuKqM1vNnyCnwgLKG+dL2YLcxOoDOJb2fXZG3ITvdJ7udLeMlarZaZzFXrZT53deT/C01vwMz83mfTyC87rrn0VYkfMp5BaiuoBWO5bkNpOhbDWbk1tRNwo1wlly3JBanjaaDpwty5Un5w7vfveXYswnR9vNr4dOYURV9Hr+CZerF0nwHjl1iZFoe0rcZeSWIxvspmsA+ST5wJ3OJt1XOX5/svnj+uc/Th5bnJHjVsoshlacsFGclxITLG5ozHMTQIE4DBCcI/8mjfW8ET0HySMnjRdyX5APpZn5JB0Fb+jQVkxialjxIsnjmBtNoDdRsG2PvHekAOj5e4G8TedMNC0HRvDS4FxBbi6RXMXbxiOH9CSdSN4rq4hpmTaSA58xEIOJeDjKpZLrrGZ0BJXcgpJrqsT0N9Y7KeC00HBgB8uhqjyqaGjDA5KbiyavKD1hI/mKBTkjN/10q+WNMBLSoncU8dREzjH1CySX0wFSXYCSmzYeJzRAbpjcPE5OJXp4GNDkAp681yfXav421/OCvDka81Y8FQ8mHiWSLdh4bU7SrmcCzi6A/J7JTZ+cz75VcpuR2z55Y/vkRYQSbX1m4cylkC9PkvMREVUkb4fI2TQU5NAj783ndPTVw+WRQ0aO89MqkVs+KOSoVtAU5KrW8aBbOrugTz65MHL2NKO0c8kHOi9MLn49lKVx6K41JXlENxLmEu/VCPkPl0bOg24aeVoAF61ihVJGbjNPppXsOEtIJOfqILLshsEpjpWN+QWScxxS61IFoLGJ3PbKKvj835LcSvkB1Ws4j6FHU4z5HZI/XJa0W1MeWNzEpXW0fVLVKbMVynrTJ6ctrLxvGyg9wmWmnD5T8smFkBsNm5oUt4J4HHlObsugOit5IjdxCi/KP6TQRcmPzib8RuRNRp7u2hTbC7Pf+UBex8dEppkgJ8cVEGh4Qk7xVnJv7uAiyH9zdKReHHMbM+6nyPE0eMp1mabNnNSSXCrTTBWPgr448mVGXhxonTH0yKNX3993msglJyxWjdMVNko7XMTjRW9c3fDS3OTuhykPSuiRn9hx6/tvSofBhAs0UtB6OeS8uc3Go/R8PxFrCMo/cn47rdV6+dvsMCHAqmTeynAR5HSSopaPnsmTnvvjuTelyjmbtjJcBPkEK2erXgnlcF3SU1q/lj6Tdj2/HqhM+9s/H/7v8nyIc0UmL2hw9Jtj8u7b6/kMT6zxcFwN/ueQ4zGNTP7Nmzwg4s8h7/Uo8CG++/oyHqBMDxmTEtS8NAVeIPDpMia/In1LrQXq375hsYl7xoPzvqRBVmheDz/m7VtFIbmG409rUvUUftZ7cC9/cvRXau+oQopu73QJkH+S7D/yuMGg4vXeuWV3GeDU6PD4P71RFUgsTrqUNt+BK+jPKO1jL7gzz6PEc6EuCZ3KE24/bOfcbqQtB3huhtoA+w5f/ywX3GK7fdHDvf9U9oEl7NH4Dz/dbg5Hw717zhd929ZtJvRM4+wVcD57Hg4xDT9Uu8vr0TxVJ0MsAZFnJdMzkkevoU1G7yHbJ+KpVHd4QuIyTJoedI/RZazBv6xd05MG44YQekbWyWM1UTGM+GxIfre/iPjql445HkvV6GYYwKegnRrHjjUD4iJvcQkr0Rf5tYdKDi2iTXLLc+6/41AOBSH05L5X225cOj5ejic7aR+XwAE9Pu3GueXrxe5ms6XTjIps9j8zjkCBe4o10jad3WzWvVZZp5C0BOJFd08tsvjxxrW+FeCO5vPZKx3zbVjIHGIA3SP57WhzMsDB8TwJ4oVZ7fP2VY45HQwrGTaTk08eJec9aeTTzF/fqGOkBurptDoYLgRAcooWvzvVVUjupXJghetVOoO5m722Eefz8PmJ5N4Bn6/b20BTdtXO1bWnZxvXaz4FqVlcRKjxuexApwvzVkzjZLfYrju5wuyWtBE+vBdPI2V1X/wQ3JnNKyP/MTiurZybay0W6Puan9p+yt3DApow5t6N4/FNgbx7ZY4cS3vce0tHoaCwDzy+UsjDiK8clxSuW5umwe7VGbjRLpBzcaPhR+0GYffTYRtHHbWmd1k6RVWmtgU+BO+PVwY+l+M4KJXu48kTezdETmsbfTgvPs85lZ0suwuoBXoGd3cPq8VicVjFh2mTD9sshp/QGz6wg0oOd6KaSCMbLwHwSQyTV6LidEI4brmqp1fjmh5hGosneo9VTp7uFkti+PB8h3UGh/F4sWdJ2b6K4cbRud05Vy/wzvFBFlPIjqE5nRHbOTkEik4t2dOB5/t6cYfx9d3thcbfytggPjlgj/s49ws6z3gx3UPcb7g/aazp+d28ib6FPR79ewgCo0H2ZXfR7BN6MM+KHjoxRjl39dU4GKwpFTGid/bxpx8G9Rwt3P4n5y09BchC+OyKRz0YC978fL/tLnjphokWp5tZDzTi06t0ohX88uan/WA86mE0cfuf3vzC1a4tPpsORz2oSq1lhfj8hQv15vj5oXSCsdimGu9+UWsOHD5++uWnT+7fBkYurNSQ/M1HQg9OABwW+KCFmp7BeZhOp4uFPHrlEkU9rEoX8Ukb4aaD0k6n/DAlPkvu05s3v35y88EP38KvgfzNR17Nt1gLPp7itLAfh4tNsf1L8GtuLk/gJ6Of6fj5cWSfgveuTs8fD7L+5pdffx2c1h7CpEbkbyCVwtUYi+W+FPTfH3n02bdam7npVdHGiJ49phbBPv1wKrvk9r/iG1w8oQSjWE47Mo36pfnx10Fc9z1wfsxIzLB8/PTpFyzzHZ7QMVTpfnnz6aNPh7PYCC7k06u9uzxVXwYlH+fg+Biu7Bkt8PGjCx7tWzhFPp6uwH38mMS9xemhJF8s3O7iyHcu3N54nHGHO43FnHRiIT5b6veTY764mk5pBe/VykHWjUL+e/8J5t9+RpsEw853x88B4n/uQROpwRGlWx+2UXi4CRzoI+j90SmfNqgP9mS6Gn7eXcS+zDLeiL7btNcW/GjiPXcKHcCzHI5MdHgYUnofwsNi3LscHWSzvKiJbTb6b7cY76labZG4a3oMSa1jib7Y9uQVPmB9BcPzMs1ll8Jr0dNLFhdm4h7CrITmaFqvoKiVkZsPIOHV+/l59/P9ls51UfjyWvgYsTCv3bntRY05VTyrjqMJPhwOi0UmpMHp3j6y3OKF3nZJtlCuFS51ONC19GJwYXo+mnVhsbI/0vPQ7nC4d5Inmjy+yB11H3YUjxm4ForOvpv9/cKmtbmcXhWHZ7Hgs5uAn+x5/RTvi0v8Nku9VoJesKFYXmIUupvvtCo3OypnR3V7k2c4ndcjfnRjURLn+FrvLzYGJ6dNcdvrYUuTZ5uNER/dtE8nV+2XH95rmO8rtP8D+Q71Ub18tyQAAAAASUVORK5CYII="}


def show_matches(rows, sub):
    """A stacked win, draw, loss bar per match."""
    body, last = "", None
    for M, label, w, l, d in rows:
        n = w + l + d
        if label != last:
            art = OPPONENT_ART.get(label)
            if art is None:
                face = ""
            else:
                face = ('<img src="%s" alt="" style="height:72px;width:auto;'
                        'display:block;"/>') % art
            body += ('<div style="display:flex;align-items:center;gap:11px;'
                     'margin-top:16px;">%s'
                     '<span style="font-size:11px;font-weight:700;color:#475569;'
                     'letter-spacing:0.04em;text-transform:uppercase;">'
                     'against %s</span></div>') % (face, label)
            last = label
        seg = ""
        for count, color in ((w, "#15803d"), (d, "#94a3b8"), (l, "#b91c1c")):
            if count > 0:
                seg += ('<span style="height:15px;width:%.1fpx;background:%s;'
                        'opacity:0.72;"></span>') % (TRACK * count / n, color)
        body += (
            '<div style="display:flex;align-items:center;gap:10px;margin-top:6px;">'
            '<span style="%sfont-size:11px;color:#475569;flex:0 0 58px;">M = %d'
            '</span>'
            '<span style="display:flex;flex:0 0 %dpx;border-radius:8px;'
            'overflow:hidden;">%s</span>'
            '<span style="%sfont-size:10.5px;color:#64748b;">'
            '%d won, %d drew, %d lost</span></div>'
        ) % (MONO, M, int(TRACK), seg, MONO, w, d, l)
    legend = ('<div style="display:flex;gap:14px;margin-top:14px;'
              'font-size:10px;color:#94a3b8;">')
    for color, name in (("#15803d", "won"), ("#94a3b8", "drew"), ("#b91c1c", "lost")):
        legend += ('<span><span style="display:inline-block;width:9px;height:9px;'
                   'border-radius:2px;background:%s;opacity:0.72;"></span> %s</span>'
                   ) % (color, name)
    display(HTML(PANEL + panel_head("Can it hold a game?", sub) + body
                 + legend + '</div></div>'))


def check_mcts_move(fn):
    print("\U0001f50d  checking your mcts_move ...\n")
    passed = True

    try:
        got = fn(WIN_NOW, 100, W, random.Random(0))
    except Exception as e:
        bad("it raised %s: %s" % (type(e).__name__, e))
        hint("a 🎯 gap is probably still a bare `...`")
        return report("Exercise 6, mcts_move", False)

    if got is Ellipsis or got is None:
        bad("it returned %r" % (got,))
        return report("Exercise 6, mcts_move", False)
    if not isinstance(got, Node):
        bad("it returned a %s, it should return a Node" % type(got).__name__)
        hint("most_visited_child(root) gives you the node to play")
        return report("Exercise 6, mcts_move", False)
    if chosen_action(WIN_NOW, got.state) not in legal_actions(WIN_NOW):
        bad("the returned node is not one move on from the position given")
        return report("Exercise 6, mcts_move", False)
    ok("returns a child of the root")

    # The strict test. Run the canonical four lines with the participant's OWN
    # pieces and the same seed, then compare the tree your mcts_move actually
    # built. The returned node's parent IS your root, so we can look straight at
    # it. Any reordering (rolling out before expanding, backpropagating from the
    # node descend returned rather than the expanded one) changes these counts.
    reference_root = Node(WIN_NOW, None)
    rng = random.Random(0)
    for i in range(100):
        leaf = descend(reference_root, W)
        leaf = expand(leaf)
        r = rollout_reward(leaf.state, rng)
        backprop(leaf, r)
    want = sorted((chosen_action(WIN_NOW, c.state), c.N)
                  for c in reference_root.children)

    your_root = got.parent
    if your_root is None:
        bad("the node you returned has no parent, so it is not a child of a root")
        return report("Exercise 6, mcts_move", False)
    mine = sorted((chosen_action(WIN_NOW, c.state), c.N)
                  for c in your_root.children)
    if your_root.N != 101:
        bad("after 100 passes your root has N = %d, expected 101 (it started at 1)"
            % your_root.N)
        hint("every pass ends in one backprop that walks all the way up")
        passed = False
    elif mine != want:
        bad("your tree does not match the four phases run in order")
        hint("expected visit counts per cell %s" % (want,))
        hint("yours were                     %s" % (mine,))
        hint("the usual cause is line 2: expand RETURNS the new node, so it must "
             "be `leaf = expand(leaf)`. Calling expand(leaf) and then rolling out "
             "from the old leaf scores the parent instead of the node you added.")
        passed = False
    else:
        ok("the tree matches the four phases run in order")

    if chosen_action(WIN_NOW, got.state) != 2:
        bad("from a position where X wins by taking cell 2, you played cell %d"
            % chosen_action(WIN_NOW, got.state))
        hint("check the order inside the loop: descend, then expand, then roll out "
             "from the EXPANDED leaf, then backprop from that same leaf")
        passed = False
    else:
        ok("it takes the win on the board")

    for name, state, cell in (("block now", BLOCK_NOW, 7), ("fork", FORK, 0)):
        hits = 0
        for s in range(5):
            child = fn(state, 400, W, random.Random(s))
            if chosen_action(state, child.state) == cell:
                hits = hits + 1
        if hits < 4:
            bad("on the %s position it found cell %d in only %d of 5 runs"
                % (name, cell, hits))
            hint("at M = 400 this should be reliable; if backprop does not flip the "
                 "reward, blocking is exactly what breaks first")
            passed = False
        else:
            ok("%s: found cell %d in %d of 5 runs" % (name, cell, hits))

    # Most visited, not highest V. Hunt for a small budget where they disagree.
    for seed in range(15):
        probe_root = Node(EMPTY_BOARD, None)
        rng = random.Random(seed)
        for i in range(20):
            leaf = descend(probe_root, W)
            leaf = expand(leaf)
            r = rollout_reward(leaf.state, rng)
            backprop(leaf, r)
        by_visits = most_visited_child(probe_root)
        by_value = probe_root.children[0]
        for c in probe_root.children:
            if c.V > by_value.V:
                by_value = c
        if by_visits is by_value:
            continue
        picked = fn(EMPTY_BOARD, 20, W, random.Random(seed))
        if chosen_action(EMPTY_BOARD, picked.state) == chosen_action(
                EMPTY_BOARD, by_value.state):
            bad("with a small budget you returned the child with the highest V "
                "rather than the most visited one")
            hint("a high mean off a couple of lucky rollouts is noise; a high visit "
                 "count means the search kept coming back")
            passed = False
        else:
            ok("returns the most visited child, not the highest scoring one")
        break
    else:
        ok("returns a child of the root")
    return report("Exercise 6, mcts_move", passed)


In [ ]:
def mcts_move(state, n_attempts, w, rng):
    """The whole search. Four lines in the loop, one per phase.

    state        the position to move from
    n_attempts   how many passes to spend
    w            the exploration weight
    rng          a random.Random, handed on to the rollout
    returns      the Node to play
    """
    root = Node(state, None)

    for i in range(n_attempts):
        # 🎯 TARGET 7  Four lines, one per phase, in this order. The pseudocode
        # at the very top of this notebook has the same four, in the same order,
        # so scroll back to it and work from there. One difference to watch: the
        # rollout here also takes rng, which the pseudocode leaves out.

        #   1. selection: walk down from the root, with the weight w, to the
        #      node this pass will work on. Keep it in a variable.
        #   2. expansion: grow one node from that one, and carry on with the
        #      NEW node, because expand hands back the node it just created.
        #      This step reassigns the variable from step 1.
        #   3. simulation: play the rest of the game out at random and score it.
        #      It takes the leaf's STATE, not the leaf itself, plus rng.
        #   4. backpropagation: carry that reward back up, starting FROM THE
        #      LEAF you just expanded, not from wherever the rollout ended.
        ...

    # 🎯 TARGET 8  The move we actually play: the child of the root that
    # the search visited most.

    # One of the given helpers, in the cell titled "Given: most_visited_child,
    # rollout, rollout_reward", already does exactly that. Not the child with
    # the highest V: a high average off two lucky rollouts is noise, while a
    # high visit count means the search kept coming back to it.
    return ...


check_mcts_move(mcts_move)


In [ ]:
# @title 🎯 Hints for exercise 6, open me if you are stuck { display-mode: "form" }
print(r"""TARGET 7, the four lines inside the loop:

    leaf = descend(root, w)
    leaf = expand(leaf)
    r = rollout_reward(leaf.state, rng)
    backprop(leaf, r)

TARGET 8, the return:

    return most_visited_child(root)

Two things the checker looks at closely. The second line REASSIGNS leaf, because
expand returns the node it just made; calling expand(leaf) and leaving leaf
alone scores the parent instead of the node you added. And backprop starts at
that same leaf, because the rollout left no nodes behind to walk up through.""")


In [ ]:
# @title V is an estimate. N is how much it is worth { display-mode: "form" }
# The two numbers a node carries, and why only one of them can grade the
# other. Rendered at build time from a real search; see part2_assemble.py.
display(HTML("""<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f8fafc,#f1f5f9);border:1px solid #e2e8f0;border-radius:16px;padding:18px 16px 16px;margin:8px 0;"><div style="font-size:17px;font-weight:800;color:#1e293b;">V is an estimate. N is how much it is worth.</div><div style="font-size:12px;color:#64748b;margin-top:5px;line-height:1.5;">Each move is a poll. <b>V</b> is the result, <b>N</b> is how many were asked, and the bar is the room the truth still has to move in.</div><div style="display:flex;gap:22px;flex-wrap:wrap;margin-top:16px;"><div style="flex:1 1 320px;min-width:300px;"><div style="font-size:11px;font-weight:700;color:#475569;letter-spacing:0.04em;text-transform:uppercase;">after 24 passes</div><div style="font-size:10.5px;color:#94a3b8;margin-top:3px;line-height:1.4;">every bar still overlaps every other</div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#1e293b;flex:0 0 42px;font-weight:800;">cell 3</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:45.6px;width:71.8px;top:0;height:15px;border-radius:8px;background:#3b82f6;opacity:0.34;"></span><span style="position:absolute;left:81.5px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#3b82f6;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#1e293b;flex:0 0 54px;font-weight:800;">N = 7</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#1e293b;font-weight:800;">V 0.429</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 7</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:45.6px;width:71.8px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:81.5px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 7</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.429</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 8</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:24.5px;width:77.5px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:63.3px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 6</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.333</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 5</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:14.4px;width:85.1px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:57.0px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 5</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.300</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 6</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:0.0px;width:47.5px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:0.0px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 4</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.000</span></div><div style="display:flex;gap:9px;margin-top:5px;"><span style="flex:0 0 42px;"></span><span style="flex:0 0 190px;display:flex;justify-content:space-between;font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:9px;color:#cbd5e1;"><span>0.0</span><span>0.5</span><span>1.0</span></span></div></div><div style="flex:1 1 320px;min-width:300px;"><div style="font-size:11px;font-weight:700;color:#475569;letter-spacing:0.04em;text-transform:uppercase;">after 400 passes</div><div style="font-size:10.5px;color:#94a3b8;margin-top:3px;line-height:1.4;">one move pulls clear of the rest</div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#1e293b;flex:0 0 42px;font-weight:800;">cell 7</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:88.4px;width:12.5px;top:0;height:15px;border-radius:8px;background:#3b82f6;opacity:0.34;"></span><span style="position:absolute;left:94.6px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#3b82f6;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#1e293b;flex:0 0 54px;font-weight:800;">N = 231</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#1e293b;font-weight:800;">V 0.498</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 3</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:27.4px;width:27.9px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:41.2px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 46</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.217</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 5</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:26.0px;width:28.3px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:40.1px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 45</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.211</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 8</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:23.0px;width:29.1px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:37.6px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 43</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.198</span></div><div style="display:flex;align-items:center;gap:9px;margin-top:6px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 42px;font-weight:400;">cell 6</span><span style="position:relative;flex:0 0 190px;height:15px;border-radius:8px;background:rgba(148,163,184,0.13);"><span style="position:absolute;left:18.2px;width:30.0px;top:0;height:15px;border-radius:8px;background:#94a3b8;opacity:0.34;"></span><span style="position:absolute;left:33.2px;top:-1px;width:2.5px;height:17px;border-radius:2px;background:#94a3b8;"></span></span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;flex:0 0 54px;font-weight:400;">N = 40</span><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10.5px;color:#64748b;font-weight:400;">V 0.175</span></div><div style="display:flex;gap:9px;margin-top:5px;"><span style="flex:0 0 42px;"></span><span style="flex:0 0 190px;display:flex;justify-content:space-between;font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:9px;color:#cbd5e1;"><span>0.0</span><span>0.5</span><span>1.0</span></span></div></div></div><div style="display:flex;gap:22px;flex-wrap:wrap;margin-top:18px;align-items:flex-start;"><div style="flex:1 1 230px;min-width:220px;"><div style="font-size:11px;font-weight:700;color:#475569;letter-spacing:0.04em;text-transform:uppercase;">Ask more, know more</div><div style="display:flex;align-items:center;gap:9px;margin-top:5px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10px;color:#94a3b8;flex:0 0 52px;">N = 4</span><span style="height:11px;border-radius:6px;background:#3b82f6;opacity:0.42;width:95.0px;"></span></div><div style="display:flex;align-items:center;gap:9px;margin-top:5px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10px;color:#94a3b8;flex:0 0 52px;">N = 16</span><span style="height:11px;border-radius:6px;background:#3b82f6;opacity:0.42;width:47.5px;"></span></div><div style="display:flex;align-items:center;gap:9px;margin-top:5px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10px;color:#94a3b8;flex:0 0 52px;">N = 64</span><span style="height:11px;border-radius:6px;background:#3b82f6;opacity:0.42;width:23.8px;"></span></div><div style="display:flex;align-items:center;gap:9px;margin-top:5px;"><span style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:10px;color:#94a3b8;flex:0 0 52px;">N = 231</span><span style="height:11px;border-radius:6px;background:#3b82f6;opacity:0.42;width:12.5px;"></span></div></div><div style="flex:1 1 230px;min-width:220px;"><div style="font-size:11px;font-weight:700;color:#475569;letter-spacing:0.04em;text-transform:uppercase;">And do they ever disagree</div><div style="display:flex;gap:8px;margin-top:7px;"><div style="flex:1 1 120px;text-align:center;padding:9px 6px;border-radius:10px;background:rgba(148,163,184,0.12);"><div style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:17px;font-weight:800;color:#334155;">8 of 104</div><div style="font-size:9.5px;color:#94a3b8;margin-top:2px;">disagreed at M = 25</div></div><div style="flex:1 1 120px;text-align:center;padding:9px 6px;border-radius:10px;background:rgba(148,163,184,0.12);"><div style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:17px;font-weight:800;color:#334155;">2 of 104</div><div style="font-size:9.5px;color:#94a3b8;margin-top:2px;">disagreed at M = 400</div></div></div><div style="font-size:9.5px;color:#94a3b8;margin-top:6px;line-height:1.4;">out of 104 graded positions, most visited against highest V</div></div></div><div style="font-size:11px;font-weight:700;color:#475569;margin-top:20px;letter-spacing:0.04em;text-transform:uppercase;">So why play the most visited one</div><div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:9px;"><div style="flex:1 1 196px;min-width:186px;padding:11px 12px;border-radius:11px;background:#ffffff;border:1px solid #e2e8f0;"><div style="font-size:17px;line-height:1;">&#128499;</div><div style="font-size:12px;font-weight:800;color:#1e293b;margin-top:6px;line-height:1.35;">It is the search's own verdict</div><div style="font-size:10.5px;color:#64748b;margin-top:5px;line-height:1.45;">V averages some playouts. N tallies every single time the search weighed the options and chose to come back here.</div><div style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:9.5px;color:#3b82f6;font-weight:700;margin-top:8px;line-height:1.35;">231 of 400 passes went to one move</div></div><div style="flex:1 1 196px;min-width:186px;padding:11px 12px;border-radius:11px;background:#ffffff;border:1px solid #e2e8f0;"><div style="font-size:17px;line-height:1;">&#128274;</div><div style="font-size:12px;font-weight:800;color:#1e293b;margin-top:6px;line-height:1.35;">It has a floor, and V has none</div><div style="font-size:10.5px;color:#64748b;margin-top:5px;line-height:1.45;">The most visited child is the best tested one by definition. The best average can be a move hardly anybody looked at.</div><div style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:9.5px;color:#3b82f6;font-weight:700;margin-top:8px;line-height:1.35;">at M = 10, highest V answered off as few as 2 playouts</div></div><div style="flex:1 1 196px;min-width:186px;padding:11px 12px;border-radius:11px;background:#ffffff;border:1px solid #e2e8f0;"><div style="font-size:17px;line-height:1;">&#128200;</div><div style="font-size:12px;font-weight:800;color:#1e293b;margin-top:6px;line-height:1.35;">It is built for games that never settle</div><div style="font-size:10.5px;color:#64748b;margin-top:5px;line-height:1.45;">Here every move gets tested, so the two agree. Give a game hundreds of moves a turn and most stay thin forever.</div><div style="font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:9.5px;color:#3b82f6;font-weight:700;margin-top:8px;line-height:1.35;">why real engines all play the visit count</div></div></div><div style="font-size:11.5px;color:#475569;margin-top:16px;line-height:1.5;"><b>They are not rival scores.</b> V is what the search thinks, N is how hard it tested the thought. Play the most visited child and you play the move it actually had time to check.</div></div>"""))


### Watch one search think

`N` is how many passes went through each move, `V` is what the search came to believe it
is worth *to X, who moved there*. The good move should have taken most of the budget.


In [ ]:
root = Node(FORK, None)
rng = random.Random(0)
for i in range(400):
    leaf = descend(root, W)
    leaf = expand(leaf)
    r = rollout_reward(leaf.state, rng)
    backprop(leaf, r)

show_search(FORK, root,
            "X to move, and cell 0 makes two threats at once, so it is the only "
            "good move. The bar is N, how many of the 400 passes went through "
            "each move. V is what the search came to believe it is worth to X.")


### How often is it right?

Tic tac toe is small enough to solve outright, so every answer can be graded. The cell
below asks the search to move in every position with X to move and at most four marks,
and checks each choice against a perfect player. About ten seconds.


In [ ]:
states = graded_positions()

rows = []
for M in (25, 100, 400):
    rows.append((M, agreement(states, M)))

show_agreement(rows, len(states),
               "Every position with X to move and at most four marks, graded "
               "against a solver that plays perfectly. M is how many passes the "
               "search was allowed per move.")


### And can it hold a game?


In [ ]:
rows = []
for label, opponent in (("a random player", random_mover),
                        ("a perfect player", perfect_mover)):
    for M in (100, 400):
        w, l, d = play_match(M, opponent)
        rows.append((M, label, w, l, d))

show_matches(rows, "Twenty games each, colours alternating, so the search plays "
                   "X in half of them and O in the other half.")


## 4. What the numbers say

Agreement climbs from 86.6 percent at M=25 to **99.4 percent at M=400**, and against a
perfect opponent the search goes from losing 6 games of 20 to losing 2. Run it further
and it keeps closing: at M=800 it agrees with perfect play on 826 of the 829 positions.


In [ ]:
states = graded_positions()      # recomputed here so the cell stands alone

curve = []
for M in (25, 50, 100, 200, 400, 800):
    curve.append((M, agreement(states, M)))

show_curve([("agreement", "#3b82f6", curve)], len(states),
           "The same %d positions, graded against a perfect player, as the budget "
           "doubles. Every extra pass buys a little more agreement, and the line "
           "keeps rising rather than flattening short of the answer."
           % len(states))


That is the shape you want from a search: **more compute, monotonically better play**,
converging on the right answer rather than plateauing short of it.

It matters that this happens here, because it does not always. Two things had to be true.
The rollouts had to carry real signal, which in a game this small they do. And the tree
had to be able to grow towards the good lines, which is what `expand` and the descending
`descend` are for. Take either away and the curve flattens.


### And if the playout told you nothing?

That last claim is worth testing rather than believing. Below, the search is left exactly
as you wrote it, four phases and all, and only one thing is swapped: instead of playing
the game out and scoring the ending, the reward is a coin flip that never even looks at
the board. The tree still grows one node per pass. The counters still fill up.


In [ ]:
def blind_reward(state, rng):
    """A playout that carries no information at all."""
    return rng.choice([0.0, 0.5, 1.0])


states = graded_positions()

real, blind = [], []
for M in (25, 100, 400):
    real.append((M, agreement_with(states, M, rollout_reward)))
    blind.append((M, agreement_with(states, M, blind_reward)))

show_curve([("the real playout", "#3b82f6", real),
            ("a coin flip", "#b91c1c", blind)],
           len(states),
           "The same search, the same budgets, the same %d positions. The only "
           "difference is where the number at the end of each pass comes from."
           % len(states),
           floor=40.0, title="Search cannot invent what the playout does not know")


The blue line climbs. The red one does not move, and at 400 passes it is no better than
at 25, hovering around what you would get by guessing. Every part of the algorithm is
still working: the tree grows, `descend` picks, `backprop` carries. It is simply carrying
noise, and no amount of it adds up to knowledge.

**That is the sentence worth leaving with.** Search is an amplifier. Point it at a signal
and more compute buys you more of it, which is the whole curve above. Point it at nothing
and more compute buys nothing at all.
